In [ ]:
# ============================================================
# FIXED-CANDIDATE FULL CATALOG REBUILD
# ------------------------------------------------------------
# Frozen branch:
#   source: rho_eff ~ (1/r^2) d/dr [ r * V_bar^2 ]
#   screened field equation
#   fixed Rs = 1.5
#   shape map: V/V_flat = U/U_inf
#
# Frozen amplitude law:
#   U(r_t) = 0.6 U_inf
#   V_flat^2 = C * [r_t u(r_t)]^alpha * [Vbar^2(r_t)]^beta
#
# with fixed constants:
#   C     = 164.0
#   alpha = 0.175
#   beta  = 0.55
#
# No fitting in this script.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_fixed_candidate_catalog_rebuild")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# RUN FULL CATALOG
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()
rotmod_map = {display_name_from_path(p): p for p in rotmod_files}

rows = []
fails = []

print("=" * 60)
print("RUNNING FIXED-CANDIDATE FULL CATALOG")
print("=" * 60)
print(f"Candidate files : {len(rotmod_files)}")
print(f"Fixed Rs        : {Rs_fixed}")
print(f"Fixed f         : {F_FRAC}")
print(f"Fixed alpha     : {ALPHA}")
print(f"Fixed beta      : {BETA}")
print(f"Fixed C         : {C_AMP}")
print()

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        U_inf = float(np.max(U))
        rt = radius_at_fraction(r, U, F_FRAC)

        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            raise RuntimeError("Invalid transition quantities")

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)
        chi2_mean = float(np.mean(((rot["vobs"] - V_obs_pred) / rot["ev"]) ** 2))

        obs_shape = rot["vobs"] / max(float(np.max(rot["vobs"])), 1.0e-12)
        pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
        shape_rmse = safe_rmse(obs_shape, pred_shape)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "Rs_fixed": Rs_fixed,
            "rt": rt,
            "u_t": u_t,
            "vbar2_t": vbar2_t,
            "carrier": carrier,
            "V_flat_pred": V_flat_pred,
            "rmse": rmse,
            "mae": mae,
            "chi2_mean": chi2_mean,
            "shape_rmse": shape_rmse,
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

res_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(res_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

# Simple bins for tail view
res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

print()
print("=" * 60)
print("FIXED-CANDIDATE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

RES_CSV = os.path.join(OUTDIR, "fixed_candidate_results.csv")
FAIL_CSV = os.path.join(OUTDIR, "fixed_candidate_failures.csv")
BEST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_best20.csv")
WORST_CSV_OUT = os.path.join(OUTDIR, "fixed_candidate_worst20.csv")
BIN_CSV = os.path.join(OUTDIR, "fixed_candidate_bin_summary.csv")
TXT_OUT = os.path.join(OUTDIR, "fixed_candidate_summary.txt")

res_df.to_csv(RES_CSV, index=False)
fail_df.to_csv(FAIL_CSV, index=False)
best20.to_csv(BEST_CSV_OUT, index=False)
worst20.to_csv(WORST_CSV_OUT, index=False)
bin_summary.to_csv(BIN_CSV, index=False)

with open(TXT_OUT, "w", encoding="utf-8") as f:
    f.write("FIXED-CANDIDATE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "rt", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

Auto-extracting LTG rotmods from: /content/Rotmod_LTG (4).zip
RUNNING FIXED-CANDIDATE FULL CATALOG
Candidate files : 175
Fixed Rs        : 1.5
Fixed f         : 0.6
Fixed alpha     : 0.175
Fixed beta      : 0.55
Fixed C         : 164.0

Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

FIXED-CANDIDATE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
median_rmse            = 20.38786367728187
mean_rmse              = 32.72222113537459
p90_rmse               = 77.35749478059131
p95_rmse               = 88.8575199776623
median_mae             = 17.608721572718746
median_chi2_mean       = 26.12511787253237
median_shape_rmse      = 0.15760201986858705

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 11.457480 10.528257  14.716546    0.128219
     mid 18.074875 15.106115  13.274674    0.115517
    hig

In [ ]:
# ============================================================
# HIGH-END AMPLITUDE TAIL CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Tail correction:
#   V_flat,corr = V_flat,base * [1 + eta * (Vbar2_t / V0^2)^delta]
#
# Only amplitude is modified. Source/field/shape stay fixed.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_tail_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
V0_LIST    = [120.0, 150.0, 180.0, 210.0, 240.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return (
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"])
    )

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(vbar2_from_rot(rot), 0.0)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TAIL-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_obs = np.maximum(vbar2_from_rot(rot), 0.0)
        vbar2_interp = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "vbar2_t": vbar2_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for V0 in V0_LIST:
            rmses = []
            vmax_bins = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["vbar2_t"] / (V0**2)) ** delta
                V_flat_pred = row["V_flat_base"] * correction
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmax_bins.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmax_bins = np.asarray(vmax_bins, float)

            high_mask = vmax_bins >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "V0": V0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("HIGH-END TAIL CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_tail_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_tail_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END TAIL CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TAIL-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END TAIL CORRECTION SUMMARY
 eta  delta    V0  median_rmse  p90_rmse  high_bin_median_rmse
0.00   0.25 120.0    20.387864 77.357495             63.789441
0.00   0.25 150.0    20.387864 77.357495             63.789441
0.00   0.25 180.0    20.387864 77.357495             63.789441
0.00   0.25 210.0    20.387864 77.357495             63.789441
0.00   0.25 240.0    20.387864 77.357495             63.789441
0.00   0.50 120.0    20.387864 77.357495             63.789441
0.00   0.50 150.0    20.387864 77.357495             63.789441
0.00   0.50 180.0    20.387864 77.357495             63.789441
0.00   0.50 210.0    20.387864 77.357495             63.789441
0.00   0.50 240.0    20.387864 77.357495             63.789441
0.00   0.75 120.0    20.387864 77.357495             6

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# HIGH-END FAILURE-MODE AUDIT
# ------------------------------------------------------------
# Audits the fixed candidate branch:
#
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Focus:
#   Understand what the worst high-vmax failures share.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_high_end_failure_audit")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

HIGH_VMAX_CUT = 150.0

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files
    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break
    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")
    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_components(rot):
    vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
    vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
    vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
    vbar2  = vgas2 + vdisk2 + vbul2
    return vgas2, vdisk2, vbul2, vbar2

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    _, _, _, vbar2 = vbar2_components(rot)
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("RUNNING FAILURE-MODE AUDIT")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2, vdisk2, vbul2, vbar2_obs = vbar2_components(rot)
        vgas2_i  = interp1d(rot["r"], vgas2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vdisk2_i = interp1d(rot["r"], vdisk2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_i  = interp1d(rot["r"], vbul2,  kind="linear", bounds_error=False, fill_value="extrapolate")
        vbar2_i  = interp1d(rot["r"], vbar2_obs, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vgas2_t = max(float(vgas2_i(rt)), 0.0)
        vdisk2_t = max(float(vdisk2_i(rt)), 0.0)
        vbul2_t = max(float(vbul2_i(rt)), 0.0)
        vbar2_t = max(float(vbar2_i(rt)), 1.0e-30)

        bulge_frac_t = vbul2_t / vbar2_t if vbar2_t > 0 else np.nan
        disk_frac_t = vdisk2_t / vbar2_t if vbar2_t > 0 else np.nan
        gas_frac_t = vgas2_t / vbar2_t if vbar2_t > 0 else np.nan

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_pred = np.sqrt(max(C_AMP * carrier, 0.0))
        V_theory = V_flat_pred * U / max(U_inf, 1.0e-12)

        V_obs_pred = interp1d(
            r, V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(r_obs)

        rmse = safe_rmse(rot["vobs"], V_obs_pred)
        mae = safe_mae(rot["vobs"], V_obs_pred)

        rows.append({
            "galaxy": galaxy,
            "n_points": len(r_obs),
            "r_max": float(np.max(r_obs)),
            "vmax_obs": float(np.max(rot["vobs"])),
            "rt": rt,
            "rt_over_rmax": rt / max(float(np.max(r_obs)), 1.0e-12),
            "u_t": u_t,
            "rt_ut": rt * u_t,
            "vbar2_t": vbar2_t,
            "vbar_t": np.sqrt(vbar2_t),
            "bulge_frac_t": bulge_frac_t,
            "disk_frac_t": disk_frac_t,
            "gas_frac_t": gas_frac_t,
            "V_flat_pred": V_flat_pred,
            "Vflat_over_vmax": V_flat_pred / max(float(np.max(rot["vobs"])), 1.0e-12),
            "rmse": rmse,
            "mae": mae,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)
high = df[df["vmax_obs"] >= HIGH_VMAX_CUT].copy().sort_values("rmse", ascending=False).reset_index(drop=True)

worst20 = high.head(20).copy()
best20 = high.tail(20).copy().sort_values("rmse").reset_index(drop=True)

summary_vars = [
    "vmax_obs", "r_max", "rt", "rt_over_rmax", "u_t", "rt_ut",
    "vbar2_t", "vbar_t", "bulge_frac_t", "disk_frac_t", "gas_frac_t",
    "V_flat_pred", "Vflat_over_vmax", "rmse", "mae"
]

summary_df = pd.DataFrame({
    "quantity": summary_vars,
    "worst20_median": [float(worst20[c].median()) for c in summary_vars],
    "best20_median": [float(best20[c].median()) for c in summary_vars],
})
summary_df["worst_minus_best"] = summary_df["worst20_median"] - summary_df["best20_median"]

print()
print("=" * 60)
print("HIGH-END FAILURE AUDIT SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

print()
print("=" * 60)
print("WORST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(worst20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

print()
print("=" * 60)
print("BEST 20 HIGH-VMAX GALAXIES")
print("=" * 60)
print(best20[[
    "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
    "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
]].to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "high_end_failure_summary.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "high_end_worst20.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "high_end_best20.csv"), index=False)
df.to_csv(os.path.join(OUTDIR, "high_end_full_audit.csv"), index=False)

with open(os.path.join(OUTDIR, "high_end_failure_summary.txt"), "w", encoding="utf-8") as f:
    f.write("HIGH-END FAILURE AUDIT SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))
    f.write("\n\nWORST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))
    f.write("\n\nBEST 20 HIGH-VMAX GALAXIES\n")
    f.write("=" * 60 + "\n")
    f.write(best20[[
        "galaxy", "vmax_obs", "rt", "rt_over_rmax", "vbar_t",
        "bulge_frac_t", "gas_frac_t", "V_flat_pred", "Vflat_over_vmax", "rmse"
    ]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

RUNNING FAILURE-MODE AUDIT
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

HIGH-END FAILURE AUDIT SUMMARY
       quantity  worst20_median  best20_median  worst_minus_best
       vmax_obs      274.000000     186.000000         88.000000
          r_max       44.215000      26.930000         17.285000
             rt        4.236476       3.368903          0.867573
   rt_over_rmax        0.154626       0.124820          0.029805
            u_t        0.097594       0.084961          0.012633
          rt_ut        0.423070       0.275523          0.147548
        vbar2_t    77995.021748   35983.678528      42011.343220
         vbar_t      279.272915     189.684292         89.588623
   bulge_frac_t        0.281828       0.000000          0.281828
    disk_frac_t        0.717890       0.987273         -0.269383
     gas_frac_t        0.000091       

In [ ]:
# ============================================================
# BULGE-SENSITIVE TRANSITION CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Bulge correction:
#   V_flat,corr^2 = V_flat,base^2 * [1 + eta * f_bulge(rt)^delta]
#
# where
#   f_bulge(rt) = Vbulge^2(rt) / Vbar^2(rt)
#
# Goal:
#   see if high-vmax tail improves without harming the catalog.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_bulge_transition_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
DELTA_LIST = [0.50, 0.75, 1.00, 1.25, 1.50]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING BULGE-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2  = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
        vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
        vbul2  = np.maximum(UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]), 0.0)
        vbar2  = vgas2 + vdisk2 + vbul2

        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_interp = interp1d(rot["r"], vbul2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        vbul2_t = max(float(vbul2_interp(rt)), 0.0)
        f_bulge_t = vbul2_t / vbar2_t if vbar2_t > 0 else 0.0

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "f_bulge_t": f_bulge_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        rmses = []
        vmaxs = []

        for _, row in df.iterrows():
            correction = 1.0 + eta * (row["f_bulge_t"] ** delta)
            V_flat_pred = row["V_flat_base"] * np.sqrt(correction)
            V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

            V_obs_pred = interp1d(
                row["r_grid"], V_theory,
                kind="linear", bounds_error=False, fill_value="extrapolate"
            )(row["r_obs"])

            rmse = safe_rmse(row["v_obs"], V_obs_pred)
            rmses.append(rmse)
            vmaxs.append(row["vmax_obs"])

        rmses = np.asarray(rmses, float)
        vmaxs = np.asarray(vmaxs, float)

        high_mask = vmaxs >= 150.0
        high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

        summary_rows.append({
            "eta": eta,
            "delta": delta,
            "median_rmse": float(np.median(rmses)),
            "p90_rmse": float(np.percentile(rmses, 90)),
            "high_bin_median_rmse": high_med,
        })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("BULGE-CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "bulge_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "bulge_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("BULGE-CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING BULGE-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

BULGE-CORRECTION SUMMARY
 eta  delta  median_rmse  p90_rmse  high_bin_median_rmse
0.10   1.00    20.387864 76.658395             62.747288
0.10   1.25    20.387864 76.767365             62.760592
0.10   0.75    20.387864 76.558643             62.809717
0.10   1.50    20.387864 76.870950             62.844898
0.10   0.50    20.469380 77.125908             62.872927
0.05   0.50    20.387864 76.927244             63.666174
0.05   0.75    20.387864 76.974112             63.716538
0.05   1.00    20.387864 77.033582             63.764955
0.05   1.25    20.387864 77.093313             63.789441
0.05   1.50    20.387864 77.148207             63.789441
0.00   0.50    20.387864 77.358929             63.789441
0.00   0.75    20.387864 77.358929             63.789441
0.00   1.00  

In [ ]:
# ============================================================
# TRANSITION COMPACTNESS CORRECTION TEST
# ------------------------------------------------------------
# Fixed base branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat,base^2 = 164 * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Compactness correction:
#   c_t = Vbar^2(rt) / rt
#   V_flat,corr^2 = V_flat,base^2 * [1 + eta * (c_t / c0)^delta]
#
# Goal:
#   improve the high-vmax tail without reopening the model.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_transition_compactness_correction")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
C_AMP  = 164.0
ALPHA  = 0.175
BETA   = 0.55

ETA_LIST   = [0.00, 0.05, 0.10, 0.15, 0.20, 0.30]
DELTA_LIST = [0.25, 0.50, 0.75, 1.00]
C0_LIST    = [5_000.0, 10_000.0, 20_000.0, 40_000.0, 80_000.0]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING COMPACTNESS-CORRECTION TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vbar2 = np.maximum(
            rot["vgas"] * np.abs(rot["vgas"])
            + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
            + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
            0.0
        )
        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        c_t = vbar2_t / max(rt, 1.0e-12)

        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA
        V_flat_base = np.sqrt(max(C_AMP * carrier, 0.0))

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "c_t": c_t,
            "V_flat_base": V_flat_base,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for eta in ETA_LIST:
    for delta in DELTA_LIST:
        for c0 in C0_LIST:
            rmses = []
            vmaxs = []

            for _, row in df.iterrows():
                correction = 1.0 + eta * (row["c_t"] / c0) ** delta
                V_flat_pred = row["V_flat_base"] * np.sqrt(correction)
                V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

                V_obs_pred = interp1d(
                    row["r_grid"], V_theory,
                    kind="linear", bounds_error=False, fill_value="extrapolate"
                )(row["r_obs"])

                rmse = safe_rmse(row["v_obs"], V_obs_pred)
                rmses.append(rmse)
                vmaxs.append(row["vmax_obs"])

            rmses = np.asarray(rmses, float)
            vmaxs = np.asarray(vmaxs, float)

            high_mask = vmaxs >= 150.0
            high_med = np.nan if np.sum(high_mask) == 0 else float(np.median(rmses[high_mask]))

            summary_rows.append({
                "eta": eta,
                "delta": delta,
                "c0": c0,
                "median_rmse": float(np.median(rmses)),
                "p90_rmse": float(np.percentile(rmses, 90)),
                "high_bin_median_rmse": high_med,
            })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("COMPACTNESS-CORRECTION SUMMARY")
print("=" * 60)
print(summary_df.head(25).to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "compactness_correction_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "compactness_correction_summary.txt"), "w", encoding="utf-8") as f:
    f.write("COMPACTNESS-CORRECTION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING COMPACTNESS-CORRECTION TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

COMPACTNESS-CORRECTION SUMMARY
 eta  delta      c0  median_rmse  p90_rmse  high_bin_median_rmse
0.20   0.75 80000.0    20.492094 77.523220             63.289288
0.10   0.75 40000.0    20.475352 77.384626             63.452850
0.05   0.25 10000.0    20.720630 77.126245             63.592638
0.15   0.75 80000.0    20.465813 77.402844             63.598978
0.05   1.00 20000.0    20.421821 77.538450             63.623908
0.10   1.00 40000.0    20.421821 77.538450             63.623908
0.20   1.00 80000.0    20.421821 77.538450             63.623908
0.15   1.00 40000.0    20.438896 77.450237             63.664564
0.30   1.00 80000.0    20.438896 77.450237             63.664564
0.05   0.75 20000.0    20.461319 77.411665             63.669028
0.05   0.50 10000.0    20.5

In [ ]:
# ============================================================
# TWO-POPULATION NORMALIZATION TEST
# ------------------------------------------------------------
# Fixed branch:
#   Rs = 1.5
#   U(rt) = 0.6 U_inf
#   V/V_flat = U/U_inf
#   V_flat^2 = C_pop * [rt*u(rt)]^0.175 * [Vbar^2(rt)]^0.55
#
# Split by bulge fraction at rt:
#   f_bulge(rt) = Vbulge^2(rt) / Vbar^2(rt)
#
# We test several thresholds and fit ONLY the normalization
# separately for low-bulge and high-bulge populations.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_two_population_normalization")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

F_FRAC = 0.60
ALPHA  = 0.175
BETA   = 0.55

# Bulge-fraction thresholds to try
THRESH_LIST = [0.05, 0.10, 0.20, 0.30, 0.40]

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def radius_at_fraction(r, y, frac):
    y = np.asarray(y, float)
    r = np.asarray(r, float)
    ymax = np.nanmax(y)
    if not np.isfinite(ymax) or ymax <= 0:
        return np.nan
    target = frac * ymax
    idx = np.where(y >= target)[0]
    if len(idx) == 0:
        return np.nan
    return float(r[idx[0]])

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )
    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)
    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr
        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)
        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

def fit_C(subdf):
    x = subdf["carrier"].values.astype(float)
    y = subdf["V_flat_fit"].values.astype(float) ** 2
    m = np.isfinite(x) & np.isfinite(y) & (x > 0)
    return float(np.sum(x[m] * y[m]) / np.sum(x[m] ** 2))

rotmod_files = bootstrap_rotmods()

rows = []
print("=" * 60)
print("BUILDING TWO-POP TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)
    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(max(R_MIN, float(np.min(r_obs)) * 0.5),
                               max(float(np.max(r_obs)) * 1.10, 1.0), N_R)

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)
        U_inf = float(np.max(U))

        rt = radius_at_fraction(r, U, F_FRAC)
        if not (np.isfinite(U_inf) and U_inf > 0 and np.isfinite(rt) and rt > 0):
            continue

        u_interp = interp1d(r, u, kind="linear", bounds_error=False, fill_value="extrapolate")

        vgas2 = np.maximum(rot["vgas"] * np.abs(rot["vgas"]), 0.0)
        vdisk2 = np.maximum(UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"]), 0.0)
        vbul2 = np.maximum(UPS_BUL * rot["vbul"] * np.abs(rot["vbul"]), 0.0)
        vbar2 = vgas2 + vdisk2 + vbul2

        vbar2_interp = interp1d(rot["r"], vbar2, kind="linear", bounds_error=False, fill_value="extrapolate")
        vbul2_interp = interp1d(rot["r"], vbul2, kind="linear", bounds_error=False, fill_value="extrapolate")

        u_t = max(float(u_interp(rt)), 1.0e-30)
        vbar2_t = max(float(vbar2_interp(rt)), 1.0e-30)
        vbul2_t = max(float(vbul2_interp(rt)), 0.0)

        f_bulge_t = vbul2_t / vbar2_t if vbar2_t > 0 else 0.0
        carrier = (rt * u_t) ** ALPHA * (vbar2_t) ** BETA

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "f_bulge_t": f_bulge_t,
            "carrier": carrier,
            "V_flat_fit": float(np.max(rot["vobs"])),  # same target convention used in these tests
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": U_inf,
        })

    except Exception:
        pass

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

df = pd.DataFrame(rows)

summary_rows = []

for thresh in THRESH_LIST:
    low = df[df["f_bulge_t"] < thresh].copy()
    high = df[df["f_bulge_t"] >= thresh].copy()

    if len(low) < 10 or len(high) < 10:
        continue

    C_low = fit_C(low)
    C_high = fit_C(high)

    eval_rows = []
    for _, row in df.iterrows():
        C_use = C_low if row["f_bulge_t"] < thresh else C_high
        V_flat_pred = np.sqrt(max(C_use * row["carrier"], 0.0))
        V_theory = V_flat_pred * row["U_grid"] / max(row["U_inf"], 1.0e-12)

        V_obs_pred = interp1d(
            row["r_grid"], V_theory,
            kind="linear", bounds_error=False, fill_value="extrapolate"
        )(row["r_obs"])

        rmse = safe_rmse(row["v_obs"], V_obs_pred)

        eval_rows.append({
            "vmax_obs": row["vmax_obs"],
            "rmse": rmse,
        })

    eval_df = pd.DataFrame(eval_rows)
    high_mask = eval_df["vmax_obs"] >= 150.0

    summary_rows.append({
        "thresh": thresh,
        "n_lowbulge": int(len(low)),
        "n_highbulge": int(len(high)),
        "C_lowbulge": C_low,
        "C_highbulge": C_high,
        "C_ratio_high_over_low": C_high / C_low if C_low != 0 else np.nan,
        "median_rmse": float(eval_df["rmse"].median()),
        "p90_rmse": float(np.percentile(eval_df["rmse"], 90)),
        "high_bin_median_rmse": float(eval_df.loc[high_mask, "rmse"].median()),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["high_bin_median_rmse", "median_rmse", "p90_rmse"]
).reset_index(drop=True)

print()
print("=" * 60)
print("TWO-POPULATION NORMALIZATION SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

summary_df.to_csv(os.path.join(OUTDIR, "two_population_normalization_summary.csv"), index=False)

with open(os.path.join(OUTDIR, "two_population_normalization_summary.txt"), "w", encoding="utf-8") as f:
    f.write("TWO-POPULATION NORMALIZATION SUMMARY\n")
    f.write("=" * 60 + "\n")
    f.write(summary_df.to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING TWO-POP TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

TWO-POPULATION NORMALIZATION SUMMARY
 thresh  n_lowbulge  n_highbulge  C_lowbulge  C_highbulge  C_ratio_high_over_low  median_rmse  p90_rmse  high_bin_median_rmse
   0.40         156           19  146.447719   203.300229               1.388210    20.603011 76.133411             61.506610
   0.30         155           20  147.671731   195.243356               1.322144    20.699052 76.818147             61.526319
   0.05         144           31  142.754414   174.181652               1.220149    20.969344 78.742874             63.592373
   0.20         151           24  141.596637   186.750763               1.318893    21.522831 79.221134             63.731267
   0.10         146           29  138.148873   182.260437               1.319305    21.745393 80.295051             64.23

In [ ]:
# ============================================================
# DERIVED SQRT-BRIDGE TEST
# ------------------------------------------------------------
# Field:
#   (1/r^2) d/dr (r^2 dm/dr) - (1/Rs^2)(m-m_inf) = -A rho_eff
#
# Define:
#   u(r) = m(r) - m_inf
#   U(r) = ∫_0^r u(s) ds
#
# Derived observable law:
#   g_obs(r) = K * U(r) / r
# so
#   V(r)^2 = K * U(r)
#   V(r)   = sqrt(K * U(r))
#
# One global K across the full catalog.
# Fixed source branch and fixed Rs = 1.5.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_derived_sqrt_bridge_test")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    m_interp = interp1d(r, m, kind="linear", bounds_error=False, fill_value=(m[0], m[-1]))
    return {"r": r_eval, "m": m_interp(r_eval)}

# ------------------------------------------------------------
# BUILD FIELD TABLE
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()

rows = []
fails = []

print("=" * 60)
print("BUILDING SQRT-BRIDGE FIELD TABLE")
print("=" * 60)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        field = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = field["r"]
        m = field["m"]
        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        if not np.isfinite(np.max(U)) or np.max(U) <= 0:
            raise RuntimeError("Invalid U profile")

        rows.append({
            "galaxy": galaxy,
            "r_obs": rot["r"],
            "v_obs": rot["vobs"],
            "e_obs": rot["ev"],
            "r_grid": r,
            "U_grid": U,
            "U_inf": float(np.max(U)),
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

field_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(field_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

# ------------------------------------------------------------
# FIT GLOBAL K FROM V^2 = K U
# ------------------------------------------------------------

num = 0.0
den = 0.0

for _, row in field_df.iterrows():
    U_obs = interp1d(
        row["r_grid"], row["U_grid"],
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    m = np.isfinite(U_obs) & np.isfinite(row["v_obs"]) & (U_obs > 0)
    if np.sum(m) == 0:
        continue

    x = U_obs[m]
    y = row["v_obs"][m] ** 2
    num += np.sum(x * y)
    den += np.sum(x * x)

if den <= 0:
    raise RuntimeError("Could not fit global K")

K_global = num / den

# ------------------------------------------------------------
# EVALUATE
# ------------------------------------------------------------

res_rows = []

for _, row in field_df.iterrows():
    V_theory = np.sqrt(np.maximum(K_global * row["U_grid"], 0.0))
    V_obs_pred = interp1d(
        row["r_grid"], V_theory,
        kind="linear", bounds_error=False, fill_value="extrapolate"
    )(row["r_obs"])

    rmse = safe_rmse(row["v_obs"], V_obs_pred)
    mae = safe_mae(row["v_obs"], V_obs_pred)
    chi2_mean = float(np.mean(((row["v_obs"] - V_obs_pred) / row["e_obs"]) ** 2))

    obs_shape = row["v_obs"] / max(float(np.max(row["v_obs"])), 1.0e-12)
    pred_shape = V_obs_pred / max(float(np.max(V_obs_pred)), 1.0e-12)
    shape_rmse = safe_rmse(obs_shape, pred_shape)

    V_flat_pred = float(np.sqrt(max(K_global * row["U_inf"], 0.0)))

    res_rows.append({
        "galaxy": row["galaxy"],
        "n_points": len(row["r_obs"]),
        "vmax_obs": float(np.max(row["v_obs"])),
        "U_inf": row["U_inf"],
        "V_flat_pred": V_flat_pred,
        "rmse": rmse,
        "mae": mae,
        "chi2_mean": chi2_mean,
        "shape_rmse": shape_rmse,
    })

res_df = pd.DataFrame(res_rows)

summary = {
    "successful_galaxies": int(len(res_df)),
    "failed_galaxies": int(len(fail_df)),
    "K_global": float(K_global),
    "median_rmse": float(res_df["rmse"].median()),
    "mean_rmse": float(res_df["rmse"].mean()),
    "p90_rmse": float(np.percentile(res_df["rmse"], 90)),
    "p95_rmse": float(np.percentile(res_df["rmse"], 95)),
    "median_mae": float(res_df["mae"].median()),
    "median_chi2_mean": float(res_df["chi2_mean"].median()),
    "median_shape_rmse": float(res_df["shape_rmse"].median()),
}

res_df["vmax_bin"] = pd.cut(
    res_df["vmax_obs"],
    bins=[-np.inf, 80, 150, np.inf],
    labels=["low", "mid", "high"]
)

bin_summary = (
    res_df.groupby("vmax_bin", observed=True)[["rmse", "mae", "chi2_mean", "shape_rmse"]]
    .median()
    .reset_index()
)

best20 = res_df.sort_values("rmse").head(20).reset_index(drop=True)
worst20 = res_df.sort_values("rmse", ascending=False).head(20).reset_index(drop=True)

print()
print("=" * 60)
print("DERIVED SQRT-BRIDGE SUMMARY")
print("=" * 60)
for k, v in summary.items():
    print(f"{k:22s} = {v}")

print()
print("=" * 60)
print("MEDIAN METRICS BY VMAX BIN")
print("=" * 60)
print(bin_summary.to_string(index=False))

print()
print("=" * 60)
print("BEST 20 BY RMSE")
print("=" * 60)
print(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("=" * 60)
print("WORST 20 BY RMSE")
print("=" * 60)
print(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

res_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_results.csv"), index=False)
fail_df.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_failures.csv"), index=False)
bin_summary.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_bin_summary.csv"), index=False)
best20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_best20.csv"), index=False)
worst20.to_csv(os.path.join(OUTDIR, "derived_sqrt_bridge_worst20.csv"), index=False)

with open(os.path.join(OUTDIR, "derived_sqrt_bridge_summary.txt"), "w", encoding="utf-8") as f:
    f.write("DERIVED SQRT-BRIDGE SUMMARY\n")
    f.write("=" * 60 + "\n")
    for k, v in summary.items():
        f.write(f"{k:22s} = {v}\n")
    f.write("\nMEDIAN METRICS BY VMAX BIN\n")
    f.write("=" * 60 + "\n")
    f.write(bin_summary.to_string(index=False))
    f.write("\n\nBEST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(best20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))
    f.write("\n\nWORST 20 BY RMSE\n")
    f.write("=" * 60 + "\n")
    f.write(worst20[["galaxy", "vmax_obs", "V_flat_pred", "rmse", "mae", "shape_rmse"]].to_string(index=False))

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING SQRT-BRIDGE FIELD TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

DERIVED SQRT-BRIDGE SUMMARY
successful_galaxies    = 175
failed_galaxies        = 0
K_global               = 36582.793628590174
median_rmse            = 43.36683077126389
mean_rmse              = 57.61969794883952
p90_rmse               = 121.34847005642484
p95_rmse               = 148.87374190105436
median_mae             = 41.33523616131542
median_chi2_mean       = 108.24295608329402
median_shape_rmse      = 0.1035927421678175

MEDIAN METRICS BY VMAX BIN
vmax_bin      rmse       mae  chi2_mean  shape_rmse
     low 41.847865 39.337542 149.385139    0.093540
     mid 28.832875 28.342812  42.857838    0.073662
    high 77.419750 65.099313 212.123979    0.147856

BEST 20 BY RMSE
     galaxy  vmax_obs  V_flat_pred      rmse       mae  shape_rmse
   UGC04483      24.3    

In [ ]:
# ============================================================
# GLOBAL BALANCE / ENERGY-CLOSURE TEST
# ------------------------------------------------------------
# Purpose:
#   Test whether the missing amplitude V_flat is controlled by
#   global balance integrals of the surviving screened field.
#
# Starting field:
#   (1/r^2) d/dr (r^2 dm/dr) - (1/Rs^2)(m-m_inf) = -A rho_eff
#
# Define:
#   u(r) = m(r) - m_inf
#   U(r) = ∫_0^r u(s) ds
#
# New diagnostics:
#   E1 = ∫ (du/dr)^2 r^2 dr                 # gradient / field energy
#   E2 = ∫ (u^2 / Rs^2) r^2 dr              # screening energy
#   E3 = ∫ rho_eff(r) u(r) r^2 dr           # source-field coupling
#
# We then test simple no-fit amplitude laws:
#   V_flat^2 ~ C * E3
#   V_flat^2 ~ C * (E1 + E2)
#   V_flat^2 ~ C * E3 / (E1 + E2)
#   V_flat^4 ~ C * E3
#   V_flat^4 ~ C * (E1 + E2)
#   V_flat^4 ~ C * E3 / (E1 + E2)
#
# plus optional log-regression diagnostics across:
#   [U_inf, int_U, E1, E2, E3, E3/(E1+E2), r50]
#
# One-cell Colab script.
# ============================================================

import os, re, glob, zipfile, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_global_balance_test")
os.makedirs(OUTDIR, exist_ok=True)

ZIP_CANDIDATES = [
    "/content/Rotmod_LTG (4).zip",
    "/content/Rotmod_LTG.zip",
    os.path.join(WORKDIR, "Rotmod_LTG (4).zip"),
    os.path.join(WORKDIR, "Rotmod_LTG.zip"),
]

# ------------------------------------------------------------
# FIXED SURVIVING MODEL SETTINGS
# ------------------------------------------------------------

R_MIN = 1.0e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1.0e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def display_name_from_path(path):
    base = os.path.basename(path)
    base = re.sub(r"_rotmod\.dat$", "", base, flags=re.IGNORECASE)
    base = re.sub(r"\.dat$", "", base, flags=re.IGNORECASE)
    return base

def moving_average(y, win=7):
    y = np.asarray(y, float)
    if win <= 1:
        return y.copy()
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def safe_rmse(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))

def safe_mae(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if np.sum(m) == 0:
        return np.nan
    return float(np.mean(np.abs(a[m] - b[m])))

def bootstrap_rotmods():
    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if rotmod_files:
        return rotmod_files

    zip_path = None
    for zp in ZIP_CANDIDATES:
        if os.path.exists(zp):
            zip_path = zp
            break

    if zip_path is None:
        raise FileNotFoundError("No LTG rotmod directory or zip archive found.")

    print(f"Auto-extracting LTG rotmods from: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(ROTMOD_DIR)

    rotmod_files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
    if not rotmod_files:
        raise FileNotFoundError(f"No .dat files found in {ROTMOD_DIR} after extraction.")
    return rotmod_files

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#") or s.startswith("%") or s.startswith("//"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except Exception:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    if not rows:
        raise ValueError(f"No numeric rows found in {path}")

    max_cols = max(len(r) for r in rows)
    arr = np.full((len(rows), max_cols), np.nan)
    for i, row in enumerate(rows):
        arr[i, :len(row)] = row

    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = np.isfinite(r) & np.isfinite(vobs) & np.isfinite(ev)
    mask &= np.isfinite(vgas) & np.isfinite(vdisk) & np.isfinite(vbul)
    mask &= (r > 0) & (ev > 0)

    if np.sum(mask) == 0:
        raise ValueError("No valid points after masking.")

    order = np.argsort(r[mask])
    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_mass_derivative_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1.0e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)

    if np.max(rho_eff) <= 0:
        raise RuntimeError("Source proxy is identically zero.")

    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)
    return {"r": r, "rho_eff": rho_eff}

def solve_canonical_field_fd(r_eval, rho_r, rho_vals, Rs, m_inf, A_src):
    r_eval = np.asarray(r_eval, float)
    rmax = max(R_MAX, float(np.max(r_eval)) * 1.15)
    r = np.linspace(R_MIN, rmax, N_R)
    dr = r[1] - r[0]

    rho_pchip = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho0 = float(rho_vals[0])

    rho = np.empty_like(r)
    left = r <= rho_r[0]
    mid = (r > rho_r[0]) & (r < rho_r[-1])
    right = r >= rho_r[-1]
    rho[left] = rho0
    rho[mid] = rho_pchip(r[mid])
    rho[right] = 0.0
    rho = np.maximum(rho, 0.0)

    A = lil_matrix((N_R, N_R), dtype=float)
    b = np.zeros(N_R, dtype=float)

    A[0, 0] = 1.0
    A[0, 1] = -1.0
    b[0] = 0.0

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs**2)

    A[N_R - 1, N_R - 1] = 1.0
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)

    return {
        "r_grid": r,
        "m_grid": m,
        "rho_grid": rho,
    }

def compute_r50_from_u(r, u):
    U = cumulative_trapezoid(np.maximum(u, 0.0), r, initial=0.0)
    U_inf = float(np.max(U))
    if not np.isfinite(U_inf) or U_inf <= 0:
        return np.nan
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    if idx <= 0:
        return float(r[0])
    if idx >= len(r):
        return float(r[-1])
    x0, x1 = U[idx - 1], U[idx]
    y0, y1 = r[idx - 1], r[idx]
    if x1 == x0:
        return float(y0)
    return float(y0 + (target - x0) * (y1 - y0) / (x1 - x0))

def fit_single_constant(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
    if np.sum(m) == 0:
        return np.nan
    return float(np.sum(x[m] * y[m]) / np.sum(x[m] * x[m]))

def evaluate_amp_law(df, x_col, target="vflat2"):
    if target == "vflat2":
        y = df["vmax_obs"].values**2
    elif target == "vflat4":
        y = df["vmax_obs"].values**4
    else:
        raise ValueError("target must be vflat2 or vflat4")

    x = df[x_col].values
    C = fit_single_constant(x, y)

    if not np.isfinite(C):
        return None

    y_pred = C * x
    if target == "vflat2":
        vflat_pred = np.sqrt(np.maximum(y_pred, 0.0))
    else:
        vflat_pred = np.power(np.maximum(y_pred, 0.0), 0.25)

    amp_rmse = safe_rmse(df["vmax_obs"].values, vflat_pred)
    amp_mae = safe_mae(df["vmax_obs"].values, vflat_pred)

    return {
        "driver": x_col,
        "target": target,
        "C_global": C,
        "amp_rmse": amp_rmse,
        "amp_mae": amp_mae,
    }

def log_regression(X_cols, y_col, df):
    X = df[X_cols].copy()
    y = df[y_col].copy()

    mask = np.isfinite(y)
    for c in X_cols:
        mask &= np.isfinite(X[c]) & (X[c] > 0)
    mask &= (y > 0)

    X = X.loc[mask]
    y = y.loc[mask]

    if len(X) < len(X_cols) + 5:
        return None

    Xmat = np.column_stack([np.ones(len(X))] + [np.log(X[c].values) for c in X_cols])
    yvec = np.log(y.values)

    beta, *_ = np.linalg.lstsq(Xmat, yvec, rcond=None)

    yhat = np.exp(Xmat @ beta)
    rmse = safe_rmse(y.values, yhat)
    mae = safe_mae(y.values, yhat)

    out = {
        "formula_target": y_col,
        "n_used": int(len(X)),
        "intercept_exp": float(np.exp(beta[0])),
        "rmse": rmse,
        "mae": mae,
    }
    for i, c in enumerate(X_cols, start=1):
        out[f"coef_{c}"] = float(beta[i])
    return out

# ------------------------------------------------------------
# BUILD FIELD TABLE
# ------------------------------------------------------------

rotmod_files = bootstrap_rotmods()

rows = []
fails = []

print("=" * 70)
print("BUILDING GLOBAL BALANCE FIELD TABLE")
print("=" * 70)

for i, path in enumerate(rotmod_files, start=1):
    galaxy = display_name_from_path(path)

    try:
        rot = read_rotmod_file(path)
        src = build_mass_derivative_source(rot)

        r_obs = rot["r"]
        r_theory = np.linspace(
            max(R_MIN, float(np.min(r_obs)) * 0.5),
            max(float(np.max(r_obs)) * 1.10, 1.0),
            N_R
        )

        solved = solve_canonical_field_fd(
            r_eval=r_theory,
            rho_r=src["r"],
            rho_vals=src["rho_eff"],
            Rs=Rs_fixed,
            m_inf=m_inf,
            A_src=A_src
        )

        r = solved["r_grid"]
        m = solved["m_grid"]
        rho = solved["rho_grid"]

        u = np.maximum(m - m_inf, 0.0)
        U = cumulative_trapezoid(u, r, initial=0.0)
        U = np.maximum(U, 0.0)

        if not np.isfinite(np.max(U)) or np.max(U) <= 0:
            raise RuntimeError("Invalid U profile")

        du_dr = np.gradient(u, r)

        # Global balance integrals
        E1 = float(trapezoid((du_dr**2) * (r**2), r))
        E2 = float(trapezoid((u**2 / (Rs_fixed**2)) * (r**2), r))
        E3 = float(trapezoid(rho * u * (r**2), r))

        int_U = float(trapezoid(U, r))
        U_inf = float(np.max(U))
        r50 = compute_r50_from_u(r, u)

        rows.append({
            "galaxy": galaxy,
            "vmax_obs": float(np.max(rot["vobs"])),
            "n_points": int(len(rot["r"])),
            "U_inf": U_inf,
            "int_U": int_U,
            "r50": r50,
            "E1_grad": E1,
            "E2_screen": E2,
            "E3_source_coupling": E3,
            "E_balance_sum": E1 + E2,
            "E_balance_ratio": E3 / max(E1 + E2, 1.0e-30),
        })

    except Exception as e:
        fails.append({"galaxy": galaxy, "reason": str(e)})

    if i % 20 == 0:
        print(f"Processed {i}/{len(rotmod_files)} files")

feat_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(fails)

if len(feat_df) == 0:
    raise RuntimeError("No successful galaxies processed.")

# ------------------------------------------------------------
# GLOBAL NO-FIT AMPLITUDE TESTS
# ------------------------------------------------------------

tests = []
for driver in [
    "U_inf",
    "int_U",
    "E1_grad",
    "E2_screen",
    "E3_source_coupling",
    "E_balance_sum",
    "E_balance_ratio",
]:
    out2 = evaluate_amp_law(feat_df, driver, target="vflat2")
    out4 = evaluate_amp_law(feat_df, driver, target="vflat4")
    if out2 is not None:
        tests.append(out2)
    if out4 is not None:
        tests.append(out4)

tests_df = pd.DataFrame(tests).sort_values(["amp_rmse", "amp_mae"]).reset_index(drop=True)

# ------------------------------------------------------------
# OPTIONAL LOG-REGRESSION DIAGNOSTICS
# ------------------------------------------------------------

log_models = []

candidate_sets = [
    ["U_inf"],
    ["int_U"],
    ["E3_source_coupling"],
    ["E_balance_ratio"],
    ["int_U", "r50", "U_inf"],
    ["E3_source_coupling", "r50"],
    ["E_balance_sum", "E3_source_coupling"],
    ["E_balance_ratio", "r50"],
    ["E1_grad", "E2_screen", "E3_source_coupling"],
    ["int_U", "r50", "E3_source_coupling"],
]

for cols in candidate_sets:
    out = log_regression(cols, "vmax_obs", feat_df)
    if out is not None:
        out["model_cols"] = ", ".join(cols)
        log_models.append(out)

log_df = pd.DataFrame(log_models).sort_values(["rmse", "mae"]).reset_index(drop=True)

# ------------------------------------------------------------
# PRINT SUMMARY
# ------------------------------------------------------------

print()
print("=" * 70)
print("GLOBAL BALANCE FEATURE SUMMARY")
print("=" * 70)
print(f"successful_galaxies = {len(feat_df)}")
print(f"failed_galaxies     = {len(fail_df)}")

print()
print("=" * 70)
print("BEST GLOBAL NO-FIT AMPLITUDE LAWS")
print("=" * 70)
if len(tests_df):
    print(tests_df.to_string(index=False))
else:
    print("No valid amplitude tests.")

print()
print("=" * 70)
print("BEST LOG-REGRESSION DIAGNOSTICS")
print("=" * 70)
if len(log_df):
    show_cols = ["model_cols", "intercept_exp", "rmse", "mae"] + [c for c in log_df.columns if c.startswith("coef_")]
    print(log_df[show_cols].to_string(index=False))
else:
    print("No valid log-regression models.")

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

feat_df.to_csv(os.path.join(OUTDIR, "global_balance_features.csv"), index=False)
fail_df.to_csv(os.path.join(OUTDIR, "global_balance_failures.csv"), index=False)
tests_df.to_csv(os.path.join(OUTDIR, "global_balance_nofit_tests.csv"), index=False)
log_df.to_csv(os.path.join(OUTDIR, "global_balance_log_diagnostics.csv"), index=False)

with open(os.path.join(OUTDIR, "global_balance_summary.txt"), "w", encoding="utf-8") as f:
    f.write("GLOBAL BALANCE FEATURE SUMMARY\n")
    f.write("=" * 70 + "\n")
    f.write(f"successful_galaxies = {len(feat_df)}\n")
    f.write(f"failed_galaxies     = {len(fail_df)}\n\n")

    f.write("BEST GLOBAL NO-FIT AMPLITUDE LAWS\n")
    f.write("=" * 70 + "\n")
    if len(tests_df):
        f.write(tests_df.to_string(index=False))
    else:
        f.write("No valid amplitude tests.")
    f.write("\n\n")

    f.write("BEST LOG-REGRESSION DIAGNOSTICS\n")
    f.write("=" * 70 + "\n")
    if len(log_df):
        show_cols = ["model_cols", "intercept_exp", "rmse", "mae"] + [c for c in log_df.columns if c.startswith("coef_")]
        f.write(log_df[show_cols].to_string(index=False))
    else:
        f.write("No valid log-regression models.")
    f.write("\n")

print()
print("Saved outputs in:")
print(OUTDIR)

BUILDING GLOBAL BALANCE FIELD TABLE
Processed 20/175 files
Processed 40/175 files
Processed 60/175 files
Processed 80/175 files
Processed 100/175 files
Processed 120/175 files
Processed 140/175 files
Processed 160/175 files

GLOBAL BALANCE FEATURE SUMMARY
successful_galaxies = 175
failed_galaxies     = 0

BEST GLOBAL NO-FIT AMPLITUDE LAWS
            driver target     C_global   amp_rmse   amp_mae
             int_U vflat2 3.905848e+02  79.235962 54.697965
             U_inf vflat2 2.299660e+04  80.765368 57.009928
           E1_grad vflat4 8.384261e+09  83.589439 62.504562
   E_balance_ratio vflat2 2.300375e+03  84.063098 74.686174
             int_U vflat4 3.075622e+07  86.156884 74.770500
             U_inf vflat4 1.629426e+09  87.546771 76.096023
E3_source_coupling vflat4 2.695407e+07  88.187893 59.720184
     E_balance_sum vflat4 2.695417e+08  88.188070 59.720336
         E2_screen vflat4 2.750834e+08  90.018506 61.378212
   E_balance_ratio vflat4 1.273481e+08 100.930071 90.568111

In [ ]:
# ------------------------------------------------------------
# ENERGY / LENGTH-SCALE AMPLITUDE LAWS
# ------------------------------------------------------------

def test_amp_relation(df, driver, power=2):
    x = df[driver].values
    v = df["vmax_obs"].values

    m = np.isfinite(x) & np.isfinite(v) & (x > 0) & (v > 0)
    if np.sum(m) == 0:
        return None

    if power == 2:
        y = v[m]**2
    elif power == 4:
        y = v[m]**4
    else:
        raise ValueError

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(np.maximum(y_pred, 0))
    else:
        v_pred = np.power(np.maximum(y_pred, 0), 0.25)

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))

    return {
        "driver": driver,
        "power": power,
        "C_global": C,
        "amp_rmse": rmse,
        "amp_mae": mae,
    }

# Build new drivers
feat_df["E_over_r50"] = feat_df["E_balance_sum"] / np.maximum(feat_df["r50"], 1e-6)
feat_df["E3_over_r50"] = feat_df["E3_source_coupling"] / np.maximum(feat_df["r50"], 1e-6)
feat_df["intU_over_r50"] = feat_df["int_U"] / np.maximum(feat_df["r50"], 1e-6)

tests2 = []

for drv in ["E3_over_r50", "E_over_r50", "intU_over_r50"]:
    for p in [2, 4]:
        out = test_amp_relation(feat_df, drv, power=p)
        if out:
            tests2.append(out)

tests2_df = pd.DataFrame(tests2).sort_values(["amp_rmse", "amp_mae"]).reset_index(drop=True)

print()
print("=" * 70)
print("ENERGY / LENGTH-SCALE AMPLITUDE TESTS")
print("=" * 70)
print(tests2_df.to_string(index=False))

tests2_df.to_csv(os.path.join(OUTDIR, "energy_length_amplitude_tests.csv"), index=False)


ENERGY / LENGTH-SCALE AMPLITUDE TESTS
       driver  power     C_global   amp_rmse   amp_mae
intU_over_r50      2 1.821368e+03  79.813911 66.349374
   E_over_r50      4 2.627637e+09  81.307425 58.356850
  E3_over_r50      4 2.627617e+08  81.307513 58.357092
intU_over_r50      4 1.148377e+08  98.377114 88.444586
  E3_over_r50      2 3.012415e+03 106.112388 79.475328
   E_over_r50      2 3.012433e+04 106.112613 79.475728


In [ ]:
# ============================================================
# FLUX AT SCREENING RADIUS TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_flux_rs_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r_grid, m_grid = solve_field(rho_r, rho_vals)

    u = np.maximum(m_grid - m_inf, 0)
    du_dr = np.gradient(u, r_grid)

    # Flux at screening radius Rs
    flux_rs = (Rs_fixed**2) * np.interp(Rs_fixed, r_grid, du_dr)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "flux_rs": flux_rs,
    })

df = pd.DataFrame(rows)

print("\nFLUX AT Rs TEST")
print("="*40)

for p in [2, 4]:
    rmse, mae = test_relation(df["flux_rs"].values, df["vmax"].values, p)
    print(f"flux_rs power={p} RMSE={rmse:.2f} MAE={mae:.2f}")

df.to_csv(os.path.join(OUTDIR, "flux_rs_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

FLUX AT Rs TEST
flux_rs power=2 RMSE=108.70 MAE=81.82
flux_rs power=4 RMSE=102.05 MAE=79.49

Saved to: /content/mts_realdata_workspace_v2/mts_flux_rs_test


In [ ]:
# ============================================================
# MTS FULL AMPLITUDE CLOSURE TEST
# Computes all integrals + tests scalars, flux, ratios
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_full_amplitude_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def compute_r50(r, u):
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    return r[idx] if idx < len(r) else r[-1]

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)
    r50 = compute_r50(r, u)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    flux_rs = (Rs_fixed**2) * np.interp(Rs_fixed, r, du_dr)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "r50": r50,
        "E1": E1,
        "E2": E2,
        "E3": E3,
        "flux_rs": flux_rs,
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# BUILD RATIO DRIVERS
# ------------------------------------------------------------

df["intU_over_E3"] = df["int_U"] / np.maximum(df["E3"], 1e-8)
df["intU_r50_over_E3"] = (df["int_U"] * df["r50"]) / np.maximum(df["E3"], 1e-8)
df["Uinf_r50_over_E3"] = (df["U_inf"] * df["r50"]) / np.maximum(df["E3"], 1e-8)

# ------------------------------------------------------------
# TEST EVERYTHING
# ------------------------------------------------------------

print("\nAMPLITUDE LAW TESTS")
print("="*50)

drivers = [
    "U_inf",
    "int_U",
    "E1",
    "E2",
    "E3",
    "flux_rs",
    "intU_over_E3",
    "intU_r50_over_E3",
    "Uinf_r50_over_E3"
]

for drv in drivers:
    for p in [2, 4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "full_amplitude_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

AMPLITUDE LAW TESTS
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E1                   power=2 RMSE=  104.06 MAE=   76.05
E1                   power=4 RMSE=   83.11 MAE=   61.26
E2                   power=2 RMSE=  121.75 MAE=   98.94
E2                   power=4 RMSE=   89.67 MAE=   61.02
E3                   power=2 RMSE=  120.34 MAE=   97.18
E3                   power=4 RMSE=   87.95 MAE=   59.62
flux_rs              power=2 RMSE=  108.70 MAE=   81.82
flux_rs              power=4 RMSE=  102.05 MAE=   79.49
intU_over_E3         power=2 RMSE=  123.47 MAE=   95.25
intU_over_E3         power=4 RMSE=  100.63 MAE=   76.54
intU_r50_over_E3     power=2 RMSE=  104.59 MAE=   76.89
intU_r50_over_E3     power=4 RMSE=   94.18 MAE=   78.08
Uinf_r50_over_E3     power=2 RMSE=  103.42 MAE=   76.06
Uinf

In [ ]:
# ============================================================
# MTS ENERGY BALANCE AMPLITUDE TEST
# Full one-cell script
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_energy_balance_test")
os.makedirs(OUTDIR, exist_ok=True)

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "E1": E1,
        "E2": E2,
        "E3": E3
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# ENERGY RATIO DRIVERS
# ------------------------------------------------------------

df["E1_over_E3"] = df["E1"] / np.maximum(df["E3"], 1e-8)
df["E2_over_E3"] = df["E2"] / np.maximum(df["E3"], 1e-8)
df["E12_over_E3"] = (df["E1"] + df["E2"]) / np.maximum(df["E3"], 1e-8)

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

print("\nENERGY BALANCE AMPLITUDE TESTS")
print("="*50)

drivers = ["E1_over_E3", "E2_over_E3", "E12_over_E3"]

for drv in drivers:
    for p in [2, 4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "energy_balance_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

ENERGY BALANCE AMPLITUDE TESTS
E1_over_E3           power=2 RMSE=  103.21 MAE=   80.26
E1_over_E3           power=4 RMSE=   98.25 MAE=   83.79
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
E12_over_E3          power=2 RMSE=   84.06 MAE=   74.68
E12_over_E3          power=4 RMSE=  100.93 MAE=   90.57

Saved to: /content/mts_realdata_workspace_v2/mts_energy_balance_test


In [ ]:
# ============================================================
# MTS FINAL COMBINED MODEL (END-TO-END TEST)
# Field + Energy Amplitude + Shape Law
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_final_combined_model")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def safe_rmse(a, b):
    return np.sqrt(np.mean((a - b)**2))

def safe_mae(a, b):
    return np.mean(np.abs(a - b))

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Running FINAL combined model...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)

    # --- Energy amplitude law ---
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    amp_driver = E2 / max(E3, 1e-8)

    rows.append({
        "galaxy": os.path.basename(f),
        "r_grid": r,
        "U_grid": U,
        "U_inf": U_inf,
        "amp_driver": amp_driver,
        "r_obs": rot["r"],
        "v_obs": rot["vobs"],
        "e_obs": rot["ev"],
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# FIT GLOBAL K FROM AMPLITUDE LAW
# Vflat^2 = K * amp_driver
# ------------------------------------------------------------

num = 0.0
den = 0.0

for _, row in df.iterrows():
    Vflat_obs = np.max(row["v_obs"])
    x = row["amp_driver"]
    y = Vflat_obs**2
    num += x * y
    den += x * x

K_global = num / den
print("Global amplitude constant K =", K_global)

# ------------------------------------------------------------
# EVALUATE FULL MODEL
# ------------------------------------------------------------

res_rows = []

for _, row in df.iterrows():
    r = row["r_grid"]
    U = row["U_grid"]
    U_inf = row["U_inf"]

    V_flat = np.sqrt(K_global * row["amp_driver"])
    V_theory = V_flat * np.sqrt(np.maximum(U / U_inf, 0))

    V_pred = interp1d(r, V_theory, bounds_error=False, fill_value="extrapolate")(row["r_obs"])

    rmse = safe_rmse(row["v_obs"], V_pred)
    mae = safe_mae(row["v_obs"], V_pred)

    res_rows.append({
        "galaxy": row["galaxy"],
        "vmax_obs": np.max(row["v_obs"]),
        "vflat_pred": V_flat,
        "rmse": rmse,
        "mae": mae
    })

res_df = pd.DataFrame(res_rows)

print("\nFINAL MODEL SUMMARY")
print("="*50)
print("Galaxies:", len(res_df))
print("Median RMSE:", res_df["rmse"].median())
print("Mean RMSE:", res_df["rmse"].mean())
print("P90 RMSE:", np.percentile(res_df["rmse"], 90))

res_df.to_csv(os.path.join(OUTDIR, "final_model_results.csv"), index=False)

print("\nSaved to:", OUTDIR)

Running FINAL combined model...
Global amplitude constant K = 324911.5567336142

FINAL MODEL SUMMARY
Galaxies: 175
Median RMSE: 61.30444135585258
Mean RMSE: 63.140371080233045
P90 RMSE: 93.94683968458814

Saved to: /content/mts_realdata_workspace_v2/mts_final_combined_model


In [ ]:
# ============================================================
# MTS MASTER TEST — FULL PIPELINE + ALL AMPLITUDE LAWS
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_master_test")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.full_like(vobs, 5.0)
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def compute_r50(r, u):
    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    target = 0.5 * U_inf
    idx = np.searchsorted(U, target)
    return r[idx] if idx < len(r) else r[-1]

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 0)
    du_dr = np.gradient(u, r)

    U = cumulative_trapezoid(u, r, initial=0)
    U_inf = np.max(U)
    int_U = trapezoid(U, r)
    r50 = compute_r50(r, u)

    E1 = trapezoid((du_dr**2) * r**2, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "r50": r50,
        "E1": E1,
        "E2": E2,
        "E3": E3
    })

df = pd.DataFrame(rows)

# ------------------------------------------------------------
# BUILD DRIVERS
# ------------------------------------------------------------

df["E2_over_E3"] = df["E2"] / np.maximum(df["E3"],1e-8)
df["E2_over_E3_r50"] = df["E2"] / (np.maximum(df["E3"],1e-8) * np.maximum(df["r50"],1e-8))
df["Uinf_over_r50"] = df["U_inf"] / np.maximum(df["r50"],1e-8)
df["intU_over_r50"] = df["int_U"] / np.maximum(df["r50"],1e-8)

print("\nAMPLITUDE LAW TESTS")
print("="*60)

drivers = [
    "U_inf",
    "int_U",
    "E2_over_E3",
    "E2_over_E3_r50",
    "Uinf_over_r50",
    "intU_over_r50"
]

for drv in drivers:
    for p in [2,4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

Processing galaxies...

AMPLITUDE LAW TESTS
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
E2_over_E3_r50       power=2 RMSE=   91.42 MAE=   77.13
E2_over_E3_r50       power=4 RMSE=   99.40 MAE=   88.70
Uinf_over_r50        power=2 RMSE=   85.31 MAE=   70.92
Uinf_over_r50        power=4 RMSE=   95.86 MAE=   85.58
intU_over_r50        power=2 RMSE=   86.38 MAE=   71.66
intU_over_r50        power=4 RMSE=   95.70 MAE=   85.36


In [ ]:
# ------------------------------------------------------------
# COMBINED AMPLITUDE DRIVER
# ------------------------------------------------------------

df["E2_times_Uinf_over_E3"] = (df["E2"] * df["U_inf"]) / np.maximum(df["E3"],1e-8)

print("\nCOMBINED AMPLITUDE TEST")
print("="*60)

for p in [2,4]:
    rmse, mae = test_relation(df["E2_times_Uinf_over_E3"].values, df["vmax"].values, p)
    print(f"{'E2*Uinf/E3':20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")


COMBINED AMPLITUDE TEST
E2*Uinf/E3           power=2 RMSE=   82.53 MAE=   57.97
E2*Uinf/E3           power=4 RMSE=   85.99 MAE=   73.03


In [ ]:
# ============================================================
# MTS SCREENING-LENGTH AMPLITUDE TEST (CORRECTED)
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
OUTDIR = os.path.join(WORKDIR, "mts_Rs_test")
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# MODEL PARAMETERS
# ------------------------------------------------------------

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

Rs_fixed = 1.5
m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - 1.0 / (Rs_fixed**2)

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i] - m_inf / (Rs_fixed**2)

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

def estimate_Rs(r, u):
    mask = (r > 0.6*np.max(r)) & (u > 0)
    if np.sum(mask) < 5:
        return np.nan
    slope = np.polyfit(r[mask], np.log(u[mask]), 1)[0]
    if slope >= 0:
        return np.nan
    return -1.0 / slope

def test_relation(x, v, power):
    m = (x > 0) & (v > 0)
    if power == 2:
        y = v[m]**2
    else:
        y = v[m]**4

    C = np.sum(x[m] * y) / np.sum(x[m] * x[m])
    y_pred = C * x[m]

    if power == 2:
        v_pred = np.sqrt(y_pred)
    else:
        v_pred = y_pred**0.25

    rmse = np.sqrt(np.mean((v[m] - v_pred)**2))
    mae = np.mean(np.abs(v[m] - v_pred))
    return rmse, mae

# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
rows = []

print("Processing galaxies...")

for f in files:
    rot = read_rotmod_file(f)
    rho_r, rho_vals = build_source(rot)
    r, m, rho = solve_field(rho_r, rho_vals)

    u = np.maximum(m - m_inf, 1e-12)
    U = cumulative_trapezoid(u, r, initial=0)

    U_inf = np.max(U)
    int_U = trapezoid(U, r)

    du_dr = np.gradient(u, r)
    E2 = trapezoid((u**2 / Rs_fixed**2) * r**2, r)
    E3 = trapezoid(rho * u * r**2, r)

    r50 = r[np.searchsorted(U, 0.5*U_inf)]
    Rs_est = estimate_Rs(r, u)

    rows.append({
        "galaxy": os.path.basename(f),
        "vmax": np.max(rot["vobs"]),
        "U_inf": U_inf,
        "int_U": int_U,
        "E2_over_E3": E2 / max(E3,1e-8),
        "Rs_est": Rs_est
    })

df = pd.DataFrame(rows)
df["inv_Rs"] = 1.0 / df["Rs_est"]

print("\nAMPLITUDE LAW COMPARISON")
print("="*60)

drivers = ["U_inf", "int_U", "E2_over_E3", "inv_Rs"]

for drv in drivers:
    for p in [2,4]:
        rmse, mae = test_relation(df[drv].values, df["vmax"].values, p)
        print(f"{drv:20s} power={p} RMSE={rmse:8.2f} MAE={mae:8.2f}")

df.to_csv(os.path.join(OUTDIR, "Rs_amplitude_test.csv"), index=False)

print("\nSaved to:", OUTDIR)

Processing galaxies...

AMPLITUDE LAW COMPARISON
U_inf                power=2 RMSE=   80.95 MAE=   58.84
U_inf                power=4 RMSE=   89.09 MAE=   77.36
int_U                power=2 RMSE=   81.33 MAE=   60.07
int_U                power=4 RMSE=   89.76 MAE=   78.25
E2_over_E3           power=2 RMSE=   79.12 MAE=   70.21
E2_over_E3           power=4 RMSE=   98.73 MAE=   89.26
inv_Rs               power=2 RMSE=   74.68 MAE=   54.49
inv_Rs               power=4 RMSE=   78.57 MAE=   66.53

Saved to: /content/mts_realdata_workspace_v2/mts_Rs_test


In [ ]:
# ============================================================
# NONLINEAR / DENSITY-DEPENDENT SCREENING TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.5   # screening strength — try 0.1, 0.3, 0.5, 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field_nonlinear(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# MAIN TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []

print("Running nonlinear screening model...")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field_nonlinear(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)

        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

    except:
        pass

print("\nNONLINEAR MODEL SUMMARY")
print("Galaxies:", len(rmse_list))
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

Running nonlinear screening model...

NONLINEAR MODEL SUMMARY
Galaxies: 175
Median RMSE: 43.0261836555489
Mean RMSE: 56.73852552405732
P90 RMSE: 109.05932633574832


In [ ]:
# ============================================================
# NONLINEAR SCREENING AUTO-SWEEP
# Tests multiple screening strengths automatically
# No editing required
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

BETA_LIST = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.5]

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field_nonlinear(rho_r, rho_vals, BETA):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("NONLINEAR SCREENING SWEEP")
print("====================================")

for BETA in BETA_LIST:
    rmse_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field_nonlinear(rho_r, rho_vals, BETA)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            V_flat = np.max(rot["vobs"])
            V_model = V_flat * (U / np.max(U))

            V_interp = np.interp(rot["r"], r, V_model)
            rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
            rmse_list.append(rmse)

        except:
            pass

    print(f"BETA = {BETA:4.2f} | Median RMSE = {np.median(rmse_list):6.2f} | Mean RMSE = {np.mean(rmse_list):6.2f}")

NONLINEAR SCREENING SWEEP
BETA = 0.05 | Median RMSE =  40.93 | Mean RMSE =  52.83
BETA = 0.10 | Median RMSE =  41.34 | Mean RMSE =  53.58
BETA = 0.20 | Median RMSE =  41.82 | Mean RMSE =  54.69
BETA = 0.30 | Median RMSE =  42.24 | Mean RMSE =  55.51
BETA = 0.50 | Median RMSE =  43.03 | Mean RMSE =  56.74
BETA = 0.70 | Median RMSE =  43.66 | Mean RMSE =  57.64
BETA = 1.00 | Median RMSE =  44.25 | Mean RMSE =  58.65
BETA = 1.50 | Median RMSE =  44.86 | Mean RMSE =  59.85


In [ ]:
# ============================================================
# FIELD ENERGY AMPLITUDE TEST (NONLINEAR FIELD)
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid, trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.05   # best value from sweep

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m, rho

# ------------------------------------------------------------
# ENERGY TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

E_list = []
V_list = []

print("FIELD ENERGY AMPLITUDE TEST")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m, rho = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        du = np.gradient(u, r)

        # Field energy terms
        E_grad = trapezoid((du**2) * r**2, r)
        E_screen = trapezoid((rho * u**2) * r**2, r)
        E_source = trapezoid((rho * u) * r**2, r)

        E_total = E_grad + BETA * E_screen + A_src * E_source

        V_flat = np.max(rot["vobs"])

        E_list.append(E_total)
        V_list.append(V_flat**2)

    except:
        pass

E = np.array(E_list)
V2 = np.array(V_list)

C = np.sum(E * V2) / np.sum(E * E)
V_pred = np.sqrt(C * E)

rmse = np.sqrt(np.mean((np.sqrt(V2) - V_pred)**2))
mae = np.mean(np.abs(np.sqrt(V2) - V_pred))

print("\nENERGY AMPLITUDE RESULT")
print("RMSE:", rmse)
print("MAE :", mae)

FIELD ENERGY AMPLITUDE TEST

ENERGY AMPLITUDE RESULT
RMSE: 122.21977338847826
MAE : 99.63356046920654


In [ ]:
# ============================================================
# MEASURE EFFECTIVE Rs FROM NONLINEAR FIELD
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
BETA  = 0.05

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)
        c0 = -(c_minus + c_plus) - BETA * rho[i]

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# MEASURE Rs
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

Rs_list = []
M_list = []
V_list = []

print("MEASURING EFFECTIVE Rs")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        # Define Rs as radius where U reaches 63% of saturation
        target = 0.63 * U_inf
        idx = np.searchsorted(U, target)
        Rs_eff = r[idx]

        # Baryonic mass proxy
        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        Rs_list.append(Rs_eff)
        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

Rs = np.array(Rs_list)
M = np.array(M_list)
V = np.array(V_list)

# Power law slopes
slope_Rs_M = np.polyfit(np.log(M), np.log(Rs), 1)[0]
slope_V4_M = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("\nSCALING RESULTS")
print("Rs vs M slope:", slope_Rs_M)
print("V^4 vs M slope:", slope_V4_M)

MEASURING EFFECTIVE Rs

SCALING RESULTS
Rs vs M slope: 0.07355623288897552
V^4 vs M slope: 0.745432302582731


In [ ]:
# ============================================================
# CUMULATIVE MASS SCREENING TEST
# Screening depends on enclosed mass, not density
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
M0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    # Enclosed mass
    M_enc = cumulative_trapezoid(rho * r**2, r, initial=0)

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (M_enc[i] + M0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []

print("CUMULATIVE-MASS SCREENING MODEL")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)

        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

    except:
        pass

print("\nMODEL RESULT")
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

CUMULATIVE-MASS SCREENING MODEL

MODEL RESULT
Median RMSE: 25.81479436536363
Mean RMSE: 36.65431547646416
P90 RMSE: 85.57752042115796


In [ ]:
# ============================================================
# SCREENING LAW SWEEP
# Tests multiple cumulative screening laws
# ============================================================

import os, re, glob, warnings
import numpy as np

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
M0    = 1.0
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

# ------------------------------------------------------------
# FIELD SOLVER WITH DIFFERENT SCREENING LAWS
# ------------------------------------------------------------

def solve_field(rho_r, rho_vals, mode):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    M_enc = cumulative_trapezoid(rho * r**2, r, initial=0)

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    u_tmp = np.zeros_like(r) + 1.0  # for field-dependent screening

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        if mode == "mass":
            screen = GAMMA / (M_enc[i] + M0)

        elif mode == "sqrt_mass":
            screen = GAMMA / (np.sqrt(M_enc[i] + M0))

        elif mode == "mass_pow_075":
            screen = GAMMA / ((M_enc[i] + M0)**0.75)

        elif mode == "field":
            screen = GAMMA / (u_tmp[i] + U0)

        elif mode == "mixed":
            screen = GAMMA / (np.sqrt((M_enc[i] + M0) * (u_tmp[i] + U0)))

        else:
            screen = 0.0

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN TESTS
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))
modes = ["mass", "sqrt_mass", "mass_pow_075", "field", "mixed"]

print("SCREENING LAW SWEEP")
print("========================================")

for mode in modes:
    rmse_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field(rho_r, rho_vals, mode)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            V_flat = np.max(rot["vobs"])
            V_model = V_flat * (U / np.max(U))

            V_interp = np.interp(rot["r"], r, V_model)
            rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
            rmse_list.append(rmse)

        except:
            pass

    print(f"{mode:15s} | Median RMSE = {np.median(rmse_list):6.2f} | Mean RMSE = {np.mean(rmse_list):6.2f}")

SCREENING LAW SWEEP
mass            | Median RMSE =  25.81 | Mean RMSE =  36.65
sqrt_mass       | Median RMSE =  16.33 | Mean RMSE =  27.49
mass_pow_075    | Median RMSE =  21.47 | Mean RMSE =  32.16
field           | Median RMSE =  12.28 | Mean RMSE =  21.52
mixed           | Median RMSE =  16.92 | Mean RMSE =  28.11


In [ ]:
# ============================================================
# BTFR SLOPE TEST FOR FIELD-SCREENING MODEL
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
V_list = []

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

M = np.array(M_list)
V = np.array(V_list)

slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("BTFR slope (V^4 vs M):", slope)

BTFR slope (V^4 vs M): 0.745432302582731


In [ ]:
# ============================================================
# NONLINEAR DIFFUSION (p-LAPLACIAN) FIELD TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

A_src = 0.5

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

# ------------------------------------------------------------
# NONLINEAR DIFFUSION SOLVER
# ------------------------------------------------------------

def solve_nonlinear_diffusion(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    # Initialize u
    u = np.ones_like(r) * 0.1

    # Iterative solve
    for _ in range(200):
        du = np.gradient(u, r)
        flux = u * du
        dflux = np.gradient(r**2 * flux, r) / (r**2 + 1e-8)

        u_new = u + 0.1 * (-A_src * rho - dflux)

        u_new = np.maximum(u_new, 0)
        u = 0.5 * u + 0.5 * u_new

    return r, u

# ------------------------------------------------------------
# RUN TEST
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

rmse_list = []
M_list = []
V_list = []

print("NONLINEAR DIFFUSION MODEL")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, u = solve_nonlinear_diffusion(rho_r, rho_vals)

        U = cumulative_trapezoid(u, r, initial=0)
        if np.max(U) <= 0:
            continue

        V_flat = np.max(rot["vobs"])
        V_model = V_flat * (U / np.max(U))

        V_interp = np.interp(rot["r"], r, V_model)
        rmse = np.sqrt(np.mean((rot["vobs"] - V_interp)**2))
        rmse_list.append(rmse)

        # mass proxy
        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        M_list.append(M_bar)
        V_list.append(V_flat)

    except:
        pass

print("\nMODEL FIT")
print("Median RMSE:", np.median(rmse_list))
print("Mean RMSE:", np.mean(rmse_list))
print("P90 RMSE:", np.percentile(rmse_list, 90))

# BTFR slope
M = np.array(M_list)
V = np.array(V_list)
slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("\nBTFR slope (V^4 vs M):", slope)

NONLINEAR DIFFUSION MODEL

MODEL FIT
Median RMSE: nan
Mean RMSE: nan
P90 RMSE: nan

BTFR slope (V^4 vs M): 0.745432302582731


In [ ]:
# ============================================================
# U_inf vs Mass Scaling Test
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 1.0
UPS_BUL  = 1.0
R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def vbar2_from_rot(rot):
    return np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

def build_source(rot):
    r = rot["r"]
    vbar2 = vbar2_from_rot(rot)

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
Uinf_list = []
V_list = []

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 0)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = np.max(U)

        vbar2 = vbar2_from_rot(rot)
        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        V_flat = np.max(rot["vobs"])

        M_list.append(M_bar)
        Uinf_list.append(U_inf)
        V_list.append(V_flat)

    except:
        pass

M = np.array(M_list)
Uinf = np.array(Uinf_list)
V = np.array(V_list)

alpha = np.polyfit(np.log(M), np.log(Uinf), 1)[0]
beta = np.polyfit(np.log(M), np.log(V**4), 1)[0]

print("U_inf vs M slope:", alpha)
print("V^4 vs M slope:", beta)

U_inf vs M slope: nan
V^4 vs M slope: 0.745432302582731


In [ ]:
# ============================================================
# MASS-TO-LIGHT SWEEP + BTFR SLOPE TEST
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10
GAMMA = 0.5
U0    = 1.0

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

UPS_DISK_LIST = [0.3, 0.5, 0.7, 1.0]
UPS_BUL_LIST  = [0.5, 1.0, 1.5]

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot, UPS_DISK, UPS_BUL):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff, vbar2

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("MASS-TO-LIGHT SWEEP (BTFR SLOPE)")
print("==========================================")

for UPS_DISK in UPS_DISK_LIST:
    for UPS_BUL in UPS_BUL_LIST:

        M_list = []
        V_list = []

        for f in files:
            try:
                rot = read_rotmod_file(f)
                rho_r, rho_vals, vbar2 = build_source(rot, UPS_DISK, UPS_BUL)
                r, m = solve_field(rho_r, rho_vals)

                u = np.maximum(m - m_inf, 0)
                U = cumulative_trapezoid(u, r, initial=0)
                if np.max(U) <= 0:
                    continue

                M_bar = np.trapz(vbar2 * rot["r"], rot["r"])
                V_flat = np.max(rot["vobs"])

                M_list.append(M_bar)
                V_list.append(V_flat)

            except:
                pass

        M = np.array(M_list)
        V = np.array(V_list)

        slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

        print(f"UPS_DISK={UPS_DISK:.1f}, UPS_BUL={UPS_BUL:.1f} | BTFR slope = {slope:.3f}")

MASS-TO-LIGHT SWEEP (BTFR SLOPE)
UPS_DISK=0.3, UPS_BUL=0.5 | BTFR slope = 0.799
UPS_DISK=0.3, UPS_BUL=1.0 | BTFR slope = 0.785
UPS_DISK=0.3, UPS_BUL=1.5 | BTFR slope = 0.776
UPS_DISK=0.5, UPS_BUL=0.5 | BTFR slope = 0.788
UPS_DISK=0.5, UPS_BUL=1.0 | BTFR slope = 0.779
UPS_DISK=0.5, UPS_BUL=1.5 | BTFR slope = 0.770
UPS_DISK=0.7, UPS_BUL=0.5 | BTFR slope = 0.780
UPS_DISK=0.7, UPS_BUL=1.0 | BTFR slope = 0.773
UPS_DISK=0.7, UPS_BUL=1.5 | BTFR slope = 0.767
UPS_DISK=1.0, UPS_BUL=0.5 | BTFR slope = 0.772
UPS_DISK=1.0, UPS_BUL=1.0 | BTFR slope = 0.767
UPS_DISK=1.0, UPS_BUL=1.5 | BTFR slope = 0.762


In [ ]:
# ============================================================
# MEASURE EFFECTIVE Rs FROM NONLINEAR FIELD
# ============================================================

import os, re, glob, warnings
import numpy as np
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

m_inf = 0.02
A_src = 0.10

# Nonlinear screening parameters
GAMMA = 0.5
U0    = 1.0

UPS_DISK = 0.5
UPS_BUL  = 0.7

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff, vbar2

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

M_list = []
Rs_list = []

print("MEASURING Rs vs MASS")
print("====================================")

for f in files:
    try:
        rot = read_rotmod_file(f)
        rho_r, rho_vals, vbar2 = build_source(rot)
        r, m = solve_field(rho_r, rho_vals)

        u = np.maximum(m - m_inf, 1e-8)

        # Measure Rs from outer slope
        outer = r > (0.6 * np.max(r))
        if np.sum(outer) < 10:
            continue

        slope = np.polyfit(r[outer], np.log(u[outer]), 1)[0]
        Rs_eff = -1.0 / slope

        M_bar = np.trapz(vbar2 * rot["r"], rot["r"])

        if np.isfinite(Rs_eff) and Rs_eff > 0:
            M_list.append(M_bar)
            Rs_list.append(Rs_eff)

    except:
        pass

M = np.array(M_list)
Rs = np.array(Rs_list)

slope = np.polyfit(np.log(M), np.log(Rs), 1)[0]

print(f"Rs vs M slope = {slope:.3f}")

MEASURING Rs vs MASS
Rs vs M slope = 0.055


In [ ]:
# ============================================================
# GAMMA RESPONSE SWEEP
# V(r) = Vflat * (U/Uinf)^gamma
# ============================================================

import os, re, glob, warnings
import numpy as np
import pandas as pd

from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d, PchipInterpolator
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")

R_MIN = 1e-3
R_MAX = 55.0
N_R   = 700

# Field model parameters (keep fixed)
m_inf = 0.02
A_src = 0.10
GAMMA_SCREEN = 0.5
U0 = 1.0

UPS_DISK = 0.5
UPS_BUL  = 0.7

R_CORE = 0.5
SMOOTH_WIN = 9
SOURCE_FLOOR = 1e-6

GAMMA_LIST = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

# ------------------------------------------------------------

def moving_average(y, win=7):
    if win <= 1:
        return y
    if win % 2 == 0:
        win += 1
    pad = win // 2
    yp = np.pad(y, (pad, pad), mode="edge")
    ker = np.ones(win) / win
    return np.convolve(yp, ker, mode="valid")

def read_rotmod_file(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            parts = re.split(r"\s+", s)
            vals = []
            for p in parts:
                try:
                    vals.append(float(p))
                except:
                    pass
            if len(vals) >= 2:
                rows.append(vals)

    arr = np.array(rows)
    r = arr[:, 0]
    vobs = arr[:, 1]
    ev = arr[:, 2] if arr.shape[1] >= 3 else np.ones_like(vobs)*5
    vgas  = arr[:, 3] if arr.shape[1] >= 4 else np.zeros_like(vobs)
    vdisk = arr[:, 4] if arr.shape[1] >= 5 else np.zeros_like(vobs)
    vbul  = arr[:, 5] if arr.shape[1] >= 6 else np.zeros_like(vobs)

    mask = r > 0
    order = np.argsort(r[mask])

    return {
        "r": r[mask][order],
        "vobs": vobs[mask][order],
        "ev": ev[mask][order],
        "vgas": vgas[mask][order],
        "vdisk": vdisk[mask][order],
        "vbul": vbul[mask][order],
    }

def build_source(rot):
    r = rot["r"]
    vbar2 = np.maximum(
        rot["vgas"] * np.abs(rot["vgas"])
        + UPS_DISK * rot["vdisk"] * np.abs(rot["vdisk"])
        + UPS_BUL  * rot["vbul"]  * np.abs(rot["vbul"]),
        0.0
    )

    mbar_proxy = np.sqrt(r**2 + R_CORE**2) * vbar2
    dmbar_dr = np.gradient(mbar_proxy, r)
    rho_eff = dmbar_dr / np.maximum(r**2 + R_CORE**2, 1e-8)

    rho_eff = moving_average(rho_eff, win=SMOOTH_WIN)
    rho_eff = np.maximum(rho_eff, 0.0)
    rho_eff = rho_eff / np.max(rho_eff)
    rho_eff = np.maximum(rho_eff, SOURCE_FLOOR)

    return r, rho_eff

def solve_field(rho_r, rho_vals):
    r = np.linspace(R_MIN, R_MAX, N_R)
    dr = r[1] - r[0]

    rho_interp = PchipInterpolator(rho_r, rho_vals, extrapolate=False)
    rho = np.zeros_like(r)
    mask = (r >= rho_r[0]) & (r <= rho_r[-1])
    rho[mask] = rho_interp(r[mask])

    A = lil_matrix((N_R, N_R))
    b = np.zeros(N_R)

    A[0, 0] = 1
    A[0, 1] = -1

    for i in range(1, N_R - 1):
        ri = r[i]
        rip = r[i] + 0.5 * dr
        rim = r[i] - 0.5 * dr

        c_minus = rim**2 / (ri**2 * dr**2)
        c_plus  = rip**2 / (ri**2 * dr**2)

        screen = GAMMA_SCREEN / (rho[i] + U0)

        c0 = -(c_minus + c_plus) - screen

        A[i, i - 1] = c_minus
        A[i, i]     = c0
        A[i, i + 1] = c_plus
        b[i] = -A_src * rho[i]

    A[N_R - 1, N_R - 1] = 1
    b[N_R - 1] = m_inf

    m = spsolve(A.tocsr(), b)
    return r, m

# ------------------------------------------------------------
# RUN SWEEP
# ------------------------------------------------------------

files = sorted(glob.glob(os.path.join(ROTMOD_DIR, "*.dat")))

print("GAMMA SWEEP RESULTS")
print("============================================")

for gamma in GAMMA_LIST:

    rmse_list = []
    M_list = []
    V_list = []

    for f in files:
        try:
            rot = read_rotmod_file(f)
            rho_r, rho_vals = build_source(rot)
            r, m = solve_field(rho_r, rho_vals)

            u = np.maximum(m - m_inf, 0)
            U = cumulative_trapezoid(u, r, initial=0)

            if np.max(U) <= 0:
                continue

            U_obs = interp1d(r, U, fill_value="extrapolate")(rot["r"])
            U_obs = np.maximum(U_obs, 0)

            V_flat = np.max(rot["vobs"])
            V_pred = V_flat * (U_obs / np.max(U))**gamma

            rmse = np.sqrt(np.mean((rot["vobs"] - V_pred)**2))
            rmse_list.append(rmse)

            # BTFR
            M_bar = np.trapz(rot["vgas"]**2 + rot["vdisk"]**2 + rot["vbul"]**2, rot["r"])
            M_list.append(M_bar)
            V_list.append(V_flat)

        except:
            pass

    rmse_med = np.median(rmse_list)

    M = np.array(M_list)
    V = np.array(V_list)
    slope = np.polyfit(np.log(M), np.log(V**4), 1)[0]

    print(f"gamma={gamma:.2f} | Median RMSE={rmse_med:.2f} | BTFR slope={slope:.3f}")

GAMMA SWEEP RESULTS
gamma=0.15 | Median RMSE=18.67 | BTFR slope=0.994
gamma=0.20 | Median RMSE=17.60 | BTFR slope=0.994
gamma=0.25 | Median RMSE=16.20 | BTFR slope=0.994
gamma=0.30 | Median RMSE=15.53 | BTFR slope=0.994
gamma=0.35 | Median RMSE=14.94 | BTFR slope=0.994
gamma=0.40 | Median RMSE=14.35 | BTFR slope=0.994


In [ ]:
import numpy as np
import pandas as pd
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ==============================
# PATHS
# ==============================
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6   # kpc (km/s)^2 / Msun

# ==============================
# FIELD SOLVER (STABLE)
# ==============================
def solve_field(r, rho, gamma=0.35, beta=0.1, n_iter=200):
    u = np.zeros_like(r)
    dr = np.gradient(r)

    M_enc = cumulative_trapezoid(4*np.pi*r**2*rho, r, initial=0)

    for _ in range(n_iter):
        du_dr = np.gradient(u, r)
        lap = np.gradient(du_dr, r)

        sph = np.zeros_like(r)
        sph[1:] = 2/r[1:] * du_dr[1:]

        screen = (M_enc**gamma) * u
        u = u + 0.05 * (lap + sph - beta*u**3 + rho - screen)

    return u, M_enc

# ==============================
# PROCESS GALAXY
# ==============================
def process_galaxy(file, gamma):
    try:
        data = np.loadtxt(file)
        r = data[:,0]
        v_obs = data[:,1]
        v_gas = data[:,2]
        v_disk = data[:,3]
        v_bul = data[:,4]

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc_input = r * v_bar**2 / G
        rho = np.gradient(M_enc_input, r) / (4*np.pi*r**2 + 1e-8)

        u, M_enc = solve_field(r, rho, gamma)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]

        if not np.isfinite(U_inf) or U_inf <= 0:
            return None

        idx = np.searchsorted(U, 0.5*U_inf)
        Rs = r[idx] if idx < len(r) else r[-1]

        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        M_bar = M_enc[-1]

        if not np.isfinite(rmse) or not np.isfinite(Vflat):
            return None

        return rmse, Vflat, M_bar, Rs

    except:
        return None

# ==============================
# RUN SWEEP
# ==============================
files = glob.glob(ROT_PATH)

gamma_list = [0.15,0.20,0.25,0.30,0.35,0.40,0.45]

print("TOTAL FILES:", len(files))
print("")

for GAMMA in gamma_list:
    rmse_list = []
    V_list = []
    M_list = []
    Rs_list = []

    for f in files:
        result = process_galaxy(f, GAMMA)
        if result is not None:
            rmse, Vflat, Mbar, Rs = result
            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(Mbar)
            Rs_list.append(Rs)

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)

    if len(rmse_arr) > 10:
        med_rmse = np.median(rmse_arr)

        logM = np.log10(M_arr)
        logV4 = np.log10(V_arr**4)
        slope_btfr = np.polyfit(logM, logV4, 1)[0]

        logRs = np.log10(Rs_arr)
        slope_Rs = np.polyfit(logM, logRs, 1)[0]

        print(f"gamma={GAMMA:.2f} | galaxies={len(rmse_arr)} | Median RMSE={med_rmse:.2f} | BTFR slope={slope_btfr:.3f} | Rs slope={slope_Rs:.3f}")
    else:
        print(f"gamma={GAMMA:.2f} | Not enough valid galaxies")

TOTAL FILES: 175

gamma=0.15 | Not enough valid galaxies
gamma=0.20 | Not enough valid galaxies
gamma=0.25 | Not enough valid galaxies
gamma=0.30 | Not enough valid galaxies
gamma=0.35 | Not enough valid galaxies
gamma=0.40 | Not enough valid galaxies
gamma=0.45 | Not enough valid galaxies


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

def solve_field(r, rho, gamma=0.35, beta=0.1, n_iter=300):
    N = len(r)
    u = np.zeros(N)
    dr = np.gradient(r)

    M_enc = cumulative_trapezoid(4*np.pi*r**2*rho, r, initial=0)
    S = M_enc**gamma

    for _ in range(n_iter):
        u_new = u.copy()
        for i in range(1, N-1):
            d2u = (u[i+1] - 2*u[i] + u[i-1]) / (dr[i]**2 + 1e-6)
            du = (u[i+1] - u[i-1]) / (2*dr[i] + 1e-6)
            lap = d2u + (2/(r[i]+1e-6))*du
            u_new[i] = (rho[i] + lap) / (S[i] + beta*u[i]**2 + 1e-6)
        u = 0.5*u + 0.5*u_new

    return u, M_enc

def process_galaxy(file, gamma):
    try:
        data = np.loadtxt(file)
        r = data[:,0]
        v_obs = data[:,1]
        v_gas = data[:,2]
        v_disk = data[:,3]
        v_bul = data[:,4]

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc_input = r * v_bar**2 / G
        rho = np.gradient(M_enc_input, r) / (4*np.pi*r**2 + 1e-6)

        u, M_enc = solve_field(r, rho, gamma)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]

        if not np.isfinite(U_inf) or U_inf <= 0:
            return None

        idx = np.searchsorted(U, 0.5*U_inf)
        Rs = r[idx] if idx < len(r) else r[-1]

        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        M_bar = M_enc[-1]

        if not np.isfinite(rmse) or not np.isfinite(Vflat) or not np.isfinite(M_bar):
            return None

        return rmse, Vflat, M_bar, Rs

    except:
        return None

files = glob.glob(ROT_PATH)
gamma_list = [0.15,0.20,0.25,0.30,0.35,0.40,0.45]

print("TOTAL FILES:", len(files))
print("")

for GAMMA in gamma_list:
    rmse_list = []
    V_list = []
    M_list = []
    Rs_list = []

    for f in files:
        result = process_galaxy(f, GAMMA)
        if result is not None:
            rmse, Vflat, Mbar, Rs = result
            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(Mbar)
            Rs_list.append(Rs)

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)

    # FILTER BAD VALUES
    mask = (
        np.isfinite(rmse_arr) &
        np.isfinite(V_arr) &
        np.isfinite(M_arr) &
        np.isfinite(Rs_arr) &
        (V_arr > 0) &
        (M_arr > 0) &
        (Rs_arr > 0)
    )

    rmse_arr = rmse_arr[mask]
    V_arr = V_arr[mask]
    M_arr = M_arr[mask]
    Rs_arr = Rs_arr[mask]

    if len(rmse_arr) > 10:
        med_rmse = np.median(rmse_arr)

        try:
            logM = np.log10(M_arr)
            logV4 = np.log10(V_arr**4)
            slope_btfr = np.polyfit(logM, logV4, 1)[0]

            logRs = np.log10(Rs_arr)
            slope_Rs = np.polyfit(logM, logRs, 1)[0]

            print(f"gamma={GAMMA:.2f} | galaxies={len(rmse_arr)} | Median RMSE={med_rmse:.2f} | BTFR slope={slope_btfr:.3f} | Rs slope={slope_Rs:.3f}")

        except:
            print(f"gamma={GAMMA:.2f} | Fit failed")

    else:
        print(f"gamma={GAMMA:.2f} | Not enough valid galaxies")

TOTAL FILES: 175

gamma=0.15 | galaxies=169 | Median RMSE=65.47 | BTFR slope=0.234 | Rs slope=0.240
gamma=0.20 | galaxies=169 | Median RMSE=65.49 | BTFR slope=0.235 | Rs slope=0.240
gamma=0.25 | galaxies=169 | Median RMSE=65.53 | BTFR slope=0.246 | Rs slope=0.235
gamma=0.30 | galaxies=170 | Median RMSE=67.34 | BTFR slope=0.236 | Rs slope=0.233
gamma=0.35 | galaxies=170 | Median RMSE=66.35 | BTFR slope=0.227 | Rs slope=0.227
gamma=0.40 | galaxies=170 | Median RMSE=66.05 | BTFR slope=0.206 | Rs slope=0.213
gamma=0.45 | galaxies=170 | Median RMSE=65.98 | BTFR slope=0.154 | Rs slope=0.197


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ==============================
# PATHS
# ==============================
WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

LAMBDA = 0.01   # nonlinear term strength

# ==============================
# SOLVER FOR:
# d/dr ( r^2 (1/u) du/dr ) / r^2 + lambda u^2 = rho
# ==============================
def solve_field(r, rho, lam=0.01, n_iter=400):
    N = len(r)
    u = np.ones(N) * 0.1
    dr = np.gradient(r)

    for _ in range(n_iter):
        u_new = u.copy()

        for i in range(1, N-1):
            du = (u[i+1] - u[i-1]) / (2*dr[i] + 1e-6)
            d2u = (u[i+1] - 2*u[i] + u[i-1]) / (dr[i]**2 + 1e-6)

            term = (d2u/u[i]) - (du**2)/(u[i]**2) + (2/(r[i]+1e-6))*(du/u[i])

            u_new[i] = u[i] + 0.1 * (rho[i] - term - lam*u[i]**2)

        u = 0.5*u + 0.5*u_new
        u[u <= 0] = 1e-6

    return u

# ==============================
# PROCESS GALAXY
# ==============================
def process_galaxy(file):
    try:
        data = np.loadtxt(file)
        r = data[:,0]
        v_obs = data[:,1]
        v_gas = data[:,2]
        v_disk = data[:,3]
        v_bul = data[:,4]

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r) / (4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, LAMBDA)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]

        if U_inf <= 0 or not np.isfinite(U_inf):
            return None

        idx = np.searchsorted(U, 0.5*U_inf)
        Rs = r[idx] if idx < len(r) else r[-1]

        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        M_bar = M_enc[-1]

        return rmse, Vflat, M_bar, Rs

    except:
        return None

# ==============================
# RUN ALL GALAXIES
# ==============================
files = glob.glob(ROT_PATH)

rmse_list = []
V_list = []
M_list = []
Rs_list = []

print("TOTAL FILES:", len(files))

for f in files:
    result = process_galaxy(f)
    if result is not None:
        rmse, Vflat, Mbar, Rs = result
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

# FILTER
mask = (
    np.isfinite(rmse_arr) &
    np.isfinite(V_arr) &
    np.isfinite(M_arr) &
    np.isfinite(Rs_arr) &
    (V_arr > 0) &
    (M_arr > 0) &
    (Rs_arr > 0)
)

rmse_arr = rmse_arr[mask]
V_arr = V_arr[mask]
M_arr = M_arr[mask]
Rs_arr = Rs_arr[mask]

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

# BTFR slope
logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
slope_btfr = np.polyfit(logM, logV4, 1)[0]

print("BTFR slope:", slope_btfr)

# Rs slope
logRs = np.log10(Rs_arr)
slope_Rs = np.polyfit(logM, logRs, 1)[0]

print("Rs vs M slope:", slope_Rs)

TOTAL FILES: 175
Galaxies used: 33
Median RMSE: 64.04012259631664
BTFR slope: -0.4824047364008631
Rs vs M slope: 0.40174536186915727


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

# ============================================================
# FIRST-PRINCIPLES FIELD EQUATION TEST (DISC-SCALING DERIVED)
#
# Equation:
#   ∇·( r^p ∇u ) - kappa * M(<r)^gamma * u = -rho
#
# Derived from:
#   Rs scaling (disc mass) + BTFR scaling
#
# Parameters:
#   p = -2.8
#   gamma = -2.4
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

P = -2.8
GAMMA = -2.4
KAPPA = 1e-6

# ------------------------------------------------------------
# Read SPARC rotmod safely
# ------------------------------------------------------------
def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

# ------------------------------------------------------------
# Solve field equation
# ------------------------------------------------------------
def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n, n))
    b = np.zeros(n)

    # Inner BC: du/dr = 0
    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        h = r[i+1] - r[i]
        ri = r[i]

        # Operator: (1/r^2) d/dr ( r^(p+2) du/dr )
        c1 = (ri**(P+2))/(h**2) - ((P+2)*(ri**(P+1)))/(2*h)
        c2 = -2*(ri**(P+2))/(h**2) - KAPPA*(M_enc[i]**GAMMA)
        c3 = (ri**(P+2))/(h**2) + ((P+2)*(ri**(P+1)))/(2*h)

        A[i,i-1] = c1
        A[i,i]   = c2
        A[i,i+1] = c3

        b[i] = -rho[i]

    # Outer BC: du/dr = 0
    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

# ------------------------------------------------------------
# Process one galaxy
# ------------------------------------------------------------
def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)

        # Shift so cumulative positive
        u = u - np.min(u)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs

    except:
        return None

# ------------------------------------------------------------
# Run all galaxies
# ------------------------------------------------------------
files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list = [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

Galaxies used: 168
Median RMSE: nan
BTFR slope: nan
Rs slope: 0.3785667791702822


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

# ============================================================
# STABLE FIRST-PRINCIPLES FIELD EQUATION
#
#   ∇·( r^p ∇u ) - kappa * M(<r)^gamma * u = -rho
#
# Regularised near r=0 using (r + r0)
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

P = -2.8
GAMMA = -2.4
KAPPA = 1e-6
R0 = 0.3   # regularisation radius (kpc)

# ------------------------------------------------------------
def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

# ------------------------------------------------------------
def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n, n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        h = r[i+1] - r[i]
        ri = r[i]
        re = ri + R0

        # regularised operator
        c1 = (re**(P+2))/(h**2) - ((P+2)*(re**(P+1)))/(2*h)
        c2 = -2*(re**(P+2))/(h**2) - KAPPA*(M_enc[i]**GAMMA)
        c3 = (re**(P+2))/(h**2) + ((P+2)*(re**(P+1)))/(2*h)

        A[i,i-1] = c1
        A[i,i]   = c2
        A[i,i+1] = c3

        b[i] = -rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

# ------------------------------------------------------------
def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs

    except:
        return None

# ------------------------------------------------------------
files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list = [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

Galaxies used: 170
Median RMSE: nan
BTFR slope: nan
Rs slope: 0.34928763286851006


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# STABLE RELAXATION SOLVER
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

P = -2.8
GAMMA = -2.4
KAPPA = 1e-6
R0 = 0.3

# ------------------------------------------------------------
def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

# ------------------------------------------------------------
def solve_field_relax(r, rho, M_enc, n_iter=500):
    u = np.zeros_like(r)
    dr = np.gradient(r)

    for _ in range(n_iter):
        u_new = u.copy()

        for i in range(1, len(r)-1):
            ri = r[i]
            re = ri + R0
            h = dr[i]

            K = re**P
            dK = P * re**(P-1)

            # discretised operator
            lap = (K*(u[i+1] - 2*u[i] + u[i-1])/(h*h) +
                   dK*(u[i+1] - u[i-1])/(2*h))

            screen = KAPPA * (M_enc[i]**GAMMA) * u[i]

            u_new[i] = u[i] + 0.1 * (rho[i] - lap - screen)

        u = 0.5*u + 0.5*u_new

    return u

# ------------------------------------------------------------
def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field_relax(r, rho, M_enc)

        u = u - np.min(u)
        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]

        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U / U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs

    except:
        return None

# ------------------------------------------------------------
files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list = [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

Galaxies used: 175
Median RMSE: nan
BTFR slope: nan
Rs slope: 0.3151185187803656


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

GAMMA = 0.44
KAPPA = 1e-4

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        h = r[i+1]-r[i]
        ri = r[i]

        c1 = 1/h**2 - 1/(ri*h)
        c2 = -2/h**2 - KAPPA*(M_enc[i]**GAMMA)
        c3 = 1/h**2 + 1/(ri*h)

        A[i,i-1] = c1
        A[i,i]   = c2
        A[i,i+1] = c3

        b[i] = -rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U/U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs
    except:
        return None

files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list = [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

Galaxies used: 175
Median RMSE: 1928.2058497721262
BTFR slope: 0.7226626304214065
Rs slope: 0.1799158893816362


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

GAMMA = -0.12
KAPPA = 1e-3

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        h = r[i+1]-r[i]
        ri = r[i]

        c1 = 1/h**2 - 1/(ri*h)
        c2 = -2/h**2 - KAPPA*(M_enc[i]**GAMMA)
        c3 = 1/h**2 + 1/(ri*h)

        A[i,i-1] = c1
        A[i,i]   = c2
        A[i,i+1] = c3

        b[i] = -rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)

        U = cumulative_trapezoid(u, r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U/U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs
    except:
        return None

files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list = [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

Galaxies used: 175
Median RMSE: 4536.876351421671
BTFR slope: 1.8661725480986349
Rs slope: 0.2835392774597851


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

DELTA = 0.23
WEIGHT_K = -0.8
EPS = 1e-6

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        ML = (M_enc[i-1] + EPS)**DELTA
        MC = (M_enc[i]   + EPS)**DELTA
        MR = (M_enc[i+1] + EPS)**DELTA

        A[i,i-1] = ML / hL**2
        A[i,i]   = -MC*(1/hL**2 + 1/hR**2)
        A[i,i+1] = MR / hR**2

        b[i] = -rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)

        U = cumulative_trapezoid(u * r**(WEIGHT_K), r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U/U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs, U_inf
    except:
        return None

files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs, Uinf = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)
U_arr = np.array(U_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

logU = np.log10(U_arr)
print("U_inf vs M slope:", np.polyfit(logM, logU, 1)[0])

Galaxies used: 175
Median RMSE: 159.0449124260529
BTFR slope: 1.088728377975264
Rs slope: 0.2213609365515879
U_inf vs M slope: 0.76572512553922


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

DELTA = 0.23
EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-4

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = (M_enc[i-1] + EPS)**DELTA + STIFF_FLOOR
        KC = (M_enc[i]   + EPS)**DELTA + STIFF_FLOOR
        KR = (M_enc[i+1] + EPS)**DELTA + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def run_model(weight_k):
    files = glob.glob(ROT_PATH)
    rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

    for path in files:
        try:
            r, v_obs, v_gas, v_disk, v_bul = read_file(path)

            v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
            M_enc = r * v_bar**2 / G
            rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

            u = solve_field(r, rho, M_enc)
            u = u - np.min(u)
            u[u < 0] = 0

            U = cumulative_trapezoid(u * r**(weight_k), r, initial=0)
            U_inf = U[-1]
            if U_inf <= 0:
                continue

            Rs = r[np.searchsorted(U, 0.5*U_inf)]
            if Rs <= 0:
                continue

            Vflat = np.sqrt(U_inf / Rs)
            V_pred = Vflat * (U/U_inf)

            rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(M_enc[-1])
            Rs_list.append(Rs)
            U_list.append(U_inf)

        except:
            continue

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    if len(rmse_arr) < 20:
        return None

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return (
        np.median(rmse_arr),
        np.polyfit(logM, logV4, 1)[0],
        np.polyfit(logM, logRs, 1)[0],
        np.polyfit(logM, logU, 1)[0]
    )

print("KAPPA SWEEP")
print("=================================")

for k in [-0.5, 0.0, 0.5, 1.0, 1.5]:
    res = run_model(k)
    if res:
        rmse, btfr, rs, us = res
        print(f"k={k:4.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

KAPPA SWEEP
k=-0.50 | RMSE= 115.9 | BTFR=1.089 | Rs=0.180 | U=0.724
k=0.00 | RMSE= 144.4 | BTFR=1.245 | Rs=0.203 | U=0.825
k=0.50 | RMSE= 201.8 | BTFR=1.420 | Rs=0.227 | U=0.937
k=1.00 | RMSE= 279.4 | BTFR=1.638 | Rs=0.239 | U=1.058
k=1.50 | RMSE= 352.1 | BTFR=1.856 | Rs=0.259 | U=1.188


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-4

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc, delta):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = (M_enc[i-1] + EPS)**delta + STIFF_FLOOR
        KC = (M_enc[i]   + EPS)**delta + STIFF_FLOOR
        KR = (M_enc[i+1] + EPS)**delta + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def run_model(delta, kappa):
    files = glob.glob(ROT_PATH)
    rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

    for path in files:
        try:
            r, v_obs, v_gas, v_disk, v_bul = read_file(path)

            v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
            M_enc = r * v_bar**2 / G
            rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

            u = solve_field(r, rho, M_enc, delta)
            u = u - np.min(u)
            u[u < 0] = 0

            U = cumulative_trapezoid(u * r**(kappa), r, initial=0)
            U_inf = U[-1]
            if U_inf <= 0:
                continue

            Rs = r[np.searchsorted(U, 0.5*U_inf)]
            if Rs <= 0:
                continue

            Vflat = np.sqrt(U_inf / Rs)
            V_pred = Vflat * (U/U_inf)

            rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(M_enc[-1])
            Rs_list.append(Rs)
            U_list.append(U_inf)

        except:
            continue

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return (
        np.median(rmse_arr),
        np.polyfit(logM, logV4, 1)[0],
        np.polyfit(logM, logRs, 1)[0],
        np.polyfit(logM, logU, 1)[0]
    )

print("DELTA-KAPPA SWEEP")
print("==============================================")

for delta in [0.15, 0.20, 0.25, 0.30]:
    for kappa in [-0.5, 0.0]:
        res = run_model(delta, kappa)
        if res:
            rmse, btfr, rs, us = res
            print(f"d={delta:.2f}, k={kappa:.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

DELTA-KAPPA SWEEP
d=0.15, k=-0.50 | RMSE= 347.5 | BTFR=1.259 | Rs=0.182 | U=0.812
d=0.15, k=0.00 | RMSE= 452.3 | BTFR=1.411 | Rs=0.208 | U=0.914
d=0.20, k=-0.50 | RMSE= 177.7 | BTFR=1.152 | Rs=0.181 | U=0.757
d=0.20, k=0.00 | RMSE= 214.8 | BTFR=1.307 | Rs=0.205 | U=0.858
d=0.25, k=-0.50 | RMSE=  88.0 | BTFR=1.047 | Rs=0.179 | U=0.702
d=0.25, k=0.00 | RMSE=  91.6 | BTFR=1.203 | Rs=0.202 | U=0.803
d=0.30, k=-0.50 | RMSE=  59.7 | BTFR=0.932 | Rs=0.182 | U=0.648
d=0.30, k=0.00 | RMSE=  66.8 | BTFR=1.096 | Rs=0.200 | U=0.748


In [ ]:
print("NARROW SWEEP AROUND BEST REGION")
print("==============================================")

for delta in [0.30]:
    for kappa in [-0.25, -0.10, 0.10, 0.25]:
        res = run_model(delta, kappa)
        if res:
            rmse, btfr, rs, us = res
            print(f"d={delta:.2f}, k={kappa:.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

NARROW SWEEP AROUND BEST REGION
d=0.30, k=-0.25 | RMSE=  63.2 | BTFR=1.013 | Rs=0.190 | U=0.697
d=0.30, k=-0.10 | RMSE=  64.1 | BTFR=1.059 | Rs=0.198 | U=0.727
d=0.30, k=0.10 | RMSE=  67.4 | BTFR=1.128 | Rs=0.205 | U=0.769
d=0.30, k=0.25 | RMSE=  75.0 | BTFR=1.171 | Rs=0.217 | U=0.802


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

DELTA = 0.30
WEIGHT_K = -0.25

EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-4

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = ((M_enc[i-1]/(rL+EPS))**DELTA) + STIFF_FLOOR
        KC = ((M_enc[i]/(rC+EPS))**DELTA) + STIFF_FLOOR
        KR = ((M_enc[i+1]/(rR+EPS))**DELTA) + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def process(path):
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)
        u[u < 0] = 0

        U = cumulative_trapezoid(u * r**(WEIGHT_K), r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            return None

        Rs = r[np.searchsorted(U, 0.5*U_inf)]
        if Rs <= 0:
            return None

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U/U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))
        return rmse, Vflat, M_enc[-1], Rs, U_inf
    except:
        return None

files = glob.glob(ROT_PATH)

rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

for f in files:
    out = process(f)
    if out:
        rmse, Vflat, Mbar, Rs, Uinf = out
        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs_arr = np.array(Rs_list)
U_arr = np.array(U_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])

logRs = np.log10(Rs_arr)
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])

logU = np.log10(U_arr)
print("U_inf vs M slope:", np.polyfit(logM, logU, 1)[0])

Galaxies used: 175
Median RMSE: 73.72933443858958
BTFR slope: 1.1825834282706291
Rs slope: 0.20086585599539347
U_inf vs M slope: 0.7921575701307081


In [ ]:
print("LAMBDA SWEEP (TRANSITION SCALE)")
print("==============================================")

for LAMBDA in [0.0, 1e-5, 3e-5, 1e-4, 3e-4, 1e-3]:
    # reuse run_model but with lambda inserted
    def run_lambda(lambda_val):
        files = glob.glob(ROT_PATH)
        rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

        for path in files:
            try:
                r, v_obs, v_gas, v_disk, v_bul = read_file(path)

                v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
                M_enc = r * v_bar**2 / G
                rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

                # solver with lambda
                n = len(r)
                A = lil_matrix((n,n))
                b = np.zeros(n)

                A[0,0] = 1
                A[0,1] = -1

                for i in range(1, n-1):
                    rL, rC, rR = r[i-1], r[i], r[i+1]
                    hL = rC - rL
                    hR = rR - rC

                    KL = ((M_enc[i-1]/(rL+EPS))**DELTA) + STIFF_FLOOR
                    KC = ((M_enc[i]/(rC+EPS))**DELTA) + STIFF_FLOOR
                    KR = ((M_enc[i+1]/(rR+EPS))**DELTA) + STIFF_FLOOR

                    AL = (rL**2 * KL) / hL
                    AR = (rR**2 * KR) / hR

                    A[i,i-1] = AL
                    A[i,i]   = -(AL + AR) - lambda_val
                    A[i,i+1] = AR

                    b[i] = -rC**2 * rho[i]

                A[n-1,n-2] = -1
                A[n-1,n-1] = 1

                u = spsolve(A.tocsr(), b)
                u = u - np.min(u)
                u[u < 0] = 0

                U = cumulative_trapezoid(u * r**(WEIGHT_K), r, initial=0)
                U_inf = U[-1]
                if U_inf <= 0:
                    continue

                Rs = r[np.searchsorted(U, 0.5*U_inf)]
                if Rs <= 0:
                    continue

                Vflat = np.sqrt(U_inf / Rs)
                V_pred = Vflat * (U/U_inf)

                rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

                rmse_list.append(rmse)
                V_list.append(Vflat)
                M_list.append(M_enc[-1])
                Rs_list.append(Rs)
                U_list.append(U_inf)

            except:
                continue

        if len(rmse_list) < 20:
            return None

        rmse_arr = np.array(rmse_list)
        V_arr = np.array(V_list)
        M_arr = np.array(M_list)
        Rs_arr = np.array(Rs_list)
        U_arr = np.array(U_list)

        logM = np.log10(M_arr)
        logV4 = np.log10(V_arr**4)
        logRs = np.log10(Rs_arr)
        logU = np.log10(U_arr)

        return (
            np.median(rmse_arr),
            np.polyfit(logM, logV4, 1)[0],
            np.polyfit(logM, logRs, 1)[0],
            np.polyfit(logM, logU, 1)[0]
        )

    res = run_lambda(LAMBDA)
    if res:
        rmse, btfr, rs, us = res
        print(f"lambda={LAMBDA:.0e} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

LAMBDA SWEEP (TRANSITION SCALE)
lambda=0e+00 | RMSE=   nan | BTFR=  nan | Rs=0.232 | U=  nan
lambda=1e-05 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792
lambda=3e-05 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792
lambda=1e-04 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792
lambda=3e-04 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792
lambda=1e-03 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792


In [ ]:
print("ETA SWEEP (STIFFNESS RADIAL DEPENDENCE)")
print("==============================================")

for ETA in [0.5, 1.0, 1.3, 1.6, 2.0]:
    def run_eta(eta_val):
        files = glob.glob(ROT_PATH)
        rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

        for path in files:
            try:
                r, v_obs, v_gas, v_disk, v_bul = read_file(path)

                v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
                M_enc = r * v_bar**2 / G
                rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

                n = len(r)
                A = lil_matrix((n,n))
                b = np.zeros(n)

                A[0,0] = 1
                A[0,1] = -1

                for i in range(1, n-1):
                    rL, rC, rR = r[i-1], r[i], r[i+1]
                    hL = rC - rL
                    hR = rR - rC

                    KL = ((M_enc[i-1]/(rL**eta_val + EPS))**DELTA) + STIFF_FLOOR
                    KC = ((M_enc[i]/(rC**eta_val + EPS))**DELTA) + STIFF_FLOOR
                    KR = ((M_enc[i+1]/(rR**eta_val + EPS))**DELTA) + STIFF_FLOOR

                    AL = (rL**2 * KL) / hL
                    AR = (rR**2 * KR) / hR

                    A[i,i-1] = AL
                    A[i,i]   = -(AL + AR) - LAMBDA
                    A[i,i+1] = AR

                    b[i] = -rC**2 * rho[i]

                A[n-1,n-2] = -1
                A[n-1,n-1] = 1

                u = spsolve(A.tocsr(), b)
                u = u - np.min(u)
                u[u < 0] = 0

                U = cumulative_trapezoid(u * r**(WEIGHT_K), r, initial=0)
                U_inf = U[-1]
                if U_inf <= 0:
                    continue

                Rs = r[np.searchsorted(U, 0.5*U_inf)]
                if Rs <= 0:
                    continue

                Vflat = np.sqrt(U_inf / Rs)
                V_pred = Vflat * (U/U_inf)

                rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

                rmse_list.append(rmse)
                V_list.append(Vflat)
                M_list.append(M_enc[-1])
                Rs_list.append(Rs)
                U_list.append(U_inf)

            except:
                continue

        if len(rmse_list) < 20:
            return None

        rmse_arr = np.array(rmse_list)
        V_arr = np.array(V_list)
        M_arr = np.array(M_list)
        Rs_arr = np.array(Rs_list)
        U_arr = np.array(U_list)

        logM = np.log10(M_arr)
        logV4 = np.log10(V_arr**4)
        logRs = np.log10(Rs_arr)
        logU = np.log10(U_arr)

        return (
            np.median(rmse_arr),
            np.polyfit(logM, logV4, 1)[0],
            np.polyfit(logM, logRs, 1)[0],
            np.polyfit(logM, logU, 1)[0]
        )

    res = run_eta(ETA)
    if res:
        rmse, btfr, rs, us = res
        print(f"eta={ETA:.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

ETA SWEEP (STIFFNESS RADIAL DEPENDENCE)
eta=0.50 | RMSE=  65.2 | BTFR=1.095 | Rs=0.196 | U=0.744
eta=1.00 | RMSE=  73.7 | BTFR=1.183 | Rs=0.201 | U=0.792
eta=1.30 | RMSE=  72.8 | BTFR=1.225 | Rs=0.210 | U=0.822
eta=1.60 | RMSE=  77.5 | BTFR=1.279 | Rs=0.213 | U=0.852
eta=2.00 | RMSE=  81.4 | BTFR=1.355 | Rs=0.216 | U=0.894


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

ETA = 0.5
KAPPA = -0.25
EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc, DELTA):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = ((M_enc[i-1]/(rL**ETA + EPS))**DELTA) + STIFF_FLOOR
        KC = ((M_enc[i]/(rC**ETA + EPS))**DELTA) + STIFF_FLOOR
        KR = ((M_enc[i+1]/(rR**ETA + EPS))**DELTA) + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def run_model(DELTA):
    files = glob.glob(ROT_PATH)
    rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

    for path in files:
        try:
            r, v_obs, v_gas, v_disk, v_bul = read_file(path)

            v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
            M_enc = r * v_bar**2 / G
            rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

            u = solve_field(r, rho, M_enc, DELTA)
            u = u - np.min(u)
            u[u < 0] = 0

            U = cumulative_trapezoid(u * r**(KAPPA), r, initial=0)
            U_inf = U[-1]
            if U_inf <= 0:
                continue

            Rs = r[np.searchsorted(U, 0.5*U_inf)]
            if Rs <= 0:
                continue

            Vflat = np.sqrt(U_inf / Rs)
            V_pred = Vflat * (U/U_inf)

            rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(M_enc[-1])
            Rs_list.append(Rs)
            U_list.append(U_inf)

        except:
            continue

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return (
        np.median(rmse_arr),
        np.polyfit(logM, logV4, 1)[0],
        np.polyfit(logM, logRs, 1)[0],
        np.polyfit(logM, logU, 1)[0]
    )

print("DELTA SWEEP")
print("==============================================")

for DELTA in [0.3, 0.5, 0.7, 0.8, 0.9]:
    res = run_model(DELTA)
    if res:
        rmse, btfr, rs, us = res
        print(f"delta={DELTA:.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

DELTA SWEEP
delta=0.30 | RMSE=  65.2 | BTFR=1.095 | Rs=0.196 | U=0.744
delta=0.50 | RMSE=  64.8 | BTFR=0.725 | Rs=0.192 | U=0.555
delta=0.70 | RMSE=  78.8 | BTFR=0.365 | Rs=0.197 | U=0.379
delta=0.80 | RMSE=  81.0 | BTFR=0.199 | Rs=0.213 | U=0.312
delta=0.90 | RMSE=   nan | BTFR=  nan | Rs=0.172 | U=  nan


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

DELTA = 0.30
ETA = 0.5
KAPPA = -0.25

EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = ((M_enc[i-1]/(rL**ETA + EPS))**DELTA) + STIFF_FLOOR
        KC = ((M_enc[i]/(rC**ETA + EPS))**DELTA) + STIFF_FLOOR
        KR = ((M_enc[i+1]/(rR**ETA + EPS))**DELTA) + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def compute_Rs_all(r, u, U):
    U_inf = U[-1]

    # 1) Half U
    Rs_halfU = r[np.searchsorted(U, 0.5*U_inf)]

    # 2) Half u
    u_max = np.max(u)
    Rs_halfu = r[np.searchsorted(u, 0.5*u_max)]

    # 3) Peak slope
    dU = np.gradient(U, r)
    Rs_slope = r[np.argmax(dU)]

    # 4) Curvature peak
    d2U = np.gradient(dU, r)
    Rs_curve = r[np.argmax(d2U)]

    # 5) 1 - 1/e
    Rs_1e = r[np.searchsorted(U, (1 - np.exp(-1))*U_inf)]

    return Rs_halfU, Rs_halfu, Rs_slope, Rs_curve, Rs_1e

files = glob.glob(ROT_PATH)

rmse_list = []
V_list = []
M_list = []
Rs1_list = []
Rs2_list = []
Rs3_list = []
Rs4_list = []
Rs5_list = []
U_list = []

print("Processing galaxies...\n")

for path in files:
    try:
        r, v_obs, v_gas, v_disk, v_bul = read_file(path)

        v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
        M_enc = r * v_bar**2 / G
        rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

        u = solve_field(r, rho, M_enc)
        u = u - np.min(u)
        u[u < 0] = 0

        U = cumulative_trapezoid(u * r**(KAPPA), r, initial=0)
        U_inf = U[-1]
        if U_inf <= 0:
            continue

        Rs_halfU, Rs_halfu, Rs_slope, Rs_curve, Rs_1e = compute_Rs_all(r, u, U)

        Rs = Rs_halfU  # still use halfU for velocity for now
        if Rs <= 0:
            continue

        Vflat = np.sqrt(U_inf / Rs)
        V_pred = Vflat * (U/U_inf)

        rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

        rmse_list.append(rmse)
        V_list.append(Vflat)
        M_list.append(M_enc[-1])
        Rs1_list.append(Rs_halfU)
        Rs2_list.append(Rs_halfu)
        Rs3_list.append(Rs_slope)
        Rs4_list.append(Rs_curve)
        Rs5_list.append(Rs_1e)
        U_list.append(U_inf)

    except:
        continue

rmse_arr = np.array(rmse_list)
V_arr = np.array(V_list)
M_arr = np.array(M_list)
Rs1 = np.array(Rs1_list)
Rs2 = np.array(Rs2_list)
Rs3 = np.array(Rs3_list)
Rs4 = np.array(Rs4_list)
Rs5 = np.array(Rs5_list)
U_arr = np.array(U_list)

print("Galaxies used:", len(rmse_arr))
print("Median RMSE:", np.median(rmse_arr))

logM = np.log10(M_arr)

print("\nRS SLOPES:")
print("Half U slope:", np.polyfit(logM, np.log10(Rs1), 1)[0])
print("Half u slope:", np.polyfit(logM, np.log10(Rs2), 1)[0])
print("Slope peak:", np.polyfit(logM, np.log10(Rs3), 1)[0])
print("Curvature peak:", np.polyfit(logM, np.log10(Rs4), 1)[0])
print("1 - 1/e:", np.polyfit(logM, np.log10(Rs5), 1)[0])

print("\nBTFR slope:")
print(np.polyfit(logM, np.log10(V_arr**4), 1)[0])

print("\nU_inf vs M slope:")
print(np.polyfit(logM, np.log10(U_arr), 1)[0])

Processing galaxies...

Galaxies used: 55
Median RMSE: 39.481460440149874

RS SLOPES:
Half U slope: 0.2664398877462234
Half u slope: 0.15012726714207397
Slope peak: 0.22303466457692497
Curvature peak: 0.18512756419948898
1 - 1/e: 0.2863598925932191

BTFR slope:
0.6554959861890347

U_inf vs M slope:
0.5941878808407409


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

DELTA = 0.30
ETA = 0.5

EPS = 1e-6
STIFF_FLOOR = 1e-3
LAMBDA = 1e-5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = r > 0
    return r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]

def solve_field(r, rho, M_enc):
    n = len(r)
    A = lil_matrix((n,n))
    b = np.zeros(n)

    A[0,0] = 1
    A[0,1] = -1

    for i in range(1, n-1):
        rL, rC, rR = r[i-1], r[i], r[i+1]
        hL = rC - rL
        hR = rR - rC

        KL = ((M_enc[i-1]/(rL**ETA + EPS))**DELTA) + STIFF_FLOOR
        KC = ((M_enc[i]/(rC**ETA + EPS))**DELTA) + STIFF_FLOOR
        KR = ((M_enc[i+1]/(rR**ETA + EPS))**DELTA) + STIFF_FLOOR

        AL = (rL**2 * KL) / hL
        AR = (rR**2 * KR) / hR

        A[i,i-1] = AL
        A[i,i]   = -(AL + AR) - LAMBDA
        A[i,i+1] = AR

        b[i] = -rC**2 * rho[i]

    A[n-1,n-2] = -1
    A[n-1,n-1] = 1

    u = spsolve(A.tocsr(), b)
    return u

def run_model(KAPPA):
    files = glob.glob(ROT_PATH)
    rmse_list, V_list, M_list, Rs_list, U_list = [], [], [], [], []

    for path in files:
        try:
            r, v_obs, v_gas, v_disk, v_bul = read_file(path)

            v_bar = np.sqrt(v_gas**2 + (UPS_DISK*v_disk)**2 + (UPS_BUL*v_bul)**2)
            M_enc = r * v_bar**2 / G
            rho = np.gradient(M_enc, r)/(4*np.pi*r**2 + 1e-6)

            u = solve_field(r, rho, M_enc)
            u = u - np.min(u)
            u[u < 0] = 0

            U = cumulative_trapezoid(u * r**(KAPPA), r, initial=0)
            U_inf = U[-1]
            if U_inf <= 0:
                continue

            # Rs defined at (1 - 1/e)
            target = (1 - np.exp(-1)) * U_inf
            Rs = r[np.searchsorted(U, target)]
            if Rs <= 0:
                continue

            Vflat = np.sqrt(U_inf / Rs)
            V_pred = Vflat * (U/U_inf)

            rmse = np.sqrt(np.mean((V_pred - v_obs)**2))

            rmse_list.append(rmse)
            V_list.append(Vflat)
            M_list.append(M_enc[-1])
            Rs_list.append(Rs)
            U_list.append(U_inf)

        except:
            continue

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(V_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return (
        np.median(rmse_arr),
        np.polyfit(logM, logV4, 1)[0],
        np.polyfit(logM, logRs, 1)[0],
        np.polyfit(logM, logU, 1)[0]
    )

print("KAPPA SWEEP (New Rs Definition)")
print("==============================================")

for KAPPA in [-0.5, -0.4, -0.3, -0.2, -0.1, 0.0]:
    res = run_model(KAPPA)
    if res:
        rmse, btfr, rs, us = res
        print(f"kappa={KAPPA:.2f} | RMSE={rmse:6.1f} | BTFR={btfr:5.3f} | Rs={rs:5.3f} | U={us:5.3f}")

KAPPA SWEEP (New Rs Definition)
kappa=-0.50 | RMSE=  60.9 | BTFR=0.991 | Rs=0.198 | U=0.694
kappa=-0.40 | RMSE=  63.8 | BTFR=1.018 | Rs=0.204 | U=0.713
kappa=-0.30 | RMSE=  61.0 | BTFR=1.045 | Rs=0.211 | U=0.733
kappa=-0.20 | RMSE=  62.2 | BTFR=1.078 | Rs=0.215 | U=0.754
kappa=-0.10 | RMSE=  60.4 | BTFR=1.117 | Rs=0.217 | U=0.775
kappa=0.00 | RMSE=  66.1 | BTFR=1.146 | Rs=0.224 | U=0.797


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TWO-COMPONENT RESPONSE-LAW SWEEP
#
# Frozen PDE/source:
#   div(K_eff grad chi) - mu2 chi = -rho_eff
#
# with
#   K_eff = K0(r) * (1 + B * |dchi/dr|)
#   K0(r)  = ( M(<r) / r^eta )^delta
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
# Observable:
#   Uchi(r) = ∫ chi(r) r^kappa dr
#   Rs: Uchi(Rs) = (1 - e^-1) Uchi_inf
#   Vflat^2 = A_global * Uchi_inf / Rs
#
# New response law:
#   y = Uchi/Uinf
#   V(r) = Vflat * sqrt( a*y + (1-a)*y^p )
#
# Goal:
#   improve high-mass / inner-region performance beyond single p=0.5 law
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen best-current PDE/source structure
DELTA = 0.30
ETA   = 0.50
MU2   = 1e-7
BVAL  = 0.10
KAPPA = 0.10

# Best source row so far
A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0

EPS = 1e-6
STIFF_FLOOR = 1e-3

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_clock_field_nonlinear(r, rho_eff, M_enc, B, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + B * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            K0L = ((M_enc[i - 1] / (rL**ETA + EPS))**DELTA) + STIFF_FLOOR
            K0R = ((M_enc[i + 1] / (rR**ETA + EPS))**DELTA) + STIFF_FLOOR

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - MU2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows():
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, BVAL)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })

        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global, a_mix, p_exp):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)

        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        mix = a_mix * y + (1.0 - a_mix) * np.power(y, p_exp)
        mix = np.clip(mix, 0.0, 1.0e30)

        v_pred = Vflat * np.sqrt(mix)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(a_mix, p_exp):
    rows = build_rows()
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all, a_mix, p_exp)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)

    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low, a_mix, p_exp)
    high_to_low = evaluate_rows(low, A_high, a_mix, p_exp)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("TWO-COMPONENT RESPONSE-LAW SWEEP")
print("==============================================================")

for a_mix in [0.0, 0.25, 0.50, 0.75]:
    for p_exp in [0.40, 0.50, 0.60]:
        res = run_case(a_mix, p_exp)
        print(f"\na={a_mix:.2f}, p={p_exp:.2f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

TWO-COMPONENT RESPONSE-LAW SWEEP

a=0.00, p=0.40
--------------------------------------------------------------
galaxies: 60
median RMSE: 37.547695672249596
BTFR slope: 0.9464803824215942
Rs slope: 0.31564598654589815
U slope: 0.7888861777566952
median frac residual inner: 0.33011830029853406
median frac residual mid: 0.21534297999709467
median frac residual outer: 0.20610201718728405
median RMSE low-mass: 18.65453293841312
median RMSE high-mass: 69.06373551720674
Fit LOW -> Test HIGH RMSE: 75.4886495787778
Fit HIGH -> Test LOW RMSE: 18.306333130284855

a=0.00, p=0.50
--------------------------------------------------------------
galaxies: 60
median RMSE: 35.209245116530724
BTFR slope: 0.9464803824215942
Rs slope: 0.31564598654589815
U slope: 0.7888861777566952
median frac residual inner: 0.37403181519718753
median frac residual mid: 0.22709061247691437
median frac residual outer: 0.20610201718728405
median RMSE low-mass: 18.88539958542577
median RMSE high-mass: 68.16665102781785
Fit L

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# STIFFNESS-POTENTIAL SWEEP
#
# Frozen current best structure except stiffness law:
#
#   div(K_eff grad chi) - mu2 chi = -rho_eff
#
# with
#   K_eff = K0(r) * (1 + B * |dchi/dr|)
#
# NEW:
#   K0(r) = ( M(<r) / r^eta )^delta
#           * (1 + C_phi * (|Phi_b| / Phi_ref)^q_phi )
#
# where
#   |Phi_b| proxy ~ v_bar^2
#
# Frozen source:
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
# Frozen best response law from previous sweep:
#   y = Uchi/Uinf
#   V(r) = Vflat * y^0.20
#
# Goal:
#   improve high-mass / inner-region performance further.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen structure
DELTA = 0.30
ETA   = 0.50
MU2   = 1e-7
BVAL  = 0.10
KAPPA = 0.10

# Frozen best source row
A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0   # native units: (km/s)^2 / kpc

# Frozen best response law:
# V = Vflat * (U/Uinf)^RESP_EXP
RESP_EXP = 0.20

# Potential reference in native units [(km/s)^2]
PHI_REF = 1.0e4

EPS = 1e-6
STIFF_FLOOR = 1e-3

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_phi, q_phi, B, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    # baryonic potential proxy
    phi_fac = 1.0 + C_phi * np.power(np.clip(v_bar2 / PHI_REF, 0.0, None), q_phi)
    phi_fac = np.nan_to_num(phi_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + B * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            baseL = ((M_enc[i - 1] / (rL**ETA + EPS))**DELTA) + STIFF_FLOOR
            baseR = ((M_enc[i + 1] / (rR**ETA + EPS))**DELTA) + STIFF_FLOOR

            K0L = baseL * phi_fac[i - 1]
            K0R = baseR * phi_fac[i + 1]

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - MU2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(C_phi, q_phi):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * force_factor

            chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_phi, q_phi, BVAL)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })

        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(C_phi, q_phi):
    rows = build_rows(C_phi, q_phi)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)

    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("STIFFNESS-POTENTIAL SWEEP")
print("==============================================================")

for C_phi in [0.0, 0.05, 0.10, 0.20]:
    for q_phi in [0.5, 1.0]:
        res = run_case(C_phi, q_phi)
        print(f"\nC_phi={C_phi:.2f}, q_phi={q_phi:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

STIFFNESS-POTENTIAL SWEEP

C_phi=0.00, q_phi=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 37.5476956722496
BTFR slope: 0.9464803824215942
Rs slope: 0.31564598654589815
U slope: 0.7888861777566952
median frac residual inner: 0.33011830029853406
median frac residual mid: 0.21534297999709467
median frac residual outer: 0.20610201718728405
median RMSE low-mass: 18.65453293841312
median RMSE high-mass: 69.06373551720674
Fit LOW -> Test HIGH RMSE: 75.4886495787778
Fit HIGH -> Test LOW RMSE: 18.306333130284855

C_phi=0.00, q_phi=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 37.5476956722496
BTFR slope: 0.9464803824215942
Rs slope: 0.31564598654589815
U slope: 0.7888861777566952
median frac residual inner: 0.33011830029853406
median frac residual mid: 0.21534297999709467
median frac residual outer: 0.20610201718728405
median RMSE low-mass: 18.65453293841312
median RMSE high-mass: 69.06373551720674
Fi

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# NARROW STIFFNESS-POTENTIAL SWEEP
#
# Frozen best-current model:
#   div(K_eff grad chi) - mu2 chi = -rho_eff
#
#   K_eff = K0(r) * (1 + B * |dchi/dr|)
#   K0(r) = ( M(<r) / r^eta )^delta
#           * (1 + C_phi * (|Phi_b| / Phi_ref)^q_phi )
#
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
#   Uchi(r) = ∫ chi(r) r^kappa dr
#   Rs: Uchi(Rs) = (1 - e^-1) Uchi_inf
#   Vflat^2 = A_global * Uchi_inf / Rs
#   V(r) = Vflat * (Uchi/Uinf)^RESP_EXP
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen structure
DELTA = 0.30
ETA   = 0.50
MU2   = 1e-7
BVAL  = 0.10
KAPPA = 0.10

# Frozen best source row
A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0   # native units: (km/s)^2 / kpc

# Frozen best response law
RESP_EXP = 0.20

# Potential reference in native units [(km/s)^2]
PHI_REF = 1.0e4

EPS = 1e-6
STIFF_FLOOR = 1e-3

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_phi, q_phi, B, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    phi_fac = 1.0 + C_phi * np.power(np.clip(v_bar2 / PHI_REF, 0.0, None), q_phi)
    phi_fac = np.nan_to_num(phi_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + B * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            baseL = ((M_enc[i - 1] / (rL**ETA + EPS))**DELTA) + STIFF_FLOOR
            baseR = ((M_enc[i + 1] / (rR**ETA + EPS))**DELTA) + STIFF_FLOOR

            K0L = baseL * phi_fac[i - 1]
            K0R = baseR * phi_fac[i + 1]

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - MU2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(C_phi, q_phi):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * force_factor

            chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_phi, q_phi, BVAL)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })

        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(C_phi, q_phi):
    rows = build_rows(C_phi, q_phi)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)

    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("NARROW STIFFNESS-POTENTIAL SWEEP")
print("==============================================================")

for C_phi in [0.08, 0.10, 0.12, 0.15, 0.20]:
    for q_phi in [0.40, 0.50, 0.60]:
        res = run_case(C_phi, q_phi)
        print(f"\nC_phi={C_phi:.2f}, q_phi={q_phi:.2f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

NARROW STIFFNESS-POTENTIAL SWEEP

C_phi=0.08, q_phi=0.40
--------------------------------------------------------------
galaxies: 60
median RMSE: 39.026290600123595
BTFR slope: 0.9230695757972135
Rs slope: 0.3179808997867876
U slope: 0.7795156876853944
median frac residual inner: 0.3254432252742312
median frac residual mid: 0.19187091280173427
median frac residual outer: 0.18546989733804276
median RMSE low-mass: 18.482116310763434
median RMSE high-mass: 66.53119490378798
Fit LOW -> Test HIGH RMSE: 79.99699343549264
Fit HIGH -> Test LOW RMSE: 18.62208832405741

C_phi=0.08, q_phi=0.50
--------------------------------------------------------------
galaxies: 60
median RMSE: 39.20452407345094
BTFR slope: 0.9221240967443683
Rs slope: 0.32159153142938357
U slope: 0.7826535798015678
median frac residual inner: 0.32753529896806344
median frac residual mid: 0.18982921829898958
median frac residual outer: 0.18740984572228367
median RMSE low-mass: 18.54230137514546
median RMSE high-mass: 69.234060

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FROZEN BEST-MODEL FORENSIC DIAGNOSTIC
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- FROZEN BEST MODEL --------
DELTA = 0.30
ETA   = 0.50
MU2   = 1e-7
BVAL  = 0.10
KAPPA = 0.10

A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0

C_PHI = 0.15
Q_PHI = 0.50
PHI_REF = 1.0e4

RESP_EXP = 0.20

EPS = 1e-6
STIFF_FLOOR = 1e-3

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False, "too_few_points"
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False, "bad_radius"
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False, "insufficient_radial_range"
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False, "too_few_outer_points"
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False, "bad_outer_velocity"
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False, "not_flat_outer"
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False, "insufficient_inner_resolution"
    return True, "pass"

def solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_phi, q_phi, B, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    phi_fac = 1.0 + C_phi * np.power(np.clip(v_bar2 / PHI_REF, 0.0, None), q_phi)
    phi_fac = np.nan_to_num(phi_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + B * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            baseL = ((M_enc[i - 1] / (rL**ETA + EPS))**DELTA) + STIFF_FLOOR
            baseR = ((M_enc[i + 1] / (rR**ETA + EPS))**DELTA) + STIFF_FLOOR

            K0L = baseL * phi_fac[i - 1]
            K0R = baseR * phi_fac[i + 1]

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - MU2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

# -------- Build solved rows --------
files = sorted(glob.glob(ROT_PATH))
rows = []
cut_stats = {}

print("RUNNING FROZEN BEST MODEL")
print("==============================================================")

for path in files:
    data = read_file(path)
    if data is None:
        cut_stats["read_fail"] = cut_stats.get("read_fail", 0) + 1
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    passed, reason = passes_quality_cuts(r, v_obs)
    if not passed:
        cut_stats[reason] = cut_stats.get(reason, 0) + 1
        continue

    try:
        name = os.path.basename(path)

        v_bar2 = np.maximum(
            v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
            0.0
        )

        M_enc = r * v_bar2 / G
        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

        g_b = v_bar2 / (r + EPS)
        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)

        rho_eff = rho * force_factor

        chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, C_PHI, Q_PHI, BVAL)
        if chi is None or not np.all(np.isfinite(chi)):
            cut_stats["solver_fail"] = cut_stats.get("solver_fail", 0) + 1
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            cut_stats["bad_uinf"] = cut_stats.get("bad_uinf", 0) + 1
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        if idx >= len(r):
            idx = len(r) - 1
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            cut_stats["bad_rs"] = cut_stats.get("bad_rs", 0) + 1
            continue

        rows.append({
            "name": name,
            "r": r,
            "v_obs": v_obs,
            "Mbar": M_enc[-1],
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs
        })

    except:
        cut_stats["pipeline_fail"] = cut_stats.get("pipeline_fail", 0) + 1
        continue

print("Galaxies solved:", len(rows))
print("\nCUT SUMMARY")
for k in sorted(cut_stats):
    print(f"{k}: {cut_stats[k]}")

# -------- Fit amplitude --------
def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

A_global = fit_amplitude(rows)
print("\nA_global =", A_global)

# -------- Evaluate --------
gal_stats = []

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]
    Mbar = row["Mbar"]

    Vflat2 = A_global * Uinf / Rs
    if not np.isfinite(Vflat2) or Vflat2 <= 0:
        continue

    Vflat = np.sqrt(Vflat2)
    y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
    v_pred = Vflat * np.power(y, RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    mae = np.mean(np.abs(v_pred - v_obs))

    x = r / Rs
    frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

    inner_mask = x < 0.7
    mid_mask = (x >= 0.7) & (x <= 2.0)
    outer_mask = x > 2.0

    inner = np.median(frac[inner_mask]) if np.any(inner_mask) else np.nan
    mid   = np.median(frac[mid_mask]) if np.any(mid_mask) else np.nan
    outer = np.median(frac[outer_mask]) if np.any(outer_mask) else np.nan

    gal_stats.append({
        "name": row["name"],
        "Mbar": Mbar,
        "Rs": Rs,
        "Uinf": Uinf,
        "Vflat": Vflat,
        "rmse": rmse,
        "mae": mae,
        "inner": inner,
        "mid": mid,
        "outer": outer
    })

# -------- Global summaries --------
rmse_arr = np.array([g["rmse"] for g in gal_stats])
mae_arr  = np.array([g["mae"]  for g in gal_stats])
M_arr    = np.array([g["Mbar"] for g in gal_stats])
Rs_arr   = np.array([g["Rs"]   for g in gal_stats])
U_arr    = np.array([g["Uinf"] for g in gal_stats])
V_arr    = np.array([g["Vflat"] for g in gal_stats])

logM  = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
logRs = np.log10(Rs_arr)
logU  = np.log10(U_arr)

print("\nGLOBAL SUMMARY")
print("==============================================================")
print("Galaxies used:", len(gal_stats))
print("Median RMSE:", np.median(rmse_arr))
print("Mean RMSE:", np.mean(rmse_arr))
print("Median MAE:", np.median(mae_arr))
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])
print("U slope:", np.polyfit(logM, logU, 1)[0])

inner_all = np.array([g["inner"] for g in gal_stats if np.isfinite(g["inner"])])
mid_all   = np.array([g["mid"]   for g in gal_stats if np.isfinite(g["mid"])])
outer_all = np.array([g["outer"] for g in gal_stats if np.isfinite(g["outer"])])

print("\nRADIAL FRACTIONAL RESIDUALS")
print("==============================================================")
print("Inner median:", np.median(inner_all))
print("Mid median:", np.median(mid_all))
print("Outer median:", np.median(outer_all))

# -------- Mass-bin summaries --------
mass_split = np.median(M_arr)
low  = [g for g in gal_stats if g["Mbar"] < mass_split]
high = [g for g in gal_stats if g["Mbar"] >= mass_split]

def med(key, rows):
    vals = [r[key] for r in rows if np.isfinite(r[key])]
    return np.median(vals) if len(vals) else np.nan

print("\nMASS-BIN SUMMARY")
print("==============================================================")
print("Mass split:", mass_split)
print("LOW  n =", len(low),
      "| RMSE =", med("rmse", low),
      "| inner =", med("inner", low),
      "| mid =", med("mid", low),
      "| outer =", med("outer", low))
print("HIGH n =", len(high),
      "| RMSE =", med("rmse", high),
      "| inner =", med("inner", high),
      "| mid =", med("mid", high),
      "| outer =", med("outer", high))

# -------- Worst / best galaxies --------
worst = sorted(gal_stats, key=lambda x: x["rmse"], reverse=True)[:10]
best  = sorted(gal_stats, key=lambda x: x["rmse"])[:10]

print("\n10 WORST GALAXIES")
print("==============================================================")
for g in worst:
    print(f'{g["name"]:<20} RMSE={g["rmse"]:8.3f}  MAE={g["mae"]:8.3f}  '
          f'M={g["Mbar"]:12.4e}  inner={g["inner"]:6.3f}  mid={g["mid"]:6.3f}  outer={g["outer"]:6.3f}')

print("\n10 BEST GALAXIES")
print("==============================================================")
for g in best:
    print(f'{g["name"]:<20} RMSE={g["rmse"]:8.3f}  MAE={g["mae"]:8.3f}  '
          f'M={g["Mbar"]:12.4e}  inner={g["inner"]:6.3f}  mid={g["mid"]:6.3f}  outer={g["outer"]:6.3f}')

RUNNING FROZEN BEST MODEL
Galaxies solved: 60

CUT SUMMARY
too_few_points: 115

A_global = 2.197254571250284

GLOBAL SUMMARY
Galaxies used: 60
Median RMSE: 38.35869885617602
Mean RMSE: 44.70905013432822
Median MAE: 30.283398256975453
BTFR slope: 0.9101270600343591
Rs slope: 0.31839155356049487
U slope: 0.7734550835776743

RADIAL FRACTIONAL RESIDUALS
Inner median: 0.3054870459157794
Mid median: 0.19120173680657201
Outer median: 0.19085153276475098

MASS-BIN SUMMARY
Mass split: 19950933706.131958
LOW  n = 30 | RMSE = 18.232405551816214 | inner = 0.496582299812823 | mid = 0.18616232394821006 | outer = 0.15596334842010168
HIGH n = 30 | RMSE = 63.25904240869995 | inner = 0.2744187664319836 | mid = 0.1967165702974939 | outer = 0.19874469818791013

10 WORST GALAXIES
NGC5985_rotmod.dat   RMSE= 158.615  MAE= 157.895  M=  1.3334e+11  inner= 0.594  mid= 0.521  outer= 0.501
NGC2841_rotmod.dat   RMSE= 125.959  MAE= 114.850  M=  8.8932e+10  inner= 0.473  mid= 0.250  outer= 0.202
NGC6195_rotmod.dat  

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# STIFFNESS SCREENING SWEEP
#
# Frozen model except stiffness screening:
#
#   div(K_eff grad chi) - mu2 chi = -rho_eff
#
#   K_eff = K0(r) * (1 + B * |dchi/dr|)
#
#   K0(r) = ( M(<r) / r^eta )^delta
#           * (1 + C_phi * (|Phi_b| / Phi_ref)^q_phi )
#           / (1 + (|Phi_b| / Phi_s)^n_scr )
#
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
#   Uchi(r) = ∫ chi(r) r^kappa dr
#   Rs: Uchi(Rs) = (1 - e^-1) Uchi_inf
#   Vflat^2 = A_global * Uchi_inf / Rs
#   V(r) = Vflat * (Uchi/Uinf)^RESP_EXP
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen structure
DELTA = 0.30
ETA   = 0.50
MU2   = 1e-7
BVAL  = 0.10
KAPPA = 0.10

# Frozen source row
A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0

# Frozen response law
RESP_EXP = 0.20

# Frozen best stiffness-potential region
C_PHI = 0.15
Q_PHI = 0.50
PHI_REF = 1.0e4

EPS = 1e-6
STIFF_FLOOR = 1e-3

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, phi_s, n_scr, B, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    phi_ratio = np.clip(v_bar2 / PHI_REF, 0.0, None)
    pot_boost = 1.0 + C_PHI * np.power(phi_ratio, Q_PHI)

    phi_scr_ratio = np.clip(v_bar2 / phi_s, 0.0, None)
    pot_screen = 1.0 / (1.0 + np.power(phi_scr_ratio, n_scr))

    phi_fac = pot_boost * pot_screen
    phi_fac = np.nan_to_num(phi_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + B * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            baseL = ((M_enc[i - 1] / (rL**ETA + EPS))**DELTA) + STIFF_FLOOR
            baseR = ((M_enc[i + 1] / (rR**ETA + EPS))**DELTA) + STIFF_FLOOR

            K0L = baseL * phi_fac[i - 1]
            K0R = baseR * phi_fac[i + 1]

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - MU2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(phi_s, n_scr):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, phi_s, n_scr, BVAL)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })

        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(phi_s, n_scr):
    rows = build_rows(phi_s, n_scr)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)

    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("STIFFNESS SCREENING SWEEP")
print("==============================================================")

for phi_s in [5000.0, 10000.0, 20000.0]:
    for n_scr in [0.5, 1.0, 2.0]:
        res = run_case(phi_s, n_scr)
        print(f"\nphi_s={phi_s:.0f}, n_scr={n_scr:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

STIFFNESS SCREENING SWEEP

phi_s=5000, n_scr=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 34.95257205749295
BTFR slope: 1.0713837365706407
Rs slope: 0.3140164128700809
U slope: 0.8497082811554012
median frac residual inner: 0.3148270460861145
median frac residual mid: 0.24036295205397634
median frac residual outer: 0.25358504597884973
median RMSE low-mass: 19.821431907835475
median RMSE high-mass: 68.47238941711952
Fit LOW -> Test HIGH RMSE: 74.46321284084857
Fit HIGH -> Test LOW RMSE: 19.82042619911349

phi_s=5000, n_scr=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 38.01886837048306
BTFR slope: 1.2111535747192255
Rs slope: 0.30652425764558916
U slope: 0.912101045005202
median frac residual inner: 0.3475808179830191
median frac residual mid: 0.3039715154477438
median frac residual outer: 0.2968394674803898
median RMSE low-mass: 22.34059590405096
median RMSE high-mass: 75.92250137765421
Fit L

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TERM ABLATION TEST FOR FROZEN BEST MODEL
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- FROZEN BEST MODEL --------
BASE = {
    "DELTA": 0.30,
    "ETA": 0.50,
    "MU2": 1e-7,
    "BVAL": 0.10,
    "KAPPA": 0.10,
    "A_G": 0.50,
    "M_EXP": 1.00,
    "GREF": 1000.0,
    "C_PHI": 0.15,
    "Q_PHI": 0.50,
    "PHI_REF": 1.0e4,
    "PHI_S": 20000.0,
    "N_SCR": 1.0,
    "RESP_EXP": 0.20,
}

EPS = 1e-6
STIFF_FLOOR = 1e-3

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, cfg, n_iter=80, relax=0.35):
    n = len(r)
    chi = np.zeros(n)

    phi_ratio = np.clip(v_bar2 / cfg["PHI_REF"], 0.0, None)
    pot_boost = 1.0 + cfg["C_PHI"] * np.power(phi_ratio, cfg["Q_PHI"])

    phi_scr_ratio = np.clip(v_bar2 / cfg["PHI_S"], 0.0, None)
    pot_screen = 1.0 / (1.0 + np.power(phi_scr_ratio, cfg["N_SCR"]))

    phi_fac = pot_boost * pot_screen
    phi_fac = np.nan_to_num(phi_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        grad_fac = 1.0 + cfg["BVAL"] * np.abs(dchi)

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            baseL = ((M_enc[i - 1] / (rL**cfg["ETA"] + EPS))**cfg["DELTA"]) + STIFF_FLOOR
            baseR = ((M_enc[i + 1] / (rR**cfg["ETA"] + EPS))**cfg["DELTA"]) + STIFF_FLOOR

            K0L = baseL * phi_fac[i - 1]
            K0R = baseR * phi_fac[i + 1]

            KL = K0L * grad_fac[i - 1]
            KR = K0R * grad_fac[i + 1]

            AL = (rL**2 * KL) / hL
            AR = (rR**2 * KR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - cfg["MU2"] * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(cfg):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + cfg["A_G"] * (g_b / cfg["GREF"]))**cfg["M_EXP"]
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_clock_field_nonlinear(r, rho_eff, M_enc, v_bar2, cfg)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(cfg["KAPPA"]), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global, cfg):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []
    frac_inner = []
    frac_mid = []
    frac_outer = []
    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, cfg["RESP_EXP"])

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(label, cfg):
    rows = build_rows(cfg)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all, cfg)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low, cfg)
    high_to_low = evaluate_rows(low, A_high, cfg)

    return {
        "label": label,
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

cases = []

# Full frozen best model
cases.append(("FULL_MODEL", BASE.copy()))

# Remove stiffness potential boost + screening entirely
cfg = BASE.copy()
cfg["C_PHI"] = 0.0
cases.append(("NO_PHI_STIFFNESS", cfg))

# Remove gradient stiffening
cfg = BASE.copy()
cfg["BVAL"] = 0.0
cases.append(("NO_GRADIENT_STIFFENING", cfg))

# Remove force-weighted source
cfg = BASE.copy()
cfg["A_G"] = 0.0
cases.append(("NO_FORCE_SOURCE", cfg))

# Remove kappa weighting
cfg = BASE.copy()
cfg["KAPPA"] = 0.0
cases.append(("NO_KAPPA_WEIGHT", cfg))

# Remove nonlinear response
cfg = BASE.copy()
cfg["RESP_EXP"] = 1.0
cases.append(("NO_RESPONSE_NONLINEARITY", cfg))

print("TERM ABLATION TEST")
print("==============================================================")

for label, cfg in cases:
    res = run_case(label, cfg)
    print(f"\n{label}")
    print("--------------------------------------------------------------")
    if res is None or res["full"] is None:
        print("insufficient valid galaxies")
        continue

    full = res["full"]
    print("galaxies:", full["n"])
    print("median RMSE:", full["rmse"])
    print("BTFR slope:", full["btfr"])
    print("Rs slope:", full["rs"])
    print("U slope:", full["u"])
    print("median frac residual inner:", full["inner"])
    print("median frac residual mid:", full["mid"])
    print("median frac residual outer:", full["outer"])
    print("median RMSE low-mass:", full["lowmass_rmse"])
    print("median RMSE high-mass:", full["highmass_rmse"])
    print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
    print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

TERM ABLATION TEST

FULL_MODEL
--------------------------------------------------------------
galaxies: 60
median RMSE: 33.117549414075356
BTFR slope: 1.050373349939118
Rs slope: 0.31614829614532886
U slope: 0.8413349711148879
median frac residual inner: 0.332249354986277
median frac residual mid: 0.24303622132760502
median frac residual outer: 0.24525357292441263
median RMSE low-mass: 21.0448503062876
median RMSE high-mass: 62.897177495151496
Fit LOW -> Test HIGH RMSE: 73.65897227690849
Fit HIGH -> Test LOW RMSE: 20.593160092154097

NO_PHI_STIFFNESS
--------------------------------------------------------------
galaxies: 60
median RMSE: 34.385863942376275
BTFR slope: 1.0755606927875123
Rs slope: 0.3097969949956802
U slope: 0.8475773413894364
median frac residual inner: 0.3318864281615485
median frac residual mid: 0.2299094367850974
median frac residual outer: 0.2528207349991841
median RMSE low-mass: 21.1876717454839
median RMSE high-mass: 71.46115788925397
Fit LOW -> Test HIGH RMSE: 7

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# PURE p-LAPLACIAN SWEEP
#
# Solve radial p-Laplacian-type equation:
#
#   (1/r^2) d/dr [ r^2 |dchi/dr|^(p_op-2) dchi/dr ] - mu2 * chi = -rho_eff
#
# with
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
# Then:
#   Uchi(r) = ∫ chi(r) r^kappa dr
#   Uchi(Rs) = (1 - e^-1) Uinf
#   Vflat^2 = A_global * Uinf / Rs
#   V(r) = Vflat * (Uchi/Uinf)^RESP_EXP
#
# Goal:
#   test whether the complicated stiffness model is approximating
#   a simpler p-Laplacian transport law.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen non-operator pieces
MU2   = 1e-7
KAPPA = 0.10

A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0

RESP_EXP = 0.20

EPS = 1e-6
GRAD_EPS = 1e-8

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_p_laplacian(r, rho_eff, p_op, mu2, n_iter=120, relax=0.30):
    n = len(r)
    chi = np.zeros(n)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)

        # effective conductivity for p-Laplacian
        # |grad|^(p-2), regularized
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(p_op):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_p_laplacian(r, rho_eff, p_op, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(p_op):
    rows = build_rows(p_op)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("PURE p-LAPLACIAN SWEEP")
print("==============================================================")

for p_op in [3.0, 3.5, 4.0, 4.5, 5.0]:
    res = run_case(p_op)
    print(f"\np_op={p_op:.1f}")
    print("--------------------------------------------------------------")
    if res is None or res["full"] is None:
        print("insufficient valid galaxies")
        continue

    full = res["full"]
    print("galaxies:", full["n"])
    print("median RMSE:", full["rmse"])
    print("BTFR slope:", full["btfr"])
    print("Rs slope:", full["rs"])
    print("U slope:", full["u"])
    print("median frac residual inner:", full["inner"])
    print("median frac residual mid:", full["mid"])
    print("median frac residual outer:", full["outer"])
    print("median RMSE low-mass:", full["lowmass_rmse"])
    print("median RMSE high-mass:", full["highmass_rmse"])
    print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
    print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

PURE p-LAPLACIAN SWEEP

p_op=3.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 31.950219914937552
BTFR slope: 1.2266584856722447
Rs slope: 0.30939539173365155
U slope: 0.922724634569774
median frac residual inner: 0.3222819915211902
median frac residual mid: 0.26459062534162703
median frac residual outer: 0.2636621245273222
median RMSE low-mass: 23.26174349123562
median RMSE high-mass: 69.75885339740746
Fit LOW -> Test HIGH RMSE: 74.48356908417944
Fit HIGH -> Test LOW RMSE: 22.01666434193976

p_op=3.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 30.958244006895427
BTFR slope: 1.1224988990179205
Rs slope: 0.3080175881248498
U slope: 0.8692670376338101
median frac residual inner: 0.3000272642434384
median frac residual mid: 0.18714628397197164
median frac residual outer: 0.19890198693729594
median RMSE low-mass: 17.53902290259236
median RMSE high-mass: 66.83075486375688
Fit LOW -> Test HIGH RMSE: 74.41

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# NARROW p-LAPLACIAN SWEEP
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen non-operator pieces
MU2   = 1e-7
KAPPA = 0.10

A_G   = 0.50
M_EXP = 1.00
GREF  = 1000.0

RESP_EXP = 0.20

EPS = 1e-6
GRAD_EPS = 1e-8

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_p_laplacian(r, rho_eff, p_op, mu2, n_iter=140, relax=0.28):
    n = len(r)
    chi = np.zeros(n)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(p_op):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_p_laplacian(r, rho_eff, p_op, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(p_op):
    rows = build_rows(p_op)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("NARROW PURE p-LAPLACIAN SWEEP")
print("==============================================================")

for p_op in [3.2, 3.3, 3.4, 3.5, 3.6, 3.7, 3.8]:
    res = run_case(p_op)
    print(f"\np_op={p_op:.1f}")
    print("--------------------------------------------------------------")
    if res is None or res["full"] is None:
        print("insufficient valid galaxies")
        continue

    full = res["full"]
    print("galaxies:", full["n"])
    print("median RMSE:", full["rmse"])
    print("BTFR slope:", full["btfr"])
    print("Rs slope:", full["rs"])
    print("U slope:", full["u"])
    print("median frac residual inner:", full["inner"])
    print("median frac residual mid:", full["mid"])
    print("median frac residual outer:", full["outer"])
    print("median RMSE low-mass:", full["lowmass_rmse"])
    print("median RMSE high-mass:", full["highmass_rmse"])
    print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
    print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

NARROW PURE p-LAPLACIAN SWEEP

p_op=3.2
--------------------------------------------------------------
galaxies: 60
median RMSE: 83.30599956798082
BTFR slope: 1.2137769839040184
Rs slope: 0.3183385721843715
U slope: 0.9252270641363808
median frac residual inner: 0.6315539991190291
median frac residual mid: 0.6169999953017002
median frac residual outer: 0.6285890761418701
median RMSE low-mass: 43.086636834632415
median RMSE high-mass: 135.9889704946048
Fit LOW -> Test HIGH RMSE: 71.32181570162379
Fit HIGH -> Test LOW RMSE: 44.62701102532098

p_op=3.3
--------------------------------------------------------------
galaxies: 60
median RMSE: 30.61259922846827
BTFR slope: 1.1570998457808996
Rs slope: 0.3181286852528792
U slope: 0.8966786081433289
median frac residual inner: 0.30900952869416753
median frac residual mid: 0.212778113656864
median frac residual outer: 0.22408666512329134
median RMSE low-mass: 17.750104006698322
median RMSE high-mass: 64.88573444440227
Fit LOW -> Test HIGH RMSE: 

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# p-LAPLACIAN + MILD SCREENING SWEEP
#
# Field:
#   (1/r^2) d/dr [ r^2 S(phi_b) |chi'|^(p_op-2) chi' ] - mu2 chi = -rho_eff
#
# Screening:
#   S(phi_b) = 1 / (1 + (phi_b/phi_s)^n_scr)
#
# Source:
#   rho_eff = rho * (1 + A_g * g_b/g_ref)^m
#
# Observable:
#   Uchi(r) = ∫ chi(r) r^kappa dr
#   Uchi(Rs) = (1 - e^-1) Uinf
#   Vflat^2 = A_global * Uinf / Rs
#   V(r) = Vflat * (Uchi/Uinf)^RESP_EXP
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

# Frozen source
A_G    = 0.50
M_EXP  = 1.00
GREF   = 1000.0

# Frozen readout
RESP_EXP = 0.20

EPS = 1e-6
GRAD_EPS = 1e-8

# Potential proxy reference is phi_s itself, so no separate PHI_REF needed

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        vals = [float(x) for x in parts[:5]]
                        rows.append(vals)
                    except:
                        pass
    if len(rows) == 0:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) &
        np.isfinite(v_obs) &
        np.isfinite(v_gas) &
        np.isfinite(v_disk) &
        np.isfinite(v_bul) &
        (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    order = np.argsort(r)
    return r[order], v_obs[order], v_gas[order], v_disk[order], v_bul[order]

def passes_quality_cuts(r, v_obs):
    n = len(r)
    if n < MIN_POINTS:
        return False
    rmin = np.min(r)
    rmax = np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    r_half = 0.5 * rmax
    inner_mask = r <= r_half
    if np.sum(inner_mask) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    # screening from baryonic potential proxy ~ v_bar^2
    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR

            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(phi_s, n_scr):
    files = sorted(glob.glob(ROT_PATH))
    rows = []

    for path in files:
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )

            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**(KAPPA), r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            if idx >= len(r):
                idx = len(r) - 1
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue

    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner = []
    frac_mid = []
    frac_outer = []

    lowmass_rmse = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if len(frac_inner) else np.nan,
        "mid": float(np.median(frac_mid)) if len(frac_mid) else np.nan,
        "outer": float(np.median(frac_outer)) if len(frac_outer) else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if len(lowmass_rmse) else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if len(highmass_rmse) else np.nan,
    }

def run_case(phi_s, n_scr):
    rows = build_rows(phi_s, n_scr)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("p-LAPLACIAN + MILD SCREENING SWEEP")
print("==============================================================")

for phi_s in [10000.0, 20000.0, 40000.0]:
    for n_scr in [0.5, 1.0]:
        res = run_case(phi_s, n_scr)
        print(f"\nphi_s={phi_s:.0f}, n_scr={n_scr:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

p-LAPLACIAN + MILD SCREENING SWEEP

phi_s=10000, n_scr=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 112.32017816743834
BTFR slope: 1.2956311462678434
Rs slope: 0.3206095467424779
U slope: 0.9684251198763997
median frac residual inner: 0.8407307033943472
median frac residual mid: 0.8350284779290755
median frac residual outer: 0.8340480077327289
median RMSE low-mass: 60.22765697546861
median RMSE high-mass: 179.7314884823678
Fit LOW -> Test HIGH RMSE: 63.79934261101761
Fit HIGH -> Test LOW RMSE: 62.599034286727765

phi_s=10000, n_scr=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 34.620707505616245
BTFR slope: 1.285615583651584
Rs slope: 0.31503017395339405
U slope: 0.957837965779186
median frac residual inner: 0.32443815184288205
median frac residual mid: 0.2487247103464375
median frac residual outer: 0.24151622942538742
median RMSE low-mass: 20.982997314684457
median RMSE high-mass: 70.3777614

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SCREENED p-LAPLACIAN + SOURCE-STRENGTH SWEEP
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20

# Frozen best screening row
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, p_op, mu2, phi_s, n_scr, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(A_G):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_bar2 = np.maximum(
                v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
                0.0
            )
            M_enc = r * v_bar2 / G
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, P_OP, MU2, PHI_S, N_SCR)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list, Vflat_list, M_list, Rs_list, U_list = [], [], [], [], []
    frac_inner, frac_mid, frac_outer = [], [], []
    lowmass_rmse, highmass_rmse = [], []

    for row in rows:
        r, v_obs = row["r"], row["v_obs"]
        Uchi, Uinf, Rs, Mbar = row["Uchi"], row["Uinf"], row["Rs"], row["Mbar"]

        Vflat2 = A_global * Uinf / Rs
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue
        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)
        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask   = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM  = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU  = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if frac_inner else np.nan,
        "mid": float(np.median(frac_mid)) if frac_mid else np.nan,
        "outer": float(np.median(frac_outer)) if frac_outer else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if lowmass_rmse else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if highmass_rmse else np.nan,
    }

def run_case(A_G):
    rows = build_rows(A_G)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)

    low  = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("SCREENED p-LAPLACIAN + SOURCE-STRENGTH SWEEP")
print("==============================================================")

for A_G in [0.20, 0.30, 0.40, 0.50, 0.60]:
    res = run_case(A_G)
    print(f"\nA_G={A_G:.2f}")
    print("--------------------------------------------------------------")
    if res is None or res["full"] is None:
        print("insufficient valid galaxies")
        continue

    full = res["full"]
    print("galaxies:", full["n"])
    print("median RMSE:", full["rmse"])
    print("BTFR slope:", full["btfr"])
    print("Rs slope:", full["rs"])
    print("U slope:", full["u"])
    print("median frac residual inner:", full["inner"])
    print("median frac residual mid:", full["mid"])
    print("median frac residual outer:", full["outer"])
    print("median RMSE low-mass:", full["lowmass_rmse"])
    print("median RMSE high-mass:", full["highmass_rmse"])
    print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
    print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

SCREENED p-LAPLACIAN + SOURCE-STRENGTH SWEEP

A_G=0.20
--------------------------------------------------------------
galaxies: 60
median RMSE: 29.938643204513703
BTFR slope: 1.1235776168747285
Rs slope: 0.31721791844341546
U slope: 0.8790067268807795
median frac residual inner: 0.2909837021084201
median frac residual mid: 0.16739873529974264
median frac residual outer: 0.1869089710796011
median RMSE low-mass: 16.93186002761295
median RMSE high-mass: 61.666734475880816
Fit LOW -> Test HIGH RMSE: 67.93601098263699
Fit HIGH -> Test LOW RMSE: 16.90208565652157

A_G=0.30
--------------------------------------------------------------
galaxies: 60
median RMSE: 28.065351522245486
BTFR slope: 1.1490873718629226
Rs slope: 0.3176377981799015
U slope: 0.8921814841113629
median frac residual inner: 0.2959463084882157
median frac residual mid: 0.19549362268248924
median frac residual outer: 0.20587284231126826
median RMSE low-mass: 17.140043226906656
median RMSE high-mass: 67.97727473146305
Fit LOW

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# WORST-GALAXY SIGNED-RESIDUAL FORENSIC DIAGNOSTIC
#
# Frozen current-best model:
#   screened p-Laplacian backbone
#   with current best practical row from recent tests
#
# Outputs:
#   - global summary
#   - worst 15 galaxies by RMSE
#   - signed inner/mid/outer residuals
#   - amplitude mismatch
#   - rough failure labels
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen model --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40          # current best global-RMSE compromise from last sweep
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False, "too_few_points"
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False, "bad_radius"
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False, "insufficient_radial_range"
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False, "too_few_outer_points"
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False, "bad_outer_velocity"
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False, "not_flat_outer"
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False, "insufficient_inner_resolution"
    return True, "pass"

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def median_signed(arr):
    arr = arr[np.isfinite(arr)]
    return np.median(arr) if len(arr) else np.nan

def classify_failure(s_in, s_mid, s_out, amp_frac):
    # signed residuals are (pred - obs)/obs
    zones = [("inner", s_in), ("mid", s_mid), ("outer", s_out)]
    zones = [(z, v) for z, v in zones if np.isfinite(v)]
    if not zones:
        return "unknown"

    dom_zone, dom_val = max(zones, key=lambda t: abs(t[1]))

    if np.isfinite(amp_frac) and abs(amp_frac) > 0.20:
        amp_tag = "global_high" if amp_frac > 0 else "global_low"
    else:
        amp_tag = "shape"

    zone_tag = f"{dom_zone}_{'high' if dom_val > 0 else 'low'}"
    return f"{amp_tag}:{zone_tag}"

# -------- Build solved rows --------
files = sorted(glob.glob(ROT_PATH))
rows = []
cut_stats = {}

print("RUNNING WORST-GALAXY FORENSIC DIAGNOSTIC")
print("==============================================================")

for path in files:
    data = read_file(path)
    if data is None:
        cut_stats["read_fail"] = cut_stats.get("read_fail", 0) + 1
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    passed, reason = passes_quality_cuts(r, v_obs)
    if not passed:
        cut_stats[reason] = cut_stats.get(reason, 0) + 1
        continue

    try:
        name = os.path.basename(path)

        v_bar2 = np.maximum(
            v_gas**2 + (UPS_DISK * v_disk)**2 + (UPS_BUL * v_bul)**2,
            0.0
        )

        M_enc = r * v_bar2 / G
        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

        g_b = v_bar2 / (r + EPS)
        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            cut_stats["solver_fail"] = cut_stats.get("solver_fail", 0) + 1
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            cut_stats["bad_uinf"] = cut_stats.get("bad_uinf", 0) + 1
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            cut_stats["bad_rs"] = cut_stats.get("bad_rs", 0) + 1
            continue

        rows.append({
            "name": name,
            "r": r,
            "v_obs": v_obs,
            "Mbar": M_enc[-1],
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs
        })
    except:
        cut_stats["pipeline_fail"] = cut_stats.get("pipeline_fail", 0) + 1
        continue

print("Galaxies solved:", len(rows))
print("\nCUT SUMMARY")
for k in sorted(cut_stats):
    print(f"{k}: {cut_stats[k]}")

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

A_global = fit_amplitude(rows)
print("\nA_global =", A_global)

# -------- Evaluate all galaxies --------
gal_stats = []

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]
    Mbar = row["Mbar"]

    Vflat2 = A_global * Uinf / Rs
    if not np.isfinite(Vflat2) or Vflat2 <= 0:
        continue

    Vflat = np.sqrt(Vflat2)
    y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
    v_pred = Vflat * np.power(y, RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    mae = np.mean(np.abs(v_pred - v_obs))

    frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)
    frac_abs = np.abs(frac_signed)

    x = r / Rs
    inner_mask = x < 0.7
    mid_mask   = (x >= 0.7) & (x <= 2.0)
    outer_mask = x > 2.0

    s_inner = median_signed(frac_signed[inner_mask]) if np.any(inner_mask) else np.nan
    s_mid   = median_signed(frac_signed[mid_mask])   if np.any(mid_mask) else np.nan
    s_outer = median_signed(frac_signed[outer_mask]) if np.any(outer_mask) else np.nan

    a_inner = np.median(frac_abs[inner_mask]) if np.any(inner_mask) else np.nan
    a_mid   = np.median(frac_abs[mid_mask])   if np.any(mid_mask) else np.nan
    a_outer = np.median(frac_abs[outer_mask]) if np.any(outer_mask) else np.nan

    vobs_max = np.max(v_obs)
    amp_frac = (Vflat - vobs_max) / max(vobs_max, 1.0)

    label = classify_failure(s_inner, s_mid, s_outer, amp_frac)

    gal_stats.append({
        "name": row["name"],
        "Mbar": Mbar,
        "Rs": Rs,
        "Uinf": Uinf,
        "Vflat_pred": Vflat,
        "Vobs_max": vobs_max,
        "amp_frac": amp_frac,
        "rmse": rmse,
        "mae": mae,
        "s_inner": s_inner,
        "s_mid": s_mid,
        "s_outer": s_outer,
        "a_inner": a_inner,
        "a_mid": a_mid,
        "a_outer": a_outer,
        "label": label
    })

# -------- Global summary --------
rmse_arr = np.array([g["rmse"] for g in gal_stats])
M_arr    = np.array([g["Mbar"] for g in gal_stats])
Rs_arr   = np.array([g["Rs"] for g in gal_stats])
U_arr    = np.array([g["Uinf"] for g in gal_stats])
V_arr    = np.array([g["Vflat_pred"] for g in gal_stats])

logM  = np.log10(M_arr)
logV4 = np.log10(V_arr**4)
logRs = np.log10(Rs_arr)
logU  = np.log10(U_arr)

print("\nGLOBAL SUMMARY")
print("==============================================================")
print("Galaxies used:", len(gal_stats))
print("Median RMSE:", np.median(rmse_arr))
print("Mean RMSE:", np.mean(rmse_arr))
print("BTFR slope:", np.polyfit(logM, logV4, 1)[0])
print("Rs slope:", np.polyfit(logM, logRs, 1)[0])
print("U slope:", np.polyfit(logM, logU, 1)[0])

print("\nGLOBAL SIGNED RESIDUAL MEDIANS")
print("==============================================================")
print("Inner:", np.median([g["s_inner"] for g in gal_stats if np.isfinite(g["s_inner"])]))
print("Mid  :", np.median([g["s_mid"]   for g in gal_stats if np.isfinite(g["s_mid"])]))
print("Outer:", np.median([g["s_outer"] for g in gal_stats if np.isfinite(g["s_outer"])]))

# -------- Worst galaxies --------
worst = sorted(gal_stats, key=lambda x: x["rmse"], reverse=True)[:15]

print("\nWORST 15 GALAXIES (SIGNED FAILURE TABLE)")
print("========================================================================================================================")
print(f'{"name":<20} {"RMSE":>8} {"Mbar":>12} {"Vf_pred":>9} {"Vmax":>9} {"amp%":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"label":>20}')
for g in worst:
    print(
        f'{g["name"]:<20} '
        f'{g["rmse"]:8.2f} '
        f'{g["Mbar"]:12.3e} '
        f'{g["Vflat_pred"]:9.2f} '
        f'{g["Vobs_max"]:9.2f} '
        f'{100*g["amp_frac"]:8.1f} '
        f'{g["s_inner"]:8.3f} '
        f'{g["s_mid"]:8.3f} '
        f'{g["s_outer"]:8.3f} '
        f'{g["label"]:>20}'
    )

# -------- Failure-mode counts --------
from collections import Counter
labels = Counter([g["label"] for g in worst])

print("\nFAILURE MODE COUNTS IN WORST 15")
print("==============================================================")
for k, v in labels.most_common():
    print(f"{k}: {v}")

# -------- Best galaxies too, for contrast --------
best = sorted(gal_stats, key=lambda x: x["rmse"])[:10]

print("\nBEST 10 GALAXIES (SIGNED FAILURE TABLE)")
print("========================================================================================================================")
print(f'{"name":<20} {"RMSE":>8} {"Mbar":>12} {"Vf_pred":>9} {"Vmax":>9} {"amp%":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"label":>20}')
for g in best:
    print(
        f'{g["name"]:<20} '
        f'{g["rmse"]:8.2f} '
        f'{g["Mbar"]:12.3e} '
        f'{g["Vflat_pred"]:9.2f} '
        f'{g["Vobs_max"]:9.2f} '
        f'{100*g["amp_frac"]:8.1f} '
        f'{g["s_inner"]:8.3f} '
        f'{g["s_mid"]:8.3f} '
        f'{g["s_out"]:8.3f} '
        f'{g["label"]:>20}'
    )

RUNNING WORST-GALAXY FORENSIC DIAGNOSTIC
Galaxies solved: 60

CUT SUMMARY
too_few_points: 115

A_global = 1.0340397357490716

GLOBAL SUMMARY
Galaxies used: 60
Median RMSE: 27.723895695091663
Mean RMSE: 42.19329213248235
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284

GLOBAL SIGNED RESIDUAL MEDIANS
Inner: -0.1415816087880079
Mid  : -0.13659295715801179
Outer: -0.12192394023268141

WORST 15 GALAXIES (SIGNED FAILURE TABLE)
name                     RMSE         Mbar   Vf_pred      Vmax     amp%     s_in    s_mid    s_out                label
NGC5985_rotmod.dat     144.10    1.333e+11    163.00    305.00    -46.6   -0.551   -0.462   -0.434 global_low:inner_low
NGC2841_rotmod.dat     112.74    8.893e+10    248.72    323.00    -23.0   -0.428   -0.178   -0.122 global_low:inner_low
NGC4013_rotmod.dat      93.71    3.395e+10     97.57    198.00    -50.7   -0.637   -0.440   -0.455 global_low:inner_low
UGC11914_rotmod.dat     93.62    5.576e+10    215.13    3

KeyError: 's_out'

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# WORST-vs-BEST FEATURE COMPARISON
#
# Goal:
#   Compare the bad galaxies against the good galaxies using
#   directly observable structural features from the rotmod files.
#
# Features extracted:
#   - Mbar
#   - Vobs_max
#   - Vbar_max
#   - Vbar/Vobs max ratio
#   - bulge fraction proxy
#   - disk fraction proxy
#   - gas fraction proxy
#   - radius of Vobs max
#   - radius of Vbar max
#   - outer slope of observed curve
#   - inner slope of observed curve
#   - concentration proxy: Vobs(r<0.3 rmax)/Vmax
#   - concentration proxy: Vbar(r<0.3 rmax)/Vbar_max
#
# It also re-runs the frozen model so we can compare worst vs best
# using the same RMSE ranking.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen model from last forensic run --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def safe_grad(y, x):
    if len(y) < 3:
        return np.full_like(y, np.nan)
    return np.gradient(y, x)

def feature_extract(name, r, v_obs, v_gas, v_disk, v_bul):
    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul

    v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
    v_bar  = np.sqrt(v_bar2)

    M_enc = r * v_bar2 / G
    Mbar = M_enc[-1]

    vmax_obs = np.max(v_obs)
    vmax_bar = np.max(v_bar)

    i_vmax_obs = int(np.argmax(v_obs))
    i_vmax_bar = int(np.argmax(v_bar))

    r_vmax_obs = r[i_vmax_obs]
    r_vmax_bar = r[i_vmax_bar]

    # component fractions at radius of observed max
    vg = abs(v_gas[i_vmax_obs])
    vd = abs(v_disk_s[i_vmax_obs])
    vb = abs(v_bul_s[i_vmax_obs])
    denom = vg + vd + vb + EPS

    gas_frac  = vg / denom
    disk_frac = vd / denom
    bulge_frac = vb / denom

    # inner concentration proxies
    rmax = np.max(r)
    inner_core = r <= 0.3 * rmax
    if np.any(inner_core):
        c_obs = np.max(v_obs[inner_core]) / max(vmax_obs, EPS)
        c_bar = np.max(v_bar[inner_core]) / max(vmax_bar, EPS)
    else:
        c_obs = np.nan
        c_bar = np.nan

    # inner / outer slopes
    dvdr_obs = safe_grad(v_obs, r)

    inner_zone = r <= 0.3 * rmax
    outer_zone = r >= 0.7 * rmax

    inner_slope = np.median(dvdr_obs[inner_zone]) if np.any(inner_zone) else np.nan
    outer_slope = np.median(dvdr_obs[outer_zone]) if np.any(outer_zone) else np.nan

    # outer flatness ratio
    if np.any(outer_zone):
        outer_med = np.median(v_obs[outer_zone])
        outer_edge = v_obs[-1]
        outer_flat_ratio = outer_med / max(outer_edge, EPS)
    else:
        outer_flat_ratio = np.nan

    return {
        "name": name,
        "Mbar": Mbar,
        "Vobs_max": vmax_obs,
        "Vbar_max": vmax_bar,
        "Vbar_over_Vobs": vmax_bar / max(vmax_obs, EPS),
        "r_vmax_obs": r_vmax_obs,
        "r_vmax_bar": r_vmax_bar,
        "rmax": rmax,
        "gas_frac": gas_frac,
        "disk_frac": disk_frac,
        "bulge_frac": bulge_frac,
        "c_obs": c_obs,
        "c_bar": c_bar,
        "inner_slope": inner_slope,
        "outer_slope": outer_slope,
        "outer_flat_ratio": outer_flat_ratio,
        "r": r,
        "v_obs": v_obs,
        "v_bar2": v_bar2,
        "M_enc": M_enc
    }

print("RUNNING WORST-vs-BEST FEATURE COMPARISON")
print("==============================================================")

features = []
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    name = os.path.basename(path)
    feat = feature_extract(name, r, v_obs, v_gas, v_disk, v_bul)

    try:
        v_bar2 = feat["v_bar2"]
        M_enc = feat["M_enc"]
        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

        g_b = v_bar2 / (r + EPS)
        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": name,
            "r": r,
            "v_obs": v_obs,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": feat["Mbar"]
        })
        features.append(feat)
    except:
        continue

# Fit amplitude
num = 0.0
den = 0.0
for row in rows:
    y = np.max(row["v_obs"])**2
    x = row["Uinf"] / row["Rs"]
    num += x * y
    den += x * x
A_global = num / den

# Attach model outputs + rmse
feat_map = {f["name"]: f for f in features}
all_stats = []

for row in rows:
    name = row["name"]
    f = feat_map[name]
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat = np.sqrt(A_global * Uinf / Rs)
    v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    mae = np.mean(np.abs(v_pred - v_obs))

    all_stats.append({
        **{k: v for k, v in f.items() if k not in ["r", "v_obs", "v_bar2", "M_enc"]},
        "Rs": Rs,
        "Uinf": Uinf,
        "Vflat_pred": Vflat,
        "rmse": rmse,
        "mae": mae
    })

worst = sorted(all_stats, key=lambda x: x["rmse"], reverse=True)[:15]
best  = sorted(all_stats, key=lambda x: x["rmse"])[:15]

def median_of(group, key):
    vals = [g[key] for g in group if np.isfinite(g[key])]
    return np.median(vals) if vals else np.nan

keys = [
    "Mbar",
    "Vobs_max",
    "Vbar_max",
    "Vbar_over_Vobs",
    "r_vmax_obs",
    "r_vmax_bar",
    "gas_frac",
    "disk_frac",
    "bulge_frac",
    "c_obs",
    "c_bar",
    "inner_slope",
    "outer_slope",
    "outer_flat_ratio",
    "Vflat_pred",
    "Rs",
    "Uinf",
    "rmse"
]

print("\nGLOBAL A_global")
print("==============================================================")
print("A_global =", A_global)

print("\nWORST 15 GALAXIES")
print("==============================================================")
for g in worst:
    print(
        f'{g["name"]:<20} '
        f'RMSE={g["rmse"]:7.2f}  '
        f'M={g["Mbar"]:10.3e}  '
        f'Vmax={g["Vobs_max"]:7.1f}  '
        f'Vflat_pred={g["Vflat_pred"]:7.1f}  '
        f'bulge_frac={g["bulge_frac"]:5.2f}  '
        f'gas_frac={g["gas_frac"]:5.2f}  '
        f'outer_slope={g["outer_slope"]:7.3f}'
    )

print("\nBEST 15 GALAXIES")
print("==============================================================")
for g in best:
    print(
        f'{g["name"]:<20} '
        f'RMSE={g["rmse"]:7.2f}  '
        f'M={g["Mbar"]:10.3e}  '
        f'Vmax={g["Vobs_max"]:7.1f}  '
        f'Vflat_pred={g["Vflat_pred"]:7.1f}  '
        f'bulge_frac={g["bulge_frac"]:5.2f}  '
        f'gas_frac={g["gas_frac"]:5.2f}  '
        f'outer_slope={g["outer_slope"]:7.3f}'
    )

print("\nWORST vs BEST MEDIAN FEATURE COMPARISON")
print("==============================================================")
for k in keys:
    w = median_of(worst, k)
    b = median_of(best, k)
    ratio = w / b if np.isfinite(w) and np.isfinite(b) and abs(b) > 1e-30 else np.nan
    print(f"{k:<18} worst={w:12.5g}   best={b:12.5g}   ratio={ratio:10.4g}")

# Simple categorical pattern counts
def cat_counts(group, key, thresholds):
    counts = {}
    vals = [g[key] for g in group if np.isfinite(g[key])]
    for label, lo, hi in thresholds:
        counts[label] = sum((v >= lo) and (v < hi) for v in vals)
    return counts

print("\nBULGE FRACTION BINS (at observed Vmax radius)")
print("==============================================================")
bins = [
    ("low(<0.1)", 0.0, 0.1),
    ("mid(0.1-0.3)", 0.1, 0.3),
    ("high(>0.3)", 0.3, 10.0),
]
print("Worst:", cat_counts(worst, "bulge_frac", bins))
print("Best :", cat_counts(best, "bulge_frac", bins))

print("\nGAS FRACTION BINS (at observed Vmax radius)")
print("==============================================================")
print("Worst:", cat_counts(worst, "gas_frac", bins))
print("Best :", cat_counts(best, "gas_frac", bins))

print("\nOUTER SLOPE BINS")
print("==============================================================")
slope_bins = [
    ("declining(<-1)", -1e9, -1.0),
    ("mild_decline(-1 to -0.2)", -1.0, -0.2),
    ("flatish(-0.2 to 0.2)", -0.2, 0.2),
    ("rising(>0.2)", 0.2, 1e9),
]
print("Worst:", cat_counts(worst, "outer_slope", slope_bins))
print("Best :", cat_counts(best, "outer_slope", slope_bins))

RUNNING WORST-vs-BEST FEATURE COMPARISON

GLOBAL A_global
A_global = 1.0340397357490716

WORST 15 GALAXIES
NGC5985_rotmod.dat   RMSE= 144.10  M= 1.333e+11  Vmax=  305.0  Vflat_pred=  163.0  bulge_frac= 0.90  gas_frac= 0.06  outer_slope= -1.014
NGC2841_rotmod.dat   RMSE= 112.74  M= 8.893e+10  Vmax=  323.0  Vflat_pred=  248.7  bulge_frac= 0.89  gas_frac= 0.04  outer_slope=  1.461
NGC4013_rotmod.dat   RMSE=  93.71  M= 3.395e+10  Vmax=  198.0  Vflat_pred=   97.6  bulge_frac= 0.93  gas_frac= 0.07  outer_slope=  0.000
UGC11914_rotmod.dat  RMSE=  93.62  M= 5.576e+10  Vmax=  305.0  Vflat_pred=  215.1  bulge_frac= 0.89  gas_frac= 0.03  outer_slope=  6.876
UGC06787_rotmod.dat  RMSE=  89.34  M= 2.954e+10  Vmax=  276.0  Vflat_pred=  196.9  bulge_frac= 0.64  gas_frac= 0.34  outer_slope=  0.497
NGC6195_rotmod.dat   RMSE=  86.59  M= 1.523e+11  Vmax=  258.0  Vflat_pred=  191.4  bulge_frac= 0.77  gas_frac= 0.07  outer_slope= -0.558
UGC02953_rotmod.dat  RMSE=  86.43  M= 1.182e+11  Vmax=  319.0  Vflat_pr

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# RMSE vs COMPACTNESS DIAGNOSTIC
#
# Goal:
#   Test whether model error correlates with baryonic compactness.
#
# Compactness metrics tested:
#   C1 = Vbar(0.3 Rmax) / Vbar_max
#   C2 = r(Vbar_max) / Rmax
#   C3 = M(<0.3 Rmax) / M(<Rmax)
#
# Also prints correlation coefficients with RMSE.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen model (same as before) --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

print("RMSE vs COMPACTNESS DIAGNOSTIC")
print("==============================================================")

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        name = os.path.basename(path)

        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        v_bar  = np.sqrt(v_bar2)

        M_enc = r * v_bar2 / G
        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

        g_b = v_bar2 / (r + EPS)
        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        # Amplitude later, store row
        rows.append({
            "name": name,
            "r": r,
            "v_obs": v_obs,
            "v_bar": v_bar,
            "v_bar2": v_bar2,
            "M_enc": M_enc,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs
        })
    except:
        continue

# Fit amplitude
num = 0.0
den = 0.0
for row in rows:
    y = np.max(row["v_obs"])**2
    x = row["Uinf"] / row["Rs"]
    num += x * y
    den += x * x
A_global = num / den

print("A_global =", A_global)
print()

# Compute RMSE + compactness
rmse_list = []
C1_list = []
C2_list = []
C3_list = []
names = []

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    v_bar = row["v_bar"]
    v_bar2 = row["v_bar2"]
    M_enc = row["M_enc"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat = np.sqrt(A_global * Uinf / Rs)
    v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))

    rmax = np.max(r)
    vmax_bar = np.max(v_bar)
    i_vmax_bar = int(np.argmax(v_bar))
    r_vmax_bar = r[i_vmax_bar]

    # C1
    inner = r <= 0.3 * rmax
    if np.any(inner):
        C1 = np.max(v_bar[inner]) / max(vmax_bar, EPS)
        M_inner = M_enc[inner][-1]
    else:
        C1 = np.nan
        M_inner = np.nan

    # C2
    C2 = r_vmax_bar / rmax

    # C3
    M_total = M_enc[-1]
    C3 = M_inner / max(M_total, EPS)

    rmse_list.append(rmse)
    C1_list.append(C1)
    C2_list.append(C2)
    C3_list.append(C3)
    names.append(row["name"])

rmse_arr = np.array(rmse_list)
C1_arr = np.array(C1_list)
C2_arr = np.array(C2_list)
C3_arr = np.array(C3_list)

def corr(a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    if np.sum(mask) < 5:
        return np.nan
    return np.corrcoef(a[mask], b[mask])[0, 1]

print("CORRELATIONS WITH RMSE")
print("==============================================================")
print("corr(RMSE, C1 = Vbar(0.3R)/Vbar_max):", corr(rmse_arr, C1_arr))
print("corr(RMSE, C2 = r(Vbar_max)/Rmax):   ", corr(rmse_arr, C2_arr))
print("corr(RMSE, C3 = M(<0.3R)/Mtot):     ", corr(rmse_arr, C3_arr))

# Print top compact and bottom compact galaxies
order = np.argsort(rmse_arr)

print("\nLOW RMSE / HIGH COMPACTNESS EXAMPLES")
print("==============================================================")
for i in order[:10]:
    print(names[i], "RMSE=", rmse_arr[i], "C1=", C1_arr[i], "C2=", C2_arr[i], "C3=", C3_arr[i])

print("\nHIGH RMSE / HIGH COMPACTNESS EXAMPLES")
print("==============================================================")
for i in order[-10:]:
    print(names[i], "RMSE=", rmse_arr[i], "C1=", C1_arr[i], "C2=", C2_arr[i], "C3=", C3_arr[i])

RMSE vs COMPACTNESS DIAGNOSTIC
A_global = 1.0340397357490716

CORRELATIONS WITH RMSE
corr(RMSE, C1 = Vbar(0.3R)/Vbar_max): 0.24544815486384497
corr(RMSE, C2 = r(Vbar_max)/Rmax):    -0.25262961711108056
corr(RMSE, C3 = M(<0.3R)/Mtot):      0.3371816641587109

LOW RMSE / HIGH COMPACTNESS EXAMPLES
UGCA444_rotmod.dat RMSE= 5.864654413470385 C1= 0.6118632681171574 C2= 1.0 C3= 0.10859780944349863
NGC2366_rotmod.dat RMSE= 8.278852736849531 C1= 0.9670447192754168 C2= 0.3729372937293729 C3= 0.3131111691842487
NGC4559_rotmod.dat RMSE= 8.787664447108712 C1= 1.0 C2= 0.21840724845016693 C3= 0.6613278263796826
DDO161_rotmod.dat RMSE= 9.197066658034 C1= 1.0 C2= 0.2916978309648467 C3= 0.49149162450040124
NGC3741_rotmod.dat RMSE= 10.55361529940926 C1= 1.0 C2= 0.06714285714285714 C3= 0.26919175897150527
IC2574_rotmod.dat RMSE= 11.070121462824359 C1= 0.5905987509067521 C2= 0.8328445747800586 C3= 0.10288192792429941
NGC6015_rotmod.dat RMSE= 11.914701627623224 C1= 1.0 C2= 0.15737256243585357 C3= 0.95426300

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# COMPACTNESS-DEPENDENT AMPLITUDE SWEEP
#
# Frozen model backbone:
#   screened p-Laplacian
#
# Only change:
#   Vflat^2 = A_global * (Uinf / Rs) * (1 + Bc * C^alpha)
#
# Compactness proxy:
#   C = M(<0.3 Rmax) / M(<Rmax)
#
# Goal:
#   test whether the bad compact/bulge-heavy galaxies are mainly
#   missing an amplitude correction rather than new operator physics.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rmax = np.max(r)
            inner = r <= 0.3 * rmax
            M_inner = M_enc[inner][-1] if np.any(inner) else np.nan
            C = M_inner / max(M_enc[-1], EPS)

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "C": C
            })
        except:
            continue
    return rows

def fit_amplitude(rows, Bc, alpha):
    num = 0.0
    den = 0.0
    for row in rows:
        boost = 1.0 + Bc * (row["C"] ** alpha)
        x = (row["Uinf"] / row["Rs"]) * boost
        y = np.max(row["v_obs"])**2
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global, Bc, alpha):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner, frac_mid, frac_outer = [], [], []
    lowmass_rmse, highmass_rmse = [], []

    for row in rows:
        r, v_obs = row["r"], row["v_obs"]
        Uchi, Uinf, Rs, Mbar, C = row["Uchi"], row["Uinf"], row["Rs"], row["Mbar"], row["C"]

        boost = 1.0 + Bc * (C ** alpha)
        Vflat2 = A_global * (Uinf / Rs) * boost
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask   = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM  = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU  = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if frac_inner else np.nan,
        "mid": float(np.median(frac_mid)) if frac_mid else np.nan,
        "outer": float(np.median(frac_outer)) if frac_outer else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if lowmass_rmse else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if highmass_rmse else np.nan,
    }

def run_case(Bc, alpha):
    rows = build_rows()
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows, Bc, alpha)
    full = evaluate_rows(rows, A_all, Bc, alpha)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low  = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low, Bc, alpha)
    A_high = fit_amplitude(high, Bc, alpha)

    low_to_high = evaluate_rows(high, A_low, Bc, alpha)
    high_to_low = evaluate_rows(low, A_high, Bc, alpha)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("COMPACTNESS-DEPENDENT AMPLITUDE SWEEP")
print("==============================================================")

for Bc in [0.0, 0.5, 1.0, 2.0]:
    for alpha in [0.5, 1.0, 2.0]:
        res = run_case(Bc, alpha)
        print(f"\nBc={Bc:.1f}, alpha={alpha:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

COMPACTNESS-DEPENDENT AMPLITUDE SWEEP

Bc=0.0, alpha=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median RMSE low-mass: 17.525954837779878
median RMSE high-mass: 69.01015888909751
Fit LOW -> Test HIGH RMSE: 66.36458209704
Fit HIGH -> Test LOW RMSE: 17.490191732455735

Bc=0.0, alpha=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median RMSE low-mass: 17.525954837779878
median RMSE high-mass: 69.01015888909751


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SHAPE-FAILURE CLASSIFIER FOR WORST vs BEST GALAXIES
#
# Goal:
#   Compare observed, baryonic, and predicted curve shapes
#   and classify failures into:
#     - early-rise failure
#     - plateau-amplitude failure
#     - outer-tail failure
#     - two-stage structure failure
#
# It prints:
#   1) global summary
#   2) worst 12 galaxies with shape diagnostics
#   3) best 12 galaxies with same diagnostics
#   4) failure-type counts in worst sample
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def med_signed(arr):
    arr = arr[np.isfinite(arr)]
    return np.median(arr) if len(arr) else np.nan

def classify_shape_failure(r, v_obs, v_pred, Rs):
    x = r / max(Rs, EPS)
    frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

    inner = x < 0.7
    mid   = (x >= 0.7) & (x <= 2.0)
    outer = x > 2.0

    s_in  = med_signed(frac_signed[inner]) if np.any(inner) else np.nan
    s_mid = med_signed(frac_signed[mid])   if np.any(mid) else np.nan
    s_out = med_signed(frac_signed[outer]) if np.any(outer) else np.nan

    # observed vs predicted normalized shapes
    vmax_obs = max(np.max(v_obs), EPS)
    vmax_pred = max(np.max(v_pred), EPS)

    obs_n = v_obs / vmax_obs
    pred_n = v_pred / vmax_pred

    # normalized shape residuals
    shape_res = pred_n - obs_n
    sh_in  = med_signed(shape_res[inner]) if np.any(inner) else np.nan
    sh_mid = med_signed(shape_res[mid])   if np.any(mid) else np.nan
    sh_out = med_signed(shape_res[outer]) if np.any(outer) else np.nan

    # amplitude mismatch
    amp_frac = (vmax_pred - vmax_obs) / vmax_obs

    # classifier logic
    # 1) plateau-amplitude failure: same shape-ish, wrong amplitude overall
    if np.isfinite(amp_frac) and abs(amp_frac) > 0.20:
        if all(np.isfinite(v) for v in [sh_in, sh_mid, sh_out]) and max(abs(sh_in), abs(sh_mid), abs(sh_out)) < 0.12:
            return "plateau-amplitude", s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac

    # 2) early-rise failure: inner wrong more than mid/outer in normalized shape
    if np.isfinite(sh_in) and abs(sh_in) > 0.12:
        comp = max(abs(sh_mid) if np.isfinite(sh_mid) else 0, abs(sh_out) if np.isfinite(sh_out) else 0)
        if abs(sh_in) > comp + 0.05:
            return "early-rise", s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac

    # 3) outer-tail failure: outer wrong more than inner/mid in normalized shape
    if np.isfinite(sh_out) and abs(sh_out) > 0.12:
        comp = max(abs(sh_in) if np.isfinite(sh_in) else 0, abs(sh_mid) if np.isfinite(sh_mid) else 0)
        if abs(sh_out) > comp + 0.05:
            return "outer-tail", s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac

    # 4) two-stage structure: inner and outer go opposite ways or strong mixed pattern
    if all(np.isfinite(v) for v in [sh_in, sh_mid, sh_out]):
        if (sh_in * sh_out < 0) or (abs(sh_in) > 0.10 and abs(sh_out) > 0.10 and abs(sh_mid) < max(abs(sh_in), abs(sh_out))):
            return "two-stage", s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac

    return "mixed/weak", s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac

print("RUNNING SHAPE-FAILURE CLASSIFIER")
print("==============================================================")

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)
        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_bar": np.sqrt(v_bar2),
            "Mbar": M_enc[-1],
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs
        })
    except:
        continue

# Fit global amplitude
num = 0.0
den = 0.0
for row in rows:
    y = np.max(row["v_obs"])**2
    x = row["Uinf"] / row["Rs"]
    num += x * y
    den += x * x
A_global = num / den

print("A_global =", A_global)
print()

stats = []

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat = np.sqrt(A_global * Uinf / Rs)
    v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    shape_label, s_in, s_mid, s_out, sh_in, sh_mid, sh_out, amp_frac = classify_shape_failure(r, v_obs, v_pred, Rs)

    stats.append({
        "name": row["name"],
        "Mbar": row["Mbar"],
        "Rs": Rs,
        "Vflat_pred": Vflat,
        "Vmax_obs": np.max(v_obs),
        "rmse": rmse,
        "shape_label": shape_label,
        "s_in": s_in,
        "s_mid": s_mid,
        "s_out": s_out,
        "sh_in": sh_in,
        "sh_mid": sh_mid,
        "sh_out": sh_out,
        "amp_frac": amp_frac
    })

worst = sorted(stats, key=lambda x: x["rmse"], reverse=True)[:12]
best  = sorted(stats, key=lambda x: x["rmse"])[:12]

print("WORST 12 GALAXIES")
print("================================================================================================================================")
print(f'{"name":<20} {"RMSE":>8} {"Mbar":>12} {"Vf":>8} {"Vmax":>8} {"amp%":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"sh_in":>8} {"sh_mid":>8} {"sh_out":>8} {"type":>18}')
for g in worst:
    print(
        f'{g["name"]:<20} '
        f'{g["rmse"]:8.2f} '
        f'{g["Mbar"]:12.3e} '
        f'{g["Vflat_pred"]:8.1f} '
        f'{g["Vmax_obs"]:8.1f} '
        f'{100*g["amp_frac"]:8.1f} '
        f'{g["s_in"]:8.3f} '
        f'{g["s_mid"]:8.3f} '
        f'{g["s_out"]:8.3f} '
        f'{g["sh_in"]:8.3f} '
        f'{g["sh_mid"]:8.3f} '
        f'{g["sh_out"]:8.3f} '
        f'{g["shape_label"]:>18}'
    )

print("\nBEST 12 GALAXIES")
print("================================================================================================================================")
print(f'{"name":<20} {"RMSE":>8} {"Mbar":>12} {"Vf":>8} {"Vmax":>8} {"amp%":>8} {"s_in":>8} {"s_mid":>8} {"s_out":>8} {"sh_in":>8} {"sh_mid":>8} {"sh_out":>8} {"type":>18}')
for g in best:
    print(
        f'{g["name"]:<20} '
        f'{g["rmse"]:8.2f} '
        f'{g["Mbar"]:12.3e} '
        f'{g["Vflat_pred"]:8.1f} '
        f'{g["Vmax_obs"]:8.1f} '
        f'{100*g["amp_frac"]:8.1f} '
        f'{g["s_in"]:8.3f} '
        f'{g["s_mid"]:8.3f} '
        f'{g["s_out"]:8.3f} '
        f'{g["sh_in"]:8.3f} '
        f'{g["sh_mid"]:8.3f} '
        f'{g["sh_out"]:8.3f} '
        f'{g["shape_label"]:>18}'
    )

from collections import Counter
counts = Counter(g["shape_label"] for g in worst)

print("\nFAILURE TYPE COUNTS IN WORST 12")
print("==============================================================")
for k, v in counts.most_common():
    print(f"{k}: {v}")

RUNNING SHAPE-FAILURE CLASSIFIER
A_global = 1.0340397357490716

WORST 12 GALAXIES
name                     RMSE         Mbar       Vf     Vmax     amp%     s_in    s_mid    s_out    sh_in   sh_mid   sh_out               type
NGC5985_rotmod.dat     144.10    1.333e+11    163.0    305.0    -46.6   -0.551   -0.462   -0.434   -0.143    0.006    0.056         early-rise
NGC2841_rotmod.dat     112.74    8.893e+10    248.7    323.0    -23.0   -0.428   -0.178   -0.122   -0.253    0.060    0.123         early-rise
NGC4013_rotmod.dat      93.71    3.395e+10     97.6    198.0    -50.7   -0.637   -0.440   -0.455   -0.259    0.117    0.096         early-rise
UGC11914_rotmod.dat     93.62    5.576e+10    215.1    305.0    -29.5   -0.421   -0.276   -0.251   -0.145    0.025    0.058         early-rise
UGC06787_rotmod.dat     89.34    2.954e+10    196.9    276.0    -28.6   -0.352   -0.197   -0.215   -0.079    0.108    0.090  plateau-amplitude
NGC6195_rotmod.dat      86.59    1.523e+11    191.4    258.0

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# VARIABLE-p OPERATOR SWEEP
#
# Field equation:
#   (1/r^2) d/dr [ r^2 S_scr(r) |chi'|^(p_eff(r)-2) chi' ] - mu2 chi = -rho_eff
#
# with:
#   p_eff(r) = p0 + beta * (vbar^2 / phi_p)^m_p
#
# Frozen backbone from current best branch:
#   source boost on
#   screened p-Laplacian backbone
#   same accumulation + readout
#
# Goal:
#   specifically target early-rise failures in compact/bulge-heavy galaxies
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P0     = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

# Existing mild screening
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# Quality cuts
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_variable_p_operator(r, rho_eff, v_bar2, p0, beta, phi_p, m_p, phi_s, n_scr, mu2,
                              n_iter=180, relax=0.22):
    n = len(r)
    chi = np.zeros(n)

    # Existing screening factor
    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    # Local transport exponent boost
    p_eff = p0 + beta * np.power(np.clip(v_bar2 / phi_p, 0.0, None), m_p)
    p_eff = np.nan_to_num(p_eff, nan=p0, posinf=p0 + beta, neginf=p0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)

        # local p-Laplacian conductivity
        sigma = s_fac * np.power(np.abs(dchi) + GRAD_EPS, p_eff - 2.0)
        sigma = np.nan_to_num(sigma, nan=0.0, posinf=1e30, neginf=0.0)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(beta, phi_p, m_p):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_variable_p_operator(
                r=r,
                rho_eff=rho_eff,
                v_bar2=v_bar2,
                p0=P0,
                beta=beta,
                phi_p=phi_p,
                m_p=m_p,
                phi_s=PHI_S,
                n_scr=N_SCR,
                mu2=MU2
            )
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        y = np.max(row["v_obs"])**2
        x = row["Uinf"] / row["Rs"]
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner, frac_mid, frac_outer = [], [], []
    signed_inner = []

    lowmass_rmse, highmass_rmse = [], []

    for row in rows:
        r, v_obs = row["r"], row["v_obs"]
        Uchi, Uinf, Rs, Mbar = row["Uchi"], row["Uinf"], row["Rs"], row["Mbar"]

        Vflat2 = A_global * (Uinf / Rs)
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        Vflat = np.sqrt(Vflat2)
        y = np.clip(Uchi / Uinf, 0.0, 1.0e30)
        v_pred = Vflat * np.power(y, RESP_EXP)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask   = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
            signed_inner.append(np.median(frac_signed[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(Vflat)
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM  = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU  = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if frac_inner else np.nan,
        "mid": float(np.median(frac_mid)) if frac_mid else np.nan,
        "outer": float(np.median(frac_outer)) if frac_outer else np.nan,
        "signed_inner": float(np.median(signed_inner)) if signed_inner else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if lowmass_rmse else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if highmass_rmse else np.nan,
    }

def run_case(beta, phi_p, m_p):
    rows = build_rows(beta, phi_p, m_p)
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows)
    full = evaluate_rows(rows, A_all)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low  = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low)
    A_high = fit_amplitude(high)

    low_to_high = evaluate_rows(high, A_low)
    high_to_low = evaluate_rows(low, A_high)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("VARIABLE-p OPERATOR SWEEP")
print("==============================================================")

# beta = size of local p boost
# phi_p = field scale where p starts increasing
# m_p   = how sharply p responds
for beta in [0.0, 0.3, 0.6, 1.0]:
    for phi_p in [10000.0, 20000.0, 40000.0]:
        for m_p in [0.5, 1.0]:
            res = run_case(beta, phi_p, m_p)
            print(f"\nbeta={beta:.1f}, phi_p={phi_p:.0f}, m_p={m_p:.1f}")
            print("--------------------------------------------------------------")
            if res is None or res["full"] is None:
                print("insufficient valid galaxies")
                continue

            full = res["full"]
            print("galaxies:", full["n"])
            print("median RMSE:", full["rmse"])
            print("BTFR slope:", full["btfr"])
            print("Rs slope:", full["rs"])
            print("U slope:", full["u"])
            print("median frac residual inner:", full["inner"])
            print("median frac residual mid:", full["mid"])
            print("median frac residual outer:", full["outer"])
            print("median signed inner residual:", full["signed_inner"])
            print("median RMSE low-mass:", full["lowmass_rmse"])
            print("median RMSE high-mass:", full["highmass_rmse"])
            print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
            print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

VARIABLE-p OPERATOR SWEEP

beta=0.0, phi_p=10000, m_p=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.731834368318815
BTFR slope: 1.1672374015336275
Rs slope: 0.31881104458934
U slope: 0.9024297453561537
median frac residual inner: 0.290236286384407
median frac residual mid: 0.20221271171469704
median frac residual outer: 0.21489484690644883
median signed inner residual: -0.1413945667598086
median RMSE low-mass: 17.514391790986974
median RMSE high-mass: 68.9886153106876
Fit LOW -> Test HIGH RMSE: 66.37129707024303
Fit HIGH -> Test LOW RMSE: 17.48299146264566

beta=0.0, phi_p=10000, m_p=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.731834368318815
BTFR slope: 1.1672374015336275
Rs slope: 0.31881104458934
U slope: 0.9024297453561537
median frac residual inner: 0.290236286384407
median frac residual mid: 0.20221271171469704
median frac residual outer: 0.21489484690644883
median signed inner r

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TWO-COMPONENT OBSERVABLE TEST
#
# Frozen backbone:
#   screened p-Laplacian field model
#
# Test observable:
#   V_pred^2(r) = V_field^2(r) + lambda_c * V_central^2(r)
#
# where
#   V_field(r)   = Vflat * (U/Uinf)^RESP_EXP
#   V_central^2  = Vbar^2 * exp[-(r/r_c)^2]
#
# Goal:
#   test whether bad compact/bulge-heavy galaxies need an
#   additional compact central mode rather than more tuning
#   of the same single field.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bar2": v_bar2,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue
    return rows

def fit_amplitude(rows, lambda_c, r_c):
    num = 0.0
    den = 0.0
    for row in rows:
        r = row["r"]
        v_bar2 = row["v_bar2"]

        central2 = v_bar2 * np.exp(- (r / max(r_c, EPS))**2)
        field_shape2 = np.power(np.clip(row["Uchi"] / row["Uinf"], 0.0, None), 2.0 * RESP_EXP)

        # Fit amplitude only to the field piece, central term stays explicit
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2 - lambda_c * np.max(central2)
        if np.isfinite(x) and np.isfinite(y) and x > 0 and y > 0:
            num += x * y
            den += x * x

    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global, lambda_c, r_c):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner, frac_mid, frac_outer = [], [], []
    signed_inner = []

    lowmass_rmse, highmass_rmse = [], []

    for row in rows:
        r, v_obs = row["r"], row["v_obs"]
        Uchi, Uinf, Rs, Mbar = row["Uchi"], row["Uinf"], row["Rs"], row["Mbar"]
        v_bar2 = row["v_bar2"]

        Vflat2 = A_global * (Uinf / Rs)
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        field_shape2 = np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)
        central2 = v_bar2 * np.exp(- (r / max(r_c, EPS))**2)

        v_pred2 = Vflat2 * field_shape2 + lambda_c * central2
        v_pred2 = np.clip(v_pred2, 0.0, None)
        v_pred = np.sqrt(v_pred2)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask   = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
            signed_inner.append(np.median(frac_signed[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(np.sqrt(Vflat2))
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM  = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU  = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if frac_inner else np.nan,
        "mid": float(np.median(frac_mid)) if frac_mid else np.nan,
        "outer": float(np.median(frac_outer)) if frac_outer else np.nan,
        "signed_inner": float(np.median(signed_inner)) if signed_inner else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if lowmass_rmse else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if highmass_rmse else np.nan,
    }

def run_case(lambda_c, r_c):
    rows = build_rows()
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows, lambda_c, r_c)
    full = evaluate_rows(rows, A_all, lambda_c, r_c)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low  = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low, lambda_c, r_c)
    A_high = fit_amplitude(high, lambda_c, r_c)

    low_to_high = evaluate_rows(high, A_low, lambda_c, r_c)
    high_to_low = evaluate_rows(low, A_high, lambda_c, r_c)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("TWO-COMPONENT OBSERVABLE SWEEP")
print("==============================================================")

for lambda_c in [0.0, 0.05, 0.10, 0.20]:
    for r_c in [0.5, 1.0, 2.0, 4.0]:
        res = run_case(lambda_c, r_c)
        print(f"\nlambda_c={lambda_c:.2f}, r_c={r_c:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median signed inner residual:", full["signed_inner"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

TWO-COMPONENT OBSERVABLE SWEEP

lambda_c=0.00, r_c=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median signed inner residual: -0.14158160878800785
median RMSE low-mass: 17.52595483777988
median RMSE high-mass: 69.01015888909751
Fit LOW -> Test HIGH RMSE: 66.36458209704
Fit HIGH -> Test LOW RMSE: 17.490191732455735

lambda_c=0.00, r_c=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median signed inner residual:

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# BULGE-CHANNEL TWO-COMPONENT TEST
#
# Frozen backbone:
#   screened p-Laplacian field model
#
# Test observable:
#   V_pred^2(r) = V_field^2(r) + lambda_b * V_bulge,scaled^2(r) * exp[-(r/r_b)^2]
#
# where
#   V_field(r) = Vflat * (U/Uinf)^RESP_EXP
#   V_bulge,scaled = UPS_BUL * V_bul
#
# Goal:
#   test whether the missing compact branch is specifically
#   bulge-linked rather than just central-baryon linked.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None
        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)
            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul2": np.maximum(v_bul_s**2, 0.0),
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue
    return rows

def fit_amplitude(rows, lambda_b, r_b):
    num = 0.0
    den = 0.0
    for row in rows:
        r = row["r"]
        v_bul2 = row["v_bul2"]

        bulge2 = v_bul2 * np.exp(- (r / max(r_b, EPS))**2)
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2 - lambda_b * np.max(bulge2)

        if np.isfinite(x) and np.isfinite(y) and x > 0 and y > 0:
            num += x * y
            den += x * x

    return num / den if den > 0 else np.nan

def evaluate_rows(rows, A_global, lambda_b, r_b):
    rmse_list = []
    Vflat_list = []
    M_list = []
    Rs_list = []
    U_list = []

    frac_inner, frac_mid, frac_outer = [], [], []
    signed_inner = []

    lowmass_rmse, highmass_rmse = [], []

    for row in rows:
        r, v_obs = row["r"], row["v_obs"]
        Uchi, Uinf, Rs, Mbar = row["Uchi"], row["Uinf"], row["Rs"], row["Mbar"]
        v_bul2 = row["v_bul2"]

        Vflat2 = A_global * (Uinf / Rs)
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        field_shape2 = np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)
        bulge2 = v_bul2 * np.exp(- (r / max(r_b, EPS))**2)

        v_pred2 = Vflat2 * field_shape2 + lambda_b * bulge2
        v_pred2 = np.clip(v_pred2, 0.0, None)
        v_pred = np.sqrt(v_pred2)

        if not np.all(np.isfinite(v_pred)):
            continue

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        x = r / Rs
        frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        inner_mask = x < 0.7
        mid_mask   = (x >= 0.7) & (x <= 2.0)
        outer_mask = x > 2.0

        if np.any(inner_mask):
            frac_inner.append(np.median(frac[inner_mask]))
            signed_inner.append(np.median(frac_signed[inner_mask]))
        if np.any(mid_mask):
            frac_mid.append(np.median(frac[mid_mask]))
        if np.any(outer_mask):
            frac_outer.append(np.median(frac[outer_mask]))

        if Mbar < 1e10:
            lowmass_rmse.append(rmse)
        else:
            highmass_rmse.append(rmse)

        rmse_list.append(rmse)
        Vflat_list.append(np.sqrt(Vflat2))
        M_list.append(Mbar)
        Rs_list.append(Rs)
        U_list.append(Uinf)

    if len(rmse_list) < 20:
        return None

    rmse_arr = np.array(rmse_list)
    V_arr = np.array(Vflat_list)
    M_arr = np.array(M_list)
    Rs_arr = np.array(Rs_list)
    U_arr = np.array(U_list)

    logM  = np.log10(M_arr)
    logV4 = np.log10(V_arr**4)
    logRs = np.log10(Rs_arr)
    logU  = np.log10(U_arr)

    return {
        "n": len(rmse_arr),
        "rmse": float(np.median(rmse_arr)),
        "btfr": float(np.polyfit(logM, logV4, 1)[0]),
        "rs": float(np.polyfit(logM, logRs, 1)[0]),
        "u": float(np.polyfit(logM, logU, 1)[0]),
        "inner": float(np.median(frac_inner)) if frac_inner else np.nan,
        "mid": float(np.median(frac_mid)) if frac_mid else np.nan,
        "outer": float(np.median(frac_outer)) if frac_outer else np.nan,
        "signed_inner": float(np.median(signed_inner)) if signed_inner else np.nan,
        "lowmass_rmse": float(np.median(lowmass_rmse)) if lowmass_rmse else np.nan,
        "highmass_rmse": float(np.median(highmass_rmse)) if highmass_rmse else np.nan,
    }

def run_case(lambda_b, r_b):
    rows = build_rows()
    if len(rows) < 20:
        return None

    A_all = fit_amplitude(rows, lambda_b, r_b)
    full = evaluate_rows(rows, A_all, lambda_b, r_b)

    M_all = np.array([row["Mbar"] for row in rows])
    mass_split = np.median(M_all)
    low  = [row for row in rows if row["Mbar"] < mass_split]
    high = [row for row in rows if row["Mbar"] >= mass_split]

    A_low = fit_amplitude(low, lambda_b, r_b)
    A_high = fit_amplitude(high, lambda_b, r_b)

    low_to_high = evaluate_rows(high, A_low, lambda_b, r_b)
    high_to_low = evaluate_rows(low, A_high, lambda_b, r_b)

    return {
        "full": full,
        "low_to_high_rmse": None if low_to_high is None else low_to_high["rmse"],
        "high_to_low_rmse": None if high_to_low is None else high_to_low["rmse"],
    }

print("BULGE-CHANNEL TWO-COMPONENT SWEEP")
print("==============================================================")

for lambda_b in [0.0, 0.1, 0.2, 0.4, 0.8]:
    for r_b in [0.5, 1.0, 2.0, 4.0]:
        res = run_case(lambda_b, r_b)
        print(f"\nlambda_b={lambda_b:.2f}, r_b={r_b:.1f}")
        print("--------------------------------------------------------------")
        if res is None or res["full"] is None:
            print("insufficient valid galaxies")
            continue

        full = res["full"]
        print("galaxies:", full["n"])
        print("median RMSE:", full["rmse"])
        print("BTFR slope:", full["btfr"])
        print("Rs slope:", full["rs"])
        print("U slope:", full["u"])
        print("median frac residual inner:", full["inner"])
        print("median frac residual mid:", full["mid"])
        print("median frac residual outer:", full["outer"])
        print("median signed inner residual:", full["signed_inner"])
        print("median RMSE low-mass:", full["lowmass_rmse"])
        print("median RMSE high-mass:", full["highmass_rmse"])
        print("Fit LOW -> Test HIGH RMSE:", res["low_to_high_rmse"])
        print("Fit HIGH -> Test LOW RMSE:", res["high_to_low_rmse"])

BULGE-CHANNEL TWO-COMPONENT SWEEP

lambda_b=0.00, r_b=0.5
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median signed inner residual: -0.14158160878800785
median RMSE low-mass: 17.52595483777988
median RMSE high-mass: 69.01015888909751
Fit LOW -> Test HIGH RMSE: 66.36458209704
Fit HIGH -> Test LOW RMSE: 17.490191732455735

lambda_b=0.00, r_b=1.0
--------------------------------------------------------------
galaxies: 60
median RMSE: 27.723895695091663
BTFR slope: 1.1662264202811767
Rs slope: 0.31881104458934
U slope: 0.9019242547299284
median frac residual inner: 0.29017237515901806
median frac residual mid: 0.20238391592430593
median frac residual outer: 0.21506359293466007
median signed inner residu

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# REGIME-CHANGE (YIELD-LIKE) OPERATOR TEST
#
# p_eff switches from p1 -> p2 when local baryonic field g_b
# exceeds threshold g_c (smoothed logistic transition).
#
# This is the "material changes regime when overloaded" test.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Backbone constants --------
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

# ============================================================

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# ============================================================
# Regime-switch screened p-Laplacian
# ============================================================

def solve_regime_operator(r, rho_eff, g_b, v_bar2, p1, p2, g_c, dg,
                          phi_s, n_scr, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))

    # logistic regime switch
    switch = 1.0 / (1.0 + np.exp(-(g_b - g_c) / max(dg, EPS)))
    p_eff = p1 + (p2 - p1) * switch

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_eff - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except:
            return None

        chi = (1 - relax) * chi + relax * chi_new

    return chi

# ============================================================
# Build galaxies
# ============================================================

def build_rows(p1, p2, g_c, dg):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            rho_eff = rho * force_factor

            chi = solve_regime_operator(
                r, rho_eff, g_b, v_bar2,
                p1=p1, p2=p2, g_c=g_c, dg=dg,
                phi_s=PHI_S, n_scr=N_SCR, mu2=MU2
            )
            if chi is None:
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]

            rows.append({
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs
            })
        except:
            continue
    return rows

# ============================================================
# Evaluation
# ============================================================

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den

def evaluate(rows, A):
    rmse_list = []
    signed_inner = []
    highmass_rmse = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A * (Uinf / Rs)
        y = np.clip(Uchi / Uinf, 0, None)
        v_pred = np.sqrt(Vflat2) * y**RESP_EXP

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        rmse_list.append(rmse)

        inner_mask = (r / Rs) < 0.7
        if np.any(inner_mask):
            frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)
            signed_inner.append(np.median(frac_signed[inner_mask]))

        if Mbar > 1e10:
            highmass_rmse.append(rmse)

    return {
        "rmse": float(np.median(rmse_list)),
        "signed_inner": float(np.median(signed_inner)),
        "highmass_rmse": float(np.median(highmass_rmse)),
    }

# ============================================================
# Sweep
# ============================================================

print("REGIME-CHANGE OPERATOR SWEEP")
print("==============================================================")

for p2 in [3.5, 4.0, 4.5, 5.0]:
    for g_c in [2000, 4000, 8000]:
        rows = build_rows(p1=3.5, p2=p2, g_c=g_c, dg=1000)
        if len(rows) < 20:
            continue

        A = fit_amplitude(rows)
        res = evaluate(rows, A)

        print(f"\np2={p2:.1f}, g_c={g_c}")
        print("--------------------------------------------------------------")
        print("median RMSE:", res["rmse"])
        print("median signed inner residual:", res["signed_inner"])
        print("median RMSE high-mass:", res["highmass_rmse"])

REGIME-CHANGE OPERATOR SWEEP

p2=3.5, g_c=2000
--------------------------------------------------------------
median RMSE: 27.723895695091663
median signed inner residual: -0.14158160878800785
median RMSE high-mass: 69.01015888909751

p2=3.5, g_c=4000
--------------------------------------------------------------
median RMSE: 27.723895695091663
median signed inner residual: -0.14158160878800785
median RMSE high-mass: 69.01015888909751

p2=3.5, g_c=8000
--------------------------------------------------------------
median RMSE: 27.723895695091663
median signed inner residual: -0.14158160878800785
median RMSE high-mass: 69.01015888909751

p2=4.0, g_c=2000
--------------------------------------------------------------
median RMSE: 124.4842934825889
median signed inner residual: -0.9923826892270241
median RMSE high-mass: 210.50742568317676

p2=4.0, g_c=4000
--------------------------------------------------------------
median RMSE: 30.826424123928142
median signed inner residual: -0.165106

In [ ]:
import numpy as np

# ============================================================
# NEAR-CRITICAL p-LAPLACIAN SCALING DIAGNOSTIC
#
# Goal:
#   Study the transport equation
#
#     div(|grad chi|^(p-2) grad chi) = rho
#
#   near the critical value p = 4, and extract:
#
#   1) outer-field exponent
#   2) accumulated-field exponent
#   3) whether the accumulation becomes logarithmic
#   4) effective velocity scaling from V^2 ~ U / R
#
# This is NOT a galaxy-fit cell.
# It is an operator-scaling cell.
# ============================================================

# ------------------------------------------------------------
# Assumptions used
# ------------------------------------------------------------
#
# In the outer region, assume the effective source scales so that:
#
#   g_b * g_obs ~ r^{-3}
#
# and the transported field behaves like:
#
#   m(r) ~ r^{-a(p)}
#
# with the p-Laplacian critical relation:
#
#   a(p) = 3 / (p - 1)
#
# Then accumulated response is
#
#   U(r) ~ ∫ r^{-a(p)} dr
#
# so:
#   - if a = 1  -> logarithmic accumulation
#   - if a < 1  -> power-law growth
#   - if a > 1  -> saturating / too stiff
#
# The critical point is therefore:
#
#   a = 1  ->  3/(p-1) = 1  ->  p = 4
#
# ------------------------------------------------------------

def outer_exponent_a(p):
    return 3.0 / (p - 1.0)

def accumulation_type(p, tol=1e-9):
    a = outer_exponent_a(p)
    if abs(a - 1.0) < tol:
        return "log"
    elif a < 1.0:
        return "power_growth"
    else:
        return "stiff/saturating"

def accumulation_power_mu(p):
    """
    If U(r) ~ r^mu for a != 1, then
      mu = 1 - a = 1 - 3/(p-1)
    At p = 4 this is replaced by logarithm.
    """
    a = outer_exponent_a(p)
    if abs(a - 1.0) < 1e-9:
        return np.nan
    return 1.0 - a

def velocity_mass_exponent_from_U_R(alpha_U, alpha_R):
    """
    If:
      U_inf ~ M^alpha_U
      R_s   ~ M^alpha_R
      V^2 ~ U_inf / R_s
    then:
      V^2 ~ M^(alpha_U - alpha_R)
      V^4 ~ M^(2 alpha_U - 2 alpha_R)
    """
    alpha_V2 = alpha_U - alpha_R
    alpha_V4 = 2.0 * alpha_V2
    return alpha_V2, alpha_V4

# ------------------------------------------------------------
# Scan p near the critical region
# ------------------------------------------------------------
p_list = [2.0, 2.5, 3.0, 3.3, 3.5, 3.7, 3.8, 3.9, 3.95, 4.0, 4.05, 4.1, 4.5, 5.0]

print("NEAR-CRITICAL p-LAPLACIAN OPERATOR SCALING")
print("="*70)
print(f'{"p":>6} {"a=3/(p-1)":>14} {"U-type":>18} {"mu=1-a":>14}')
print("-"*70)

for p in p_list:
    a = outer_exponent_a(p)
    typ = accumulation_type(p)
    mu = accumulation_power_mu(p)
    mu_str = "log" if np.isnan(mu) else f"{mu:.6f}"
    print(f"{p:6.2f} {a:14.6f} {typ:>18} {mu_str:>14}")

print("\n")
print("CRITICAL INTERPRETATION")
print("="*70)
for p in [3.5, 3.8, 3.9, 4.0, 4.1]:
    a = outer_exponent_a(p)
    typ = accumulation_type(p)
    print(f"p={p:.2f} -> outer field exponent a={a:.6f} -> accumulation regime: {typ}")

# ------------------------------------------------------------
# Sensitivity around p = 4
# ------------------------------------------------------------
print("\n")
print("SENSITIVITY AROUND p = 4")
print("="*70)
for eps in [1e-1, 5e-2, 1e-2, 1e-3]:
    p_lo = 4.0 - eps
    p_hi = 4.0 + eps
    a_lo = outer_exponent_a(p_lo)
    a_hi = outer_exponent_a(p_hi)
    mu_lo = accumulation_power_mu(p_lo)
    mu_hi = accumulation_power_mu(p_hi)
    print(f"eps={eps:.3e}")
    print(f"  p=4-{eps:.3e}: a={a_lo:.9f}, mu={mu_lo:.9f}  (slightly diffusive below critical)")
    print(f"  p=4+{eps:.3e}: a={a_hi:.9f}, mu={mu_hi:.9f}  (slightly stiff above critical)")
    print()

# ------------------------------------------------------------
# Simple BTFR consistency toy check
# ------------------------------------------------------------
#
# This section does not derive alpha_U and alpha_R from first principles.
# It just lets you plug values in and see what BTFR slope follows.
#
# For example, if the data suggest:
#   R_s ~ M^0.32
# and if a near-critical operator implied
#   U_inf ~ M^0.82
# then:
#   V^4 ~ M^(2*(0.82 - 0.32)) = M^1.00
#
print("\n")
print("TOY BTFR CONSISTENCY CHECK")
print("="*70)

examples = [
    ("baseline-like", 0.90, 0.32),
    ("near-BTFR",     0.82, 0.32),
    ("too-steep U",   0.95, 0.32),
    ("too-weak U",    0.70, 0.32),
]

print(f'{"case":<16} {"alpha_U":>10} {"alpha_R":>10} {"alpha_V2":>10} {"BTFR slope on V^4":>20}')
print("-"*70)
for label, aU, aR in examples:
    aV2, aV4 = velocity_mass_exponent_from_U_R(aU, aR)
    print(f"{label:<16} {aU:10.4f} {aR:10.4f} {aV2:10.4f} {aV4:20.4f}")

# ------------------------------------------------------------
# Main takeaway printout
# ------------------------------------------------------------
print("\n")
print("MAIN TAKEAWAYS")
print("="*70)
print("1. The p-Laplacian critical transport index for logarithmic accumulation is p = 4.")
print("2. For p < 4, accumulation grows as a power law (too diffusive if too far below 4).")
print("3. For p > 4, accumulation stiffens/localises (too weak if too far above 4).")
print("4. Therefore a working galaxy model should sit slightly below p = 4, or regulate near it.")
print("5. This makes p = 4 look like a marginal / critical transport point, not a random fit value.")

NEAR-CRITICAL p-LAPLACIAN OPERATOR SCALING
     p      a=3/(p-1)             U-type         mu=1-a
----------------------------------------------------------------------
  2.00       3.000000   stiff/saturating      -2.000000
  2.50       2.000000   stiff/saturating      -1.000000
  3.00       1.500000   stiff/saturating      -0.500000
  3.30       1.304348   stiff/saturating      -0.304348
  3.50       1.200000   stiff/saturating      -0.200000
  3.70       1.111111   stiff/saturating      -0.111111
  3.80       1.071429   stiff/saturating      -0.071429
  3.90       1.034483   stiff/saturating      -0.034483
  3.95       1.016949   stiff/saturating      -0.016949
  4.00       1.000000                log            log
  4.05       0.983607       power_growth       0.016393
  4.10       0.967742       power_growth       0.032258
  4.50       0.857143       power_growth       0.142857
  5.00       0.750000       power_growth       0.250000


CRITICAL INTERPRETATION
p=3.50 -> outer fiel

In [ ]:
import numpy as np

# ============================================================
# SMALL-epsilon ASYMPTOTIC DIAGNOSTIC AROUND p = 4 - epsilon
#
# Operator:
#   div(|grad chi|^(p-2) grad chi) = rho_eff
#
# Near-critical parameterisation:
#   p = 4 - eps
#
# Outer scaling assumption used in previous diagnostics:
#   m(r) ~ r^{-a(p)},   a(p) = 3 / (p - 1)
#
# We now expand around eps = 0, i.e. p = 4 - eps.
# ============================================================

def p_from_eps(eps):
    return 4.0 - eps

def a_of_eps(eps):
    # a = 3/(p-1) = 3/(3-eps)
    return 3.0 / (3.0 - eps)

def a_series(eps):
    # 1/(1-x) = 1 + x + x^2 + ...
    # a = 1/(1 - eps/3)
    return 1.0 + eps/3.0 + eps**2/9.0 + eps**3/27.0

def mu_of_eps(eps):
    # U ~ r^mu when not exactly logarithmic, with mu = 1 - a
    return 1.0 - a_of_eps(eps)

def mu_series(eps):
    return -(eps/3.0) - eps**2/9.0 - eps**3/27.0

def exact_U_ratio(x, eps):
    """
    For m(r) ~ r^{-a}, U(x) ~ (x^mu - 1)/mu with mu = 1-a.
    The log limit as eps->0 is ln x.
    """
    mu = mu_of_eps(eps)
    if abs(mu) < 1e-12:
        return np.log(x)
    return (x**mu - 1.0) / mu

def asymptotic_U_ratio(x, eps):
    """
    Expand:
      x^mu = exp(mu ln x) = 1 + mu L + mu^2 L^2/2 + mu^3 L^3/6 + ...
      (x^mu - 1)/mu = L + mu L^2/2 + mu^2 L^3/6 + ...
    with L = ln x
    """
    L = np.log(x)
    mu = mu_series(eps)
    return L + 0.5*mu*L**2 + (mu**2)*(L**3)/6.0

def log_stretched_match_nu(eps, x0=1.0):
    """
    Heuristic local matching:
      U(x) = log-like corrected quantity
      y = 1 - exp[-U/U0]
    Compare to:
      y ~ 1 - exp[-(x/x0)^nu]
    For x near x0=1, d ln U / d ln x acts like an effective local nu.
    Since dU/dlnx ~ x^mu, near x=1 this is close to 1, but curvature correction
    enters through second derivative. We print the first correction structure.
    """
    mu = mu_of_eps(eps)
    # local exponent at x=1 is still 1; deviation shows up in curvature.
    # We report the leading curvature coefficient in ln x expansion:
    # U = L + (mu/2)L^2 + ...
    return mu/2.0

print("SMALL-epsilon NEAR-CRITICAL ASYMPTOTICS")
print("="*78)
print("Parameterisation:  p = 4 - eps")
print()

print("1) OUTER FIELD EXPONENT")
print("-"*78)
print("Exact:")
print("  a(eps) = 3 / (3 - eps)")
print("Series:")
print("  a(eps) = 1 + eps/3 + eps^2/9 + eps^3/27 + O(eps^4)")
print()

print("2) ACCUMULATION EXPONENT")
print("-"*78)
print("If U ~ r^mu away from exact criticality, then")
print("  mu(eps) = 1 - a(eps)")
print("Exact:")
print("  mu(eps) = 1 - 3/(3 - eps)")
print("Series:")
print("  mu(eps) = -eps/3 - eps^2/9 - eps^3/27 + O(eps^4)")
print()

print("3) CONSEQUENCE")
print("-"*78)
print("Since mu(eps) < 0 for eps > 0, the near-critical law is")
print("  slightly below logarithmic growth:")
print("    U(x) = (x^mu - 1)/mu")
print("with exact log recovered only at eps = 0.")
print()

eps_list = [0.00, 0.03, 0.06, 0.10, 0.20, 0.30, 0.50]
print("NUMERICAL TABLE")
print("-"*78)
print(f'{"eps":>8} {"p=4-eps":>12} {"a(eps)":>12} {"mu(eps)":>12}')
for eps in eps_list:
    print(f'{eps:8.3f} {p_from_eps(eps):12.6f} {a_of_eps(eps):12.6f} {mu_of_eps(eps):12.6f}')

print()
print("4) LOG-LAW CORRECTION")
print("-"*78)
print("Let x = r/r0 and L = ln x.")
print("Then")
print("  U(x) = (x^mu - 1)/mu")
print("       = L + (mu/2)L^2 + (mu^2/6)L^3 + ...")
print()
print("Substitute mu(eps) = -eps/3 + O(eps^2):")
print("  U(x) = L - (eps/6)L^2 + O(eps^2 L^2, eps L^3)")
print()
print("So the first near-critical correction is:")
print("  exact log  ->  log minus a quadratic-in-log suppression")
print()
print("This is the cleanest asymptotic statement in the whole branch.")
print()

print("5) SAMPLE EXACT vs ASYMPTOTIC U(x)")
print("-"*78)
x_list = [1.5, 2.0, 3.0, 5.0, 10.0]
for eps in [0.03, 0.06, 0.10, 0.20]:
    print(f"\neps = {eps:.3f}   (p = {4-eps:.3f})")
    print(f'{"x":>8} {"U_exact":>14} {"U_asympt":>14} {"ln x":>14}')
    for x in x_list:
        ue = exact_U_ratio(x, eps)
        ua = asymptotic_U_ratio(x, eps)
        lx = np.log(x)
        print(f'{x:8.3f} {ue:14.6f} {ua:14.6f} {lx:14.6f}')

print()
print("6) IMPLICATION FOR RESPONSE LAW")
print("-"*78)
print("If")
print("  y(x) = 1 - exp[-U(x)/U0],")
print("and near criticality")
print("  U(x) = ln x - (eps/6)(ln x)^2 + ... ,")
print("then the exact marginal law is no longer a pure logarithm once eps>0.")
print()
print("So near-critical transport naturally produces:")
print("  - slower-than-log accumulation")
print("  - slightly suppressed outer build-up")
print("  - BTFR slope above 1 if U_inf(M) grows too fast")
print()

print("7) HEURISTIC LINK TO YOUR FITTED BTFR DRIFT")
print("-"*78)
print("Your toy BTFR check said:")
print("  alpha_U ~ 0.90 with alpha_R ~ 0.32  -> V^4 slope ~ 1.16")
print("while")
print("  alpha_U ~ 0.82 with alpha_R ~ 0.32  -> V^4 slope ~ 1.00")
print()
print("So the remaining first-principles task is:")
print("  explain how near-critical transport changes alpha_U away from the")
print("  exact logarithmic limit, and why bulge-heavy systems sit closest")
print("  to the critical edge.")
print()

print("8) LOCAL CURVATURE COEFFICIENT")
print("-"*78)
print("From")
print("  U = L + (mu/2)L^2 + ...")
print("the leading curvature coefficient is mu/2.")
print("Numerically:")
print(f'{"eps":>8} {"mu/2":>14}')
for eps in [0.03, 0.06, 0.10, 0.20]:
    print(f'{eps:8.3f} {0.5*mu_of_eps(eps):14.6f}')

print()
print("MAIN RESULT")
print("="*78)
print("For p = 4 - eps with eps > 0 small:")
print("  a(eps) = 1 + eps/3 + O(eps^2)")
print("  mu(eps) = -eps/3 + O(eps^2)")
print("  U(x) = ln x - (eps/6)(ln x)^2 + ...")
print()
print("So the critical operator gives an exact logarithm at p = 4,")
print("and the observed near-critical branch is the slightly subcritical")
print("deformation obtained by eps > 0.")

SMALL-epsilon NEAR-CRITICAL ASYMPTOTICS
Parameterisation:  p = 4 - eps

1) OUTER FIELD EXPONENT
------------------------------------------------------------------------------
Exact:
  a(eps) = 3 / (3 - eps)
Series:
  a(eps) = 1 + eps/3 + eps^2/9 + eps^3/27 + O(eps^4)

2) ACCUMULATION EXPONENT
------------------------------------------------------------------------------
If U ~ r^mu away from exact criticality, then
  mu(eps) = 1 - a(eps)
Exact:
  mu(eps) = 1 - 3/(3 - eps)
Series:
  mu(eps) = -eps/3 - eps^2/9 - eps^3/27 + O(eps^4)

3) CONSEQUENCE
------------------------------------------------------------------------------
Since mu(eps) < 0 for eps > 0, the near-critical law is
  slightly below logarithmic growth:
    U(x) = (x^mu - 1)/mu
with exact log recovered only at eps = 0.

NUMERICAL TABLE
------------------------------------------------------------------------------
     eps      p=4-eps       a(eps)      mu(eps)
   0.000     4.000000     1.000000     0.000000
   0.030     3.97

In [ ]:
import numpy as np

# ============================================================
# NEAR-CRITICAL ANALYTIC RESPONSE vs EMPIRICAL MASTERCURVE
#
# Compare:
#   y_emp(x) = 1 - exp[-(x/x0)^nu]
#
# against near-critical analytic transport:
#   U(x) = ln x - (eps/6)(ln x)^2
#   y_nc(x) = 1 - exp[-U(x)/U(xs)]
#
# We sweep eps and xs and report RMS difference over x-grid.
# ============================================================

# empirical mastercurve parameters
x0_emp = 0.966
nu_emp = 0.942

def y_emp(x, x0=x0_emp, nu=nu_emp):
    return 1.0 - np.exp(- (x / x0)**nu)

def U_nc(x, eps):
    L = np.log(x)
    return L - (eps/6.0) * L**2

def y_nc(x, eps, xs):
    U = U_nc(x, eps)
    Us = U_nc(xs, eps)
    if not np.isfinite(Us) or abs(Us) < 1e-12:
        return np.full_like(x, np.nan)
    z = U / Us
    return 1.0 - np.exp(-z)

# x-grid: avoid x<=0 because of logs
x_grid = np.linspace(0.25, 6.0, 500)
y_target = y_emp(x_grid)

best = []

print("NEAR-CRITICAL ANALYTIC RESPONSE vs EMPIRICAL MASTERCURVE")
print("="*78)

for eps in [0.00, 0.03, 0.06, 0.10, 0.15, 0.20, 0.30]:
    for xs in [1.0, 1.2, 1.5, 2.0, 3.0]:
        y_model = y_nc(x_grid, eps, xs)
        if np.any(~np.isfinite(y_model)):
            continue

        rms = np.sqrt(np.mean((y_model - y_target)**2))
        mae = np.mean(np.abs(y_model - y_target))

        # compare around inner / mid / outer regions
        inner = x_grid < 0.8
        mid   = (x_grid >= 0.8) & (x_grid <= 2.0)
        outer = x_grid > 2.0

        rms_in = np.sqrt(np.mean((y_model[inner] - y_target[inner])**2))
        rms_mid = np.sqrt(np.mean((y_model[mid] - y_target[mid])**2))
        rms_out = np.sqrt(np.mean((y_model[outer] - y_target[outer])**2))

        best.append((rms, mae, eps, xs, rms_in, rms_mid, rms_out))

best.sort(key=lambda t: t[0])

print(f'{"rank":>4} {"eps":>8} {"xs":>8} {"RMS":>12} {"MAE":>12} {"RMS_in":>12} {"RMS_mid":>12} {"RMS_out":>12}')
print("-"*78)
for i, row in enumerate(best[:15], start=1):
    rms, mae, eps, xs, rms_in, rms_mid, rms_out = row
    print(f"{i:4d} {eps:8.3f} {xs:8.3f} {rms:12.6f} {mae:12.6f} {rms_in:12.6f} {rms_mid:12.6f} {rms_out:12.6f}")

print("\nEMPIRICAL PARAMETERS")
print("="*78)
print("x0_emp =", x0_emp)
print("nu_emp =", nu_emp)

if best:
    rms, mae, eps, xs, rms_in, rms_mid, rms_out = best[0]
    print("\nBEST NEAR-CRITICAL MATCH")
    print("="*78)
    print("eps   =", eps)
    print("xs    =", xs)
    print("RMS   =", rms)
    print("MAE   =", mae)
    print("RMS_in  =", rms_in)
    print("RMS_mid =", rms_mid)
    print("RMS_out =", rms_out)

    print("\nINTERPRETATION")
    print("="*78)
    print("If a small eps gives a good match, then the empirical mastercurve")
    print("can be interpreted as the near-critical deformation of the exact")
    print("logarithmic transport law at p = 4.")

NEAR-CRITICAL ANALYTIC RESPONSE vs EMPIRICAL MASTERCURVE
rank      eps       xs          RMS          MAE       RMS_in      RMS_mid      RMS_out
------------------------------------------------------------------------------
   1    0.000    3.000     0.575385     0.432769     1.507621     0.535649     0.276441
   2    0.030    3.000     0.580253     0.434408     1.526818     0.535284     0.276738
   3    0.060    3.000     0.585262     0.436082     1.546490     0.534917     0.277041
   4    0.100    3.000     0.592172     0.438369     1.573486     0.534424     0.277455
   5    0.150    3.000     0.601197     0.441321     1.608520     0.533802     0.277989
   6    0.200    3.000     0.610678     0.444382     1.645064     0.533172     0.278543
   7    0.300    3.000     0.631119     0.450850     1.723043     0.531893     0.279710
   8    0.000    2.000     0.943394     0.419807     2.941672     0.477615     0.130283
   9    0.030    2.000     0.957878     0.423549     2.989812     0.4776

In [ ]:
import numpy as np

# ============================================================
# REGULARIZED NEAR-CRITICAL RESPONSE vs EMPIRICAL MASTERCURVE
#
# Test:
#   U(x) = log(1 + x/xc) - (eps/6) * log^2(1 + x/xc)
#   y(x) = 1 - exp[-U(x)/U(xs)]
#
# against empirical:
#   y_emp(x) = 1 - exp[-(x/x0)^nu]
# ============================================================

x0_emp = 0.966
nu_emp = 0.942

def y_emp(x, x0=x0_emp, nu=nu_emp):
    return 1.0 - np.exp(- (x / x0)**nu)

def U_reg_nc(x, eps, xc):
    L = np.log(1.0 + x / xc)
    return L - (eps / 6.0) * L**2

def y_reg_nc(x, eps, xc, xs):
    U = U_reg_nc(x, eps, xc)
    Us = U_reg_nc(xs, eps, xc)
    if not np.isfinite(Us) or abs(Us) < 1e-12:
        return np.full_like(x, np.nan)
    return 1.0 - np.exp(-U / Us)

x_grid = np.linspace(0.0, 6.0, 600)
y_target = y_emp(x_grid)

results = []

for eps in [0.00, 0.03, 0.06, 0.10, 0.15, 0.20]:
    for xc in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
        for xs in [1.0, 1.5, 2.0, 3.0, 4.0]:
            y_model = y_reg_nc(x_grid, eps, xc, xs)
            if np.any(~np.isfinite(y_model)):
                continue

            rms = np.sqrt(np.mean((y_model - y_target)**2))
            mae = np.mean(np.abs(y_model - y_target))

            inner = x_grid < 0.8
            mid   = (x_grid >= 0.8) & (x_grid <= 2.0)
            outer = x_grid > 2.0

            rms_in = np.sqrt(np.mean((y_model[inner] - y_target[inner])**2))
            rms_mid = np.sqrt(np.mean((y_model[mid] - y_target[mid])**2))
            rms_out = np.sqrt(np.mean((y_model[outer] - y_target[outer])**2))

            results.append((rms, mae, eps, xc, xs, rms_in, rms_mid, rms_out))

results.sort(key=lambda t: t[0])

print("REGULARIZED NEAR-CRITICAL RESPONSE vs EMPIRICAL MASTERCURVE")
print("="*90)
print(f'{"rank":>4} {"eps":>8} {"xc":>8} {"xs":>8} {"RMS":>12} {"MAE":>12} {"RMS_in":>12} {"RMS_mid":>12} {"RMS_out":>12}')
print("-"*90)
for i, row in enumerate(results[:20], start=1):
    rms, mae, eps, xc, xs, rms_in, rms_mid, rms_out = row
    print(f"{i:4d} {eps:8.3f} {xc:8.3f} {xs:8.3f} {rms:12.6f} {mae:12.6f} {rms_in:12.6f} {rms_mid:12.6f} {rms_out:12.6f}")

if results:
    rms, mae, eps, xc, xs, rms_in, rms_mid, rms_out = results[0]
    print("\nBEST MATCH")
    print("="*90)
    print("eps    =", eps)
    print("xc     =", xc)
    print("xs     =", xs)
    print("RMS    =", rms)
    print("MAE    =", mae)
    print("RMS_in =", rms_in)
    print("RMS_mid=", rms_mid)
    print("RMS_out=", rms_out)

REGULARIZED NEAR-CRITICAL RESPONSE vs EMPIRICAL MASTERCURVE
rank      eps       xc       xs          RMS          MAE       RMS_in      RMS_mid      RMS_out
------------------------------------------------------------------------------------------
   1    0.000    1.000    1.000     0.062791     0.058444     0.023549     0.042942     0.072456
   2    0.030    1.000    1.000     0.063623     0.059224     0.023975     0.043320     0.073443
   3    0.060    1.000    1.000     0.064469     0.060017     0.024405     0.043702     0.074447
   4    0.100    1.000    1.000     0.065621     0.061095     0.024981     0.044216     0.075815
   5    0.150    1.000    1.000     0.067100     0.062478     0.025709     0.044869     0.077573
   6    0.200    1.000    1.000     0.068624     0.063900     0.026445     0.045532     0.079386
   7    0.000    0.800    1.000     0.072344     0.067580     0.030782     0.047904     0.083501
   8    0.030    0.800    1.000     0.073296     0.068472     0.031279   

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TEST: DOES THE SOLVED FIELD PRODUCE
#
#   U(r) ~ A * log(1 + r / R_log) ?
#
# Frozen backbone:
#   screened p-Laplacian field model
#
# For each galaxy:
#   1) solve chi(r)
#   2) build U(r) = ∫ chi(r) r^kappa dr
#   3) fit U(r) ~ A log(1 + r/R_log)
#   4) report fit RMSE and R_log vs existing Rs
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def fit_log1p_model(r, U, Rs_ref):
    """
    Fit U(r) ~ A * log(1 + r/Rlog)
    by grid search over Rlog and linear solve for A at each trial.
    """
    mask = np.isfinite(r) & np.isfinite(U) & (r > 0)
    r = r[mask]
    U = U[mask]
    if len(r) < 8:
        return None

    rmax = np.max(r)
    # search around Rs_ref but broad enough to be informative
    lo = max(0.05, 0.2 * Rs_ref)
    hi = max(lo * 1.1, 5.0 * Rs_ref, 0.5 * rmax)
    R_grid = np.geomspace(lo, hi, 120)

    best = None
    for Rlog in R_grid:
        basis = np.log1p(r / Rlog)
        denom = np.dot(basis, basis)
        if denom <= 0:
            continue
        A = np.dot(U, basis) / denom
        U_fit = A * basis
        rmse = np.sqrt(np.mean((U_fit - U)**2))
        mae = np.mean(np.abs(U_fit - U))

        # normalized RMSE relative to U range
        Urng = max(np.max(U) - np.min(U), EPS)
        nrmse = rmse / Urng

        if best is None or rmse < best["rmse"]:
            best = {
                "A": A,
                "Rlog": Rlog,
                "rmse": rmse,
                "mae": mae,
                "nrmse": nrmse,
                "U_fit": U_fit,
            }
    return best

print("TESTING WHETHER U(r) ~ A log(1 + r/Rlog)")
print("="*78)

rows = []
cut_stats = {}

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        cut_stats["read_fail"] = cut_stats.get("read_fail", 0) + 1
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        cut_stats["quality_fail"] = cut_stats.get("quality_fail", 0) + 1
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            cut_stats["solver_fail"] = cut_stats.get("solver_fail", 0) + 1
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            cut_stats["bad_uinf"] = cut_stats.get("bad_uinf", 0) + 1
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            cut_stats["bad_Rs"] = cut_stats.get("bad_Rs", 0) + 1
            continue

        fit = fit_log1p_model(r, Uchi, Rs)
        if fit is None:
            cut_stats["fit_fail"] = cut_stats.get("fit_fail", 0) + 1
            continue

        rows.append({
            "name": os.path.basename(path),
            "Mbar": M_enc[-1],
            "Rs": Rs,
            "Uinf": Uinf,
            "Alog": fit["A"],
            "Rlog": fit["Rlog"],
            "rmse_logfit": fit["rmse"],
            "mae_logfit": fit["mae"],
            "nrmse_logfit": fit["nrmse"],
            "R_ratio": fit["Rlog"] / Rs
        })

    except:
        cut_stats["pipeline_fail"] = cut_stats.get("pipeline_fail", 0) + 1
        continue

print("Galaxies fit:", len(rows))
print("\nCUT SUMMARY")
for k in sorted(cut_stats):
    print(f"{k}: {cut_stats[k]}")

if len(rows) == 0:
    raise SystemExit("No valid galaxies.")

# pooled summaries
nrmse = np.array([row["nrmse_logfit"] for row in rows])
rmse  = np.array([row["rmse_logfit"] for row in rows])
rrat  = np.array([row["R_ratio"] for row in rows])
alog  = np.array([row["Alog"] for row in rows])
Mbar  = np.array([row["Mbar"] for row in rows])
Rlog  = np.array([row["Rlog"] for row in rows])
Rs    = np.array([row["Rs"] for row in rows])

print("\nGLOBAL LOG(1+r/R) FIT SUMMARY")
print("="*78)
print("Median absolute RMSE on U(r):", np.median(rmse))
print("Median normalized RMSE on U(r):", np.median(nrmse))
print("Mean normalized RMSE on U(r):", np.mean(nrmse))
print("Median A coefficient:", np.median(alog))
print("Median Rlog / Rs:", np.median(rrat))
print("Mean Rlog / Rs:", np.mean(rrat))

# scaling checks
logM = np.log10(Mbar)
logRlog = np.log10(Rlog)
logRs = np.log10(Rs)
logA = np.log10(np.maximum(alog, EPS))

print("\nSCALING RELATIONS")
print("="*78)
print("Rlog vs M slope:", np.polyfit(logM, logRlog, 1)[0])
print("Rs   vs M slope:", np.polyfit(logM, logRs, 1)[0])
print("Alog vs M slope:", np.polyfit(logM, logA, 1)[0])

# best/worst fits
rows_sorted_best = sorted(rows, key=lambda z: z["nrmse_logfit"])[:12]
rows_sorted_worst = sorted(rows, key=lambda z: z["nrmse_logfit"], reverse=True)[:12]

print("\nBEST 12 LOG-FIT GALAXIES")
print("="*78)
print(f'{"name":<22} {"nRMSE":>8} {"RMSE":>10} {"Alog":>10} {"Rlog":>10} {"Rs":>10} {"Rlog/Rs":>10}')
for row in rows_sorted_best:
    print(f'{row["name"]:<22} {row["nrmse_logfit"]:8.4f} {row["rmse_logfit"]:10.4f} {row["Alog"]:10.4f} {row["Rlog"]:10.4f} {row["Rs"]:10.4f} {row["R_ratio"]:10.4f}')

print("\nWORST 12 LOG-FIT GALAXIES")
print("="*78)
print(f'{"name":<22} {"nRMSE":>8} {"RMSE":>10} {"Alog":>10} {"Rlog":>10} {"Rs":>10} {"Rlog/Rs":>10}')
for row in rows_sorted_worst:
    print(f'{row["name"]:<22} {row["nrmse_logfit"]:8.4f} {row["rmse_logfit"]:10.4f} {row["Alog"]:10.4f} {row["Rlog"]:10.4f} {row["Rs"]:10.4f} {row["R_ratio"]:10.4f}')

# mass-bin summaries
mass_split = np.median(Mbar)
low = [row for row in rows if row["Mbar"] < mass_split]
high = [row for row in rows if row["Mbar"] >= mass_split]

def med(arr, key):
    return np.median([x[key] for x in arr]) if arr else np.nan

print("\nMASS-BIN LOG-FIT SUMMARY")
print("="*78)
print("Mass split:", mass_split)
print("LOW  n =", len(low),
      "| median nRMSE =", med(low, "nrmse_logfit"),
      "| median Rlog/Rs =", med(low, "R_ratio"),
      "| median Alog =", med(low, "Alog"))
print("HIGH n =", len(high),
      "| median nRMSE =", med(high, "nrmse_logfit"),
      "| median Rlog/Rs =", med(high, "R_ratio"),
      "| median Alog =", med(high, "Alog"))

TESTING WHETHER U(r) ~ A log(1 + r/Rlog)
Galaxies fit: 60

CUT SUMMARY
quality_fail: 115

GLOBAL LOG(1+r/R) FIT SUMMARY
Median absolute RMSE on U(r): 9080.116809295008
Median normalized RMSE on U(r): 0.049936680617550244
Mean normalized RMSE on U(r): 0.052756081830829825
Median A coefficient: 116039.12002848479
Median Rlog / Rs: 0.5368458905799722
Mean Rlog / Rs: 0.6484053544415043

SCALING RELATIONS
Rlog vs M slope: 0.29271647973198806
Rs   vs M slope: 0.31881104458934
Alog vs M slope: 0.8820624835582053

BEST 12 LOG-FIT GALAXIES
name                      nRMSE       RMSE       Alog       Rlog         Rs    Rlog/Rs
F583-1_rotmod.dat        0.0282   586.3304 14913.7197     4.8377     6.9700     0.6941
UGC05253_rotmod.dat      0.0324 31580.9509 532419.4668     7.8943    16.6100     0.4753
UGC02953_rotmod.dat      0.0332 95666.9774 1386527.9747     6.8006    16.8300     0.4041
UGC09133_rotmod.dat      0.0360 94684.5587 1238499.9122    11.0604    30.5000     0.3626
UGC06787_rotmod.dat    

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# CLOSED-FORM LOG MASTERCURVE TEST
#
# Part A:
#   Compare closed-form analytic mastercurve
#
#     y_log(x) = 1 - (1 + x)^(-1/ln 2)
#
#   against empirical mastercurve
#
#     y_emp(x) = 1 - exp[-(x/x0)^nu]
#
# Part B:
#   Replace the empirical/stretched-exponential response in the
#   galaxy pipeline by the log mastercurve and report metrics.
# ============================================================

# ------------------------------------------------------------
# EMPIRICAL MASTERCURVE PARAMETERS
# ------------------------------------------------------------
x0_emp = 0.966
nu_emp = 0.942

def y_emp(x, x0=x0_emp, nu=nu_emp):
    return 1.0 - np.exp(- (x / x0)**nu)

def y_log_master(x):
    k = 1.0 / np.log(2.0)
    return 1.0 - np.power(1.0 + x, -k)

# ------------------------------------------------------------
# PART A: DIRECT CURVE COMPARISON
# ------------------------------------------------------------
x_grid = np.linspace(0.0, 6.0, 600)
y_e = y_emp(x_grid)
y_l = y_log_master(x_grid)

rms = np.sqrt(np.mean((y_l - y_e)**2))
mae = np.mean(np.abs(y_l - y_e))

inner = x_grid < 0.8
mid   = (x_grid >= 0.8) & (x_grid <= 2.0)
outer = x_grid > 2.0

rms_in  = np.sqrt(np.mean((y_l[inner] - y_e[inner])**2))
rms_mid = np.sqrt(np.mean((y_l[mid]   - y_e[mid])**2))
rms_out = np.sqrt(np.mean((y_l[outer] - y_e[outer])**2))

print("PART A: CLOSED-FORM LOG MASTERCURVE vs EMPIRICAL")
print("="*78)
print("Empirical: y = 1 - exp[-(x/x0)^nu], with x0=0.966, nu=0.942")
print("Analytic : y = 1 - (1 + x)^(-1/ln2)")
print()
print("Global RMS :", rms)
print("Global MAE :", mae)
print("Inner RMS  :", rms_in)
print("Mid RMS    :", rms_mid)
print("Outer RMS  :", rms_out)

# ------------------------------------------------------------
# PART B: GALAXY PIPELINE WITH LOG MASTERCURVE
# ------------------------------------------------------------

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone from current best branch
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

rows = []
cut_stats = {}

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        cut_stats["read_fail"] = cut_stats.get("read_fail", 0) + 1
        continue

    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        cut_stats["quality_fail"] = cut_stats.get("quality_fail", 0) + 1
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            cut_stats["solver_fail"] = cut_stats.get("solver_fail", 0) + 1
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            cut_stats["bad_uinf"] = cut_stats.get("bad_uinf", 0) + 1
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            cut_stats["bad_Rs"] = cut_stats.get("bad_Rs", 0) + 1
            continue

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "Mbar": M_enc[-1],
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs
        })

    except:
        cut_stats["pipeline_fail"] = cut_stats.get("pipeline_fail", 0) + 1
        continue

print("\nPART B: GALAXY PIPELINE WITH LOG MASTERCURVE")
print("="*78)
print("Galaxies solved:", len(rows))
print("CUT SUMMARY:", cut_stats)

# Fit global amplitude
num = 0.0
den = 0.0
for row in rows:
    y = np.max(row["v_obs"])**2
    x = row["Uinf"] / row["Rs"]
    num += x * y
    den += x * x
A_global = num / den

gal_stats = []

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]
    Mbar = row["Mbar"]

    # Use x = U/Uinf as transport coordinate for the closed-form mastercurve
    x = np.clip(Uchi / Uinf, 0.0, None)

    Vflat2 = A_global * (Uinf / Rs)
    if not np.isfinite(Vflat2) or Vflat2 <= 0:
        continue
    Vflat = np.sqrt(Vflat2)

    y_shape = y_log_master(x)
    v_pred = Vflat * y_shape

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    mae = np.mean(np.abs(v_pred - v_obs))

    xr = r / Rs
    frac = np.abs(v_pred - v_obs) / np.maximum(v_obs, 1.0)
    frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

    inner = xr < 0.7
    mid   = (xr >= 0.7) & (xr <= 2.0)
    outer = xr > 2.0

    gal_stats.append({
        "name": row["name"],
        "Mbar": Mbar,
        "rmse": rmse,
        "mae": mae,
        "inner": np.median(frac[inner]) if np.any(inner) else np.nan,
        "mid": np.median(frac[mid]) if np.any(mid) else np.nan,
        "outer": np.median(frac[outer]) if np.any(outer) else np.nan,
        "signed_inner": np.median(frac_signed[inner]) if np.any(inner) else np.nan,
    })

rmse_arr = np.array([g["rmse"] for g in gal_stats])
M_arr = np.array([g["Mbar"] for g in gal_stats])

low = rmse_arr[M_arr < 1e10]
high = rmse_arr[M_arr >= 1e10]

print("\nLOG MASTERCURVE PERFORMANCE")
print("="*78)
print("Median RMSE:", np.median(rmse_arr))
print("Mean RMSE:", np.mean(rmse_arr))
print("Median signed inner residual:", np.median([g["signed_inner"] for g in gal_stats if np.isfinite(g["signed_inner"])]))
print("Median RMSE low-mass:", np.median(low) if len(low) else np.nan)
print("Median RMSE high-mass:", np.median(high) if len(high) else np.nan)

best = sorted(gal_stats, key=lambda z: z["rmse"])[:10]
worst = sorted(gal_stats, key=lambda z: z["rmse"], reverse=True)[:10]

print("\nBEST 10 GALAXIES")
print("="*78)
print(f'{"name":<22} {"RMSE":>10} {"MAE":>10} {"inner":>10} {"mid":>10} {"outer":>10}')
for g in best:
    print(f'{g["name"]:<22} {g["rmse"]:10.4f} {g["mae"]:10.4f} {g["inner"]:10.4f} {g["mid"]:10.4f} {g["outer"]:10.4f}')

print("\nWORST 10 GALAXIES")
print("="*78)
print(f'{"name":<22} {"RMSE":>10} {"MAE":>10} {"inner":>10} {"mid":>10} {"outer":>10}')
for g in worst:
    print(f'{g["name"]:<22} {g["rmse"]:10.4f} {g["mae"]:10.4f} {g["inner"]:10.4f} {g["mid"]:10.4f} {g["outer"]:10.4f}')

PART A: CLOSED-FORM LOG MASTERCURVE vs EMPIRICAL
Empirical: y = 1 - exp[-(x/x0)^nu], with x0=0.966, nu=0.942
Analytic : y = 1 - (1 + x)^(-1/ln2)

Global RMS : 0.06279092974935153
Global MAE : 0.058443542611961094
Inner RMS  : 0.02354855404625364
Mid RMS    : 0.042942368830475204
Outer RMS  : 0.07245640272892634

PART B: GALAXY PIPELINE WITH LOG MASTERCURVE
Galaxies solved: 60
CUT SUMMARY: {'quality_fail': 115}

LOG MASTERCURVE PERFORMANCE
Median RMSE: 67.62871827227065
Mean RMSE: 85.40041012061275
Median signed inner residual: -0.6837476447751056
Median RMSE low-mass: 35.42479205493142
Median RMSE high-mass: 130.17920567568757

BEST 10 GALAXIES
name                         RMSE        MAE      inner        mid      outer
UGC04305_rotmod.dat        6.4380     5.1453     0.3707     0.0951     0.0805
UGCA444_rotmod.dat        10.6900    10.0497     0.3743     0.3477     0.4202
UGC01281_rotmod.dat       16.5981    13.4333     0.1670     0.4111     0.4819
NGC2366_rotmod.dat        17.6311  

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# FIELD-STRENGTH-REGULATED p OPERATOR TEST
#
# p_eff(r) = 4 - alpha * g_b / (g_b + g_c)
#
# Inner high-field regions  -> p < 4  -> faster accumulation
# Outer low-field regions   -> p -> 4 -> log-like accumulation
#
# This directly tests the "inner regime different, outer near-critical"
# picture suggested by the diagnostics.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0

RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_field_strength_regulated_p(
    r, rho_eff, v_bar2, g_b, alpha, g_c,
    phi_s, n_scr, mu2, p_floor=3.4, p_cap=4.0,
    n_iter=180, relax=0.22
):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    p_eff = 4.0 - alpha * (g_b / (g_b + g_c + EPS))
    p_eff = np.clip(p_eff, p_floor, p_cap)
    p_eff = np.nan_to_num(p_eff, nan=4.0, posinf=p_floor, neginf=4.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = s_fac * np.power(np.abs(dchi) + GRAD_EPS, p_eff - 2.0)
        sigma = np.nan_to_num(sigma, nan=0.0, posinf=1e30, neginf=0.0)

        A = np.zeros((n, n))
        b = np.zeros(n)

        # inner Neumann BC
        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC

            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        # outer Neumann BC
        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None, None

        if not np.all(np.isfinite(chi_new)):
            return None, None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi, p_eff

def build_rows(alpha, g_c):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
            force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * force_factor

            chi, p_eff = solve_field_strength_regulated_p(
                r=r,
                rho_eff=rho_eff,
                v_bar2=v_bar2,
                g_b=g_b,
                alpha=alpha,
                g_c=g_c,
                phi_s=PHI_S,
                n_scr=N_SCR,
                mu2=MU2
            )
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            inner_mask = r <= 0.3 * np.max(r)

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "p_med": float(np.median(p_eff)),
                "p_inner": float(np.median(p_eff[inner_mask])) if np.any(inner_mask) else np.nan
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        if np.isfinite(x) and np.isfinite(y) and x > 0:
            num += x * y
            den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    rmse_list = []
    signed_inner = []
    highmass_rmse = []
    p_med_list = []
    p_inner_list = []
    low_to_eval = []

    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]
        Mbar = row["Mbar"]

        Vflat2 = A * (Uinf / Rs)
        if not np.isfinite(Vflat2) or Vflat2 <= 0:
            continue

        y = np.clip(Uchi / Uinf, 0.0, None)
        v_pred = np.sqrt(Vflat2) * y**RESP_EXP

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        if not np.isfinite(rmse):
            continue

        rmse_list.append(rmse)
        p_med_list.append(row["p_med"])
        p_inner_list.append(row["p_inner"])

        inner_mask = (r / Rs) < 0.7
        if np.any(inner_mask):
            frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)
            signed_inner.append(np.median(frac_signed[inner_mask]))

        if Mbar > 1e10:
            highmass_rmse.append(rmse)

    return {
        "rmse": float(np.median(rmse_list)),
        "signed_inner": float(np.median(signed_inner)),
        "highmass_rmse": float(np.median(highmass_rmse)),
        "p_med": float(np.median(p_med_list)),
        "p_inner": float(np.median(p_inner_list)),
    }

print("FIELD-STRENGTH-REGULATED p SWEEP")
print("="*78)
print("Baseline to beat roughly:")
print("  median RMSE ~ 27.7")
print("  median signed inner residual ~ -0.14")
print("  median RMSE high-mass ~ 69")
print()

for alpha in [0.1, 0.2, 0.3, 0.5]:
    for g_c in [500.0, 1000.0, 2000.0, 4000.0, 8000.0]:
        rows = build_rows(alpha=alpha, g_c=g_c)
        if len(rows) < 20:
            print(f"\nalpha={alpha:.2f}, g_c={g_c:.0f}")
            print("-"*78)
            print("insufficient valid galaxies")
            continue

        A = fit_amplitude(rows)
        res = evaluate(rows, A)

        print(f"\nalpha={alpha:.2f}, g_c={g_c:.0f}")
        print("-"*78)
        print("median RMSE:", res["rmse"])
        print("median signed inner residual:", res["signed_inner"])
        print("median RMSE high-mass:", res["highmass_rmse"])
        print("median p_eff:", res["p_med"])
        print("median inner p_eff:", res["p_inner"])

FIELD-STRENGTH-REGULATED p SWEEP
Baseline to beat roughly:
  median RMSE ~ 27.7
  median signed inner residual ~ -0.14
  median RMSE high-mass ~ 69


alpha=0.10, g_c=500
------------------------------------------------------------------------------
median RMSE: 130.30175645983275
median signed inner residual: -0.999789161681174
median RMSE high-mass: 212.78772704973431
median p_eff: 3.9422871598281475
median inner p_eff: 3.9223305768751424

alpha=0.10, g_c=1000
------------------------------------------------------------------------------
median RMSE: 134.8552464257346
median signed inner residual: -0.9967888821152726
median RMSE high-mass: 211.32730428842217
median p_eff: 3.9594313964332146
median inner p_eff: 3.936506119878164

alpha=0.10, g_c=2000
------------------------------------------------------------------------------
median RMSE: 109.75825567697419
median signed inner residual: -0.9234794342618795
median RMSE high-mass: 178.7912076570625
median p_eff: 3.9745487432918867
medi

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TRANSITION / RANK-LANDSCAPE DIAGNOSTIC
#
# Goal:
#   Stop looking only at best/worst tails.
#   Sort all galaxies by RMSE and inspect how properties change
#   across the full ranked sample.
#
# Reports:
#   - decile summary from best -> worst
#   - whether deterioration is smooth or cliff-like
#   - median morphology/source proxies by decile
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)
    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def med(x):
    arr = np.array([v for v in x if np.isfinite(v)])
    return np.median(arr) if len(arr) else np.nan

print("TRANSITION / RANK-LANDSCAPE DIAGNOSTIC")
print("="*90)

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        v_bar = np.sqrt(v_bar2)

        M_enc = r * v_bar2 / G
        Mbar = M_enc[-1]

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        # feature proxies
        vmax_obs = np.max(v_obs)
        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS
        gas_frac = vg / denom
        disk_frac = vd / denom
        bulge_frac = vb / denom

        rmax = np.max(r)
        inner = r <= 0.3 * rmax
        c3 = M_enc[inner][-1] / max(M_enc[-1], EPS) if np.any(inner) else np.nan

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": Mbar,
            "vmax": vmax_obs,
            "gas_frac": gas_frac,
            "disk_frac": disk_frac,
            "bulge_frac": bulge_frac,
            "compactness": c3
        })
    except Exception:
        continue

# fit amplitude on whole sample
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = np.max(row["v_obs"])**2
    num += x * y
    den += x * x
A_global = num / den

# compute residual diagnostics
for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat = np.sqrt(A_global * (Uinf / Rs))
    v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
    xr = r / Rs
    frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

    row["rmse"] = rmse
    row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
    row["signed_mid"]   = np.median(frac_signed[(xr >= 0.7) & (xr <= 2.0)]) if np.any((xr >= 0.7) & (xr <= 2.0)) else np.nan
    row["signed_outer"] = np.median(frac_signed[xr > 2.0]) if np.any(xr > 2.0) else np.nan

rows = sorted(rows, key=lambda z: z["rmse"])
n = len(rows)
print("Galaxies analysed:", n)

# deciles
edges = np.linspace(0, n, 11, dtype=int)

print("\nDECILE SUMMARY (best -> worst)")
print("="*90)
print(f'{"decile":<8} {"n":>3} {"RMSE":>10} {"s_in":>10} {"s_mid":>10} {"s_out":>10} {"logM":>10} {"Vmax":>10} {"bulge":>10} {"gas":>10} {"compact":>10}')
for i in range(10):
    chunk = rows[edges[i]:edges[i+1]]
    print(
        f'{i+1:<8} '
        f'{len(chunk):>3} '
        f'{med([c["rmse"] for c in chunk]):10.3f} '
        f'{med([c["signed_inner"] for c in chunk]):10.3f} '
        f'{med([c["signed_mid"] for c in chunk]):10.3f} '
        f'{med([c["signed_outer"] for c in chunk]):10.3f} '
        f'{med([np.log10(c["Mbar"]) for c in chunk]):10.3f} '
        f'{med([c["vmax"] for c in chunk]):10.3f} '
        f'{med([c["bulge_frac"] for c in chunk]):10.3f} '
        f'{med([c["gas_frac"] for c in chunk]):10.3f} '
        f'{med([c["compactness"] for c in chunk]):10.3f}'
    )

# adjacent jumps
dec_rmse = [med([c["rmse"] for c in rows[edges[i]:edges[i+1]]]) for i in range(10)]
jumps = np.diff(dec_rmse)

print("\nADJACENT DECILE RMSE JUMPS")
print("="*90)
for i, j in enumerate(jumps, start=1):
    print(f"{i}->{i+1}: {j:.3f}")

print("\nINTERPRETATION HINT")
print("="*90)
print("If decile RMSE rises roughly smoothly, the failure is gradual.")
print("If there is a big jump in later deciles, the model has a tail pathology / regime split.")
print("If bulge_frac / compactness rise together with RMSE, the degradation is morphology-linked.")

TRANSITION / RANK-LANDSCAPE DIAGNOSTIC
Galaxies analysed: 60

DECILE SUMMARY (best -> worst)
decile     n       RMSE       s_in      s_mid      s_out       logM       Vmax      bulge        gas    compact
1          6      8.992      0.575      0.002     -0.081      8.783     60.600      0.390      0.170      0.291
2          6     14.087      0.428     -0.010     -0.124      9.366     87.500      0.587      0.103      0.273
3          6     17.453      0.208     -0.228     -0.260      9.056     84.750      0.599      0.108      0.363
4          6     21.941     -0.096     -0.124     -0.178      9.774    109.000      0.715      0.051      0.511
5          6     24.213     -0.165     -0.214     -0.217      9.807    115.000      0.587      0.071      0.688
6          6     34.342     -0.195     -0.133      0.070     10.596    170.500      0.789      0.052      0.695
7          6     48.049     -0.236      0.003      0.060     10.676    213.000      0.924      0.029      0.845
8          

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# EFFECTIVE MODULUS / LIMIT OF PROPORTIONALITY DIAGNOSTIC
#
# Goal:
#   Treat the solved field response like a constitutive law.
#
# Definitions:
#   E_eff(r)   = g_b(r) / chi(r)
#   alpha(r)   = d ln chi / d ln g_b
#
# We then sort galaxies by RMSE and inspect deciles to see:
#   - whether the effective modulus changes gradually
#   - whether alpha departs from ~1 in inner regions
#   - where the "limit of proportionality" appears
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-8
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

print("EFFECTIVE MODULUS / LIMIT OF PROPORTIONALITY DIAGNOSTIC")
print("="*100)

rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / (r + EPS)

        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        # fit amplitude on the fly later, but store constitutive quantities now
        # E_eff = g_b / chi
        E_eff = g_b / (chi + EPS)

        # alpha = d ln chi / d ln g_b
        mask = (chi > 0) & (g_b > 0)
        alpha = np.full_like(r, np.nan)
        if np.sum(mask) >= 5:
            logchi = np.log(chi[mask] + EPS)
            loggb = np.log(g_b[mask] + EPS)
            # interpolate derivative back to masked points
            dlogchi = np.gradient(logchi, r[mask])
            dloggb  = np.gradient(loggb, r[mask])
            slope = dlogchi / (dloggb + EPS)
            alpha[mask] = slope

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "Uchi": Uchi,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": M_enc[-1],
            "g_b": g_b,
            "chi": chi,
            "E_eff": E_eff,
            "alpha": alpha,
        })

    except Exception:
        continue

# Fit global amplitude for velocity residual ranking
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = np.max(row["v_obs"])**2
    num += x * y
    den += x * x
A_global = num / den

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Uchi = row["Uchi"]
    Uinf = row["Uinf"]
    Rs = row["Rs"]

    Vflat = np.sqrt(A_global * (Uinf / Rs))
    v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)
    rmse = np.sqrt(np.mean((v_pred - v_obs)**2))

    xr = r / Rs
    inner = xr < 0.7
    mid   = (xr >= 0.7) & (xr <= 2.0)
    outer = xr > 2.0

    row["rmse"] = rmse
    row["E_in"] = med(row["E_eff"][inner]) if np.any(inner) else np.nan
    row["E_mid"] = med(row["E_eff"][mid]) if np.any(mid) else np.nan
    row["E_out"] = med(row["E_eff"][outer]) if np.any(outer) else np.nan
    row["a_in"] = med(row["alpha"][inner]) if np.any(inner) else np.nan
    row["a_mid"] = med(row["alpha"][mid]) if np.any(mid) else np.nan
    row["a_out"] = med(row["alpha"][outer]) if np.any(outer) else np.nan

rows = sorted(rows, key=lambda z: z["rmse"])
n = len(rows)
print("Galaxies analysed:", n)

edges = np.linspace(0, n, 11, dtype=int)

print("\nDECILE CONSTITUTIVE SUMMARY (best -> worst)")
print("="*100)
print(f'{"decile":<8} {"n":>3} {"RMSE":>10} {"logM":>10} {"E_in":>12} {"E_mid":>12} {"E_out":>12} {"a_in":>10} {"a_mid":>10} {"a_out":>10}')
for i in range(10):
    chunk = rows[edges[i]:edges[i+1]]
    print(
        f'{i+1:<8} '
        f'{len(chunk):>3} '
        f'{med([c["rmse"] for c in chunk]):10.3f} '
        f'{med([np.log10(c["Mbar"]) for c in chunk]):10.3f} '
        f'{med([c["E_in"] for c in chunk]):12.3f} '
        f'{med([c["E_mid"] for c in chunk]):12.3f} '
        f'{med([c["E_out"] for c in chunk]):12.3f} '
        f'{med([c["a_in"] for c in chunk]):10.3f} '
        f'{med([c["a_mid"] for c in chunk]):10.3f} '
        f'{med([c["a_out"] for c in chunk]):10.3f} '
    )

print("\nINTERPRETATION GUIDE")
print("="*100)
print("E_in, E_mid, E_out  : effective modulus analogue g_b / chi")
print("a_in, a_mid, a_out  : local constitutive slope d ln chi / d ln g_b")
print()
print("If alpha ~ 1, response is locally proportional.")
print("If alpha departs strongly from 1 in inner deciles, that is your limit-of-proportionality signal.")
print("If E_in ramps strongly with RMSE while E_out stays similar, the inner regime is the real problem.")

EFFECTIVE MODULUS / LIMIT OF PROPORTIONALITY DIAGNOSTIC
Galaxies analysed: 60

DECILE CONSTITUTIVE SUMMARY (best -> worst)
decile     n       RMSE       logM         E_in        E_mid        E_out       a_in      a_mid      a_out
1          6      8.990      8.783        0.089        0.117        0.918      0.124      1.351      5.978 
2          6     14.079      9.366        0.129        0.161        0.875      0.074      1.407      4.595 
3          6     17.446      9.056        0.255        0.162        1.001      0.341      0.746      4.123 
4          6     21.949      9.774        0.139        0.146        0.649      0.430      1.319      3.333 
5          6     24.216      9.807        0.117        0.092        0.289      0.499      0.952      4.180 
6          6     34.320     10.596        0.126        0.147        0.345      0.388      1.295      3.236 
7          6     48.026     10.676        0.183        0.083        0.204      0.422      0.884      3.091 
8          6  

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SUBLINEAR SOURCE-LAW SWEEP
#
# Test:
#   rho_eff = rho * (1 + A_G * (g_b / GREF)^gamma)^M_EXP
#
# with gamma varied to match the measured constitutive slope
# in the inner region.
#
# Goal:
#   see whether reducing the source-law exponent gamma
#   fixes the gradual degradation across deciles, especially
#   the inner underprediction in compact/bulge-heavy systems.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(gamma):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            source_boost = np.power(1.0 + A_G * np.power(np.clip(g_b / GREF, 0.0, None), gamma), M_EXP)
            source_boost = np.nan_to_num(source_boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * source_boost

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            # morphology/source proxies
            vmax_obs = np.max(v_obs)
            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS
            gas_frac = vg / denom
            disk_frac = vd / denom
            bulge_frac = vb / denom

            rmax = np.max(r)
            inner_mass_mask = r <= 0.3 * rmax
            compactness = M_enc[inner_mass_mask][-1] / max(M_enc[-1], EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": M_enc[-1],
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "vmax": vmax_obs,
                "bulge_frac": bulge_frac,
                "gas_frac": gas_frac,
                "compactness": compactness
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        row["signed_mid"]   = np.median(frac_signed[(xr >= 0.7) & (xr <= 2.0)]) if np.any((xr >= 0.7) & (xr <= 2.0)) else np.nan
        row["signed_outer"] = np.median(frac_signed[xr > 2.0]) if np.any(xr > 2.0) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])
    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    return {
        "rows_sorted": rows_sorted,
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
    }

def decile_summary(rows_sorted):
    n = len(rows_sorted)
    edges = np.linspace(0, n, 11, dtype=int)
    out = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        out.append({
            "decile": i+1,
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "logM": med([np.log10(c["Mbar"]) for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
        })
    return out

print("SUBLINEAR SOURCE-LAW SWEEP")
print("="*100)
print("Baseline to beat roughly:")
print("  median RMSE ~ 27.7")
print("  median signed inner residual ~ -0.14")
print("  median high-mass RMSE ~ 69")
print()

all_results = []

for gamma in [0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
    rows = build_rows(gamma)
    if len(rows) < 20:
        print(f"gamma={gamma:.2f} -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)
    dec = decile_summary(res["rows_sorted"])

    jumps = np.diff([d["rmse"] for d in dec])
    knee_jump = np.max(jumps) if len(jumps) else np.nan

    all_results.append({
        "gamma": gamma,
        "median_rmse": res["median_rmse"],
        "median_signed_inner": res["median_signed_inner"],
        "median_highmass_rmse": res["median_highmass_rmse"],
        "median_lowmass_rmse": res["median_lowmass_rmse"],
        "max_decile_jump": knee_jump,
        "deciles": dec
    })

    print(f"gamma={gamma:.2f}")
    print("-"*100)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile RMSE jump:", knee_jump)
    print("decile 1/5/10 signed inner:",
          dec[0]["s_in"], dec[4]["s_in"], dec[9]["s_in"])
    print()

print("\nCOMPACT SUMMARY")
print("="*100)
print(f'{"gamma":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["gamma"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

# show deciles for best gamma by median RMSE
if all_results:
    best = sorted(all_results, key=lambda z: z["median_rmse"])[0]
    print("\nBEST GAMMA DECILE SUMMARY")
    print("="*100)
    print("best gamma =", best["gamma"])
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"logM":>10} {"bulge":>10} {"gas":>10} {"compact":>10}')
    for d in best["deciles"]:
        print(f'{d["decile"]:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["logM"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["compact"]:10.3f}')

SUBLINEAR SOURCE-LAW SWEEP
Baseline to beat roughly:
  median RMSE ~ 27.7
  median signed inner residual ~ -0.14
  median high-mass RMSE ~ 69

gamma=0.20
----------------------------------------------------------------------------------------------------
median RMSE: 29.581615570410403
median signed inner residual: -0.07254991964876402
median low-mass RMSE: 13.945491174631577
median high-mass RMSE: 53.65529965600007
largest adjacent decile RMSE jump: 21.918144758030394
decile 1/5/10 signed inner: 0.10613144253740836 0.25733797400367814 -0.41241428598368146

gamma=0.30
----------------------------------------------------------------------------------------------------
median RMSE: 29.71342364285076
median signed inner residual: -0.1129016543413337
median low-mass RMSE: 14.877071118960561
median high-mass RMSE: 58.948687892033135
largest adjacent decile RMSE jump: 18.47098474132541
decile 1/5/10 signed inner: 0.2593304873696699 -0.1679880040298864 -0.38018744188786757

gamma=0.40
-------

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP
#
# Test:
#   rho_eff = rho * (1 + A_G * (g_b / GREF)^gamma_eff)^M_EXP
#
# with
#   gamma_eff = clip(gamma0 - gamma1 * C, gamma_min, gamma_max)
#
# where
#   C = M(<0.3 Rmax) / Mtot
#
# Goal:
#   keep diffuse galaxies closer to linear response,
#   while compact galaxies get the measured sublinear response.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# -------- Frozen backbone --------
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

# -------- Quality cuts --------
MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(gamma0, gamma1, gamma_min=0.2, gamma_max=1.0):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rmax = np.max(r)
            inner_mass_mask = r <= 0.3 * rmax
            compactness = M_enc[inner_mass_mask][-1] / Mtot if np.any(inner_mass_mask) else np.nan
            if not np.isfinite(compactness):
                continue

            gamma_eff = gamma0 - gamma1 * compactness
            gamma_eff = np.clip(gamma_eff, gamma_min, gamma_max)

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            source_boost = np.power(
                1.0 + A_G * np.power(np.clip(g_b / GREF, 0.0, None), gamma_eff),
                M_EXP
            )
            source_boost = np.nan_to_num(source_boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * source_boost

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS
            gas_frac = vg / denom
            bulge_frac = vb / denom

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "gamma_eff": gamma_eff,
                "bulge_frac": bulge_frac,
                "gas_frac": gas_frac
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "gamma_eff": med([c["gamma_eff"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "rows_sorted": rows_sorted,
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "median_gamma_eff": med([r["gamma_eff"] for r in rows_sorted]),
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec
    }

print("COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP")
print("="*110)
print("Baseline references:")
print("  gamma = 1.0 constant -> median RMSE ~ 27.7, signed inner ~ -0.14, high-mass RMSE ~ 69")
print("  best constant sublinear rows were around gamma ~ 0.4 to 0.5")
print()

all_results = []

# gamma_eff = gamma0 - gamma1 * compactness
for gamma0, gamma1 in [
    (1.0, 0.5),
    (1.0, 0.7),
    (1.0, 0.9),
    (0.9, 0.5),
    (0.9, 0.7),
    (0.8, 0.4),
    (0.8, 0.6),
]:
    rows = build_rows(gamma0, gamma1)
    if len(rows) < 20:
        print(f"(gamma0={gamma0:.2f}, gamma1={gamma1:.2f}) -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)

    all_results.append({
        "gamma0": gamma0,
        "gamma1": gamma1,
        "median_rmse": res["median_rmse"],
        "median_signed_inner": res["median_signed_inner"],
        "median_lowmass_rmse": res["median_lowmass_rmse"],
        "median_highmass_rmse": res["median_highmass_rmse"],
        "median_gamma_eff": res["median_gamma_eff"],
        "max_decile_jump": res["max_decile_jump"],
        "deciles": res["deciles"],
    })

    print(f"(gamma0={gamma0:.2f}, gamma1={gamma1:.2f})")
    print("-"*110)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("median gamma_eff:", res["median_gamma_eff"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print("decile 1/5/10 signed inner:",
          res["deciles"][0]["s_in"],
          res["deciles"][4]["s_in"],
          res["deciles"][9]["s_in"])
    print("decile 1/5/10 gamma_eff:",
          res["deciles"][0]["gamma_eff"],
          res["deciles"][4]["gamma_eff"],
          res["deciles"][9]["gamma_eff"])
    print()

print("\nCOMPACT SUMMARY")
print("="*110)
print(f'{"g0":>6} {"g1":>6} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"gamma_med":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["gamma0"]:6.2f} {r["gamma1"]:6.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["median_gamma_eff"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    # choose a compromise: prioritize lower high-mass RMSE and smoother jump,
    # while keeping median RMSE near baseline
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["max_decile_jump"], z["median_rmse"]))[0]
    print("\nBEST COMPACTNESS-DEPENDENT ROW")
    print("="*110)
    print("gamma0 =", best["gamma0"])
    print("gamma1 =", best["gamma1"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("median gamma_eff =", best["median_gamma_eff"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"gamma":>10} {"bulge":>10} {"gas":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["gamma_eff"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f}')

COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP
Baseline references:
  gamma = 1.0 constant -> median RMSE ~ 27.7, signed inner ~ -0.14, high-mass RMSE ~ 69
  best constant sublinear rows were around gamma ~ 0.4 to 0.5

(gamma0=1.00, gamma1=0.50)
--------------------------------------------------------------------------------------------------------------
median RMSE: 29.115112889037917
median signed inner residual: -0.08404483553377087
median low-mass RMSE: 16.02266778415489
median high-mass RMSE: 56.87572657397449
median gamma_eff: 0.6807747671120237
largest adjacent decile jump: 15.293595849189117
decile 1/5/10 signed inner: 0.21897184778149223 0.23795976883898595 -0.3836065393809345
decile 1/5/10 gamma_eff: 0.8544242679610614 0.748446501784169 0.514156893064027

(gamma0=1.00, gamma1=0.70)
--------------------------------------------------------------------------------------------------------------
median RMSE: 29.98712889030598
median signed inner residual: -0.08742133917502362
median low-m

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SIGMOID COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP
#
# rho_eff = rho * (1 + A_G * (g_b / GREF)^gamma_eff)^M_EXP
#
# gamma_eff(C) = gamma_hi - (gamma_hi - gamma_lo) /
#                (1 + exp(-(C - Cstar)/Delta))
#
# Goal:
#   model a genuine constitutive knee rather than a linear trend.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def gamma_sigmoid(C, gamma_hi, gamma_lo, Cstar, Delta):
    z = np.clip((C - Cstar) / max(Delta, 1e-4), -60, 60)
    return gamma_hi - (gamma_hi - gamma_lo) / (1.0 + np.exp(-z))

def build_rows(gamma_hi, gamma_lo, Cstar, Delta):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue

        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rmax = np.max(r)
            inner_mass_mask = r <= 0.3 * rmax
            compactness = M_enc[inner_mass_mask][-1] / Mtot if np.any(inner_mass_mask) else np.nan
            if not np.isfinite(compactness):
                continue

            gamma_eff = gamma_sigmoid(compactness, gamma_hi, gamma_lo, Cstar, Delta)
            gamma_eff = float(np.clip(gamma_eff, 0.2, 1.0))

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            source_boost = np.power(
                1.0 + A_G * np.power(np.clip(g_b / GREF, 0.0, None), gamma_eff),
                M_EXP
            )
            source_boost = np.nan_to_num(source_boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * source_boost

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS
            gas_frac = vg / denom
            bulge_frac = vb / denom

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "gamma_eff": gamma_eff,
                "bulge_frac": bulge_frac,
                "gas_frac": gas_frac
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "gamma_eff": med([c["gamma_eff"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "median_gamma_eff": med([r["gamma_eff"] for r in rows_sorted]),
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("SIGMOID COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP")
print("="*118)
print("Baseline references:")
print("  constant gamma=1.0 -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24")
print("  best linear-compactness row -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4")
print()

all_results = []

grid = [
    (1.00, 0.40, 0.55, 0.08),
    (1.00, 0.35, 0.60, 0.08),
    (0.95, 0.35, 0.60, 0.10),
    (0.90, 0.30, 0.58, 0.10),
    (0.90, 0.35, 0.55, 0.12),
    (0.85, 0.30, 0.60, 0.12),
    (0.80, 0.25, 0.62, 0.12),
]

for gamma_hi, gamma_lo, Cstar, Delta in grid:
    rows = build_rows(gamma_hi, gamma_lo, Cstar, Delta)
    if len(rows) < 20:
        print(f"(ghi={gamma_hi:.2f}, glo={gamma_lo:.2f}, C*={Cstar:.2f}, D={Delta:.2f}) -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)

    all_results.append({
        "gamma_hi": gamma_hi,
        "gamma_lo": gamma_lo,
        "Cstar": Cstar,
        "Delta": Delta,
        "median_rmse": res["median_rmse"],
        "median_signed_inner": res["median_signed_inner"],
        "median_lowmass_rmse": res["median_lowmass_rmse"],
        "median_highmass_rmse": res["median_highmass_rmse"],
        "median_gamma_eff": res["median_gamma_eff"],
        "max_decile_jump": res["max_decile_jump"],
        "deciles": res["deciles"],
    })

    print(f"(ghi={gamma_hi:.2f}, glo={gamma_lo:.2f}, C*={Cstar:.2f}, D={Delta:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("median gamma_eff:", res["median_gamma_eff"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print("decile 1/5/10 signed inner:",
          res["deciles"][0]["s_in"], res["deciles"][4]["s_in"], res["deciles"][9]["s_in"])
    print("decile 1/5/10 gamma_eff:",
          res["deciles"][0]["gamma_eff"], res["deciles"][4]["gamma_eff"], res["deciles"][9]["gamma_eff"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"ghi":>6} {"glo":>6} {"C*":>6} {"D":>6} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"gamma_med":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["gamma_hi"]:6.2f} {r["gamma_lo"]:6.2f} {r["Cstar"]:6.2f} {r["Delta"]:6.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["median_gamma_eff"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["max_decile_jump"], z["median_rmse"]))[0]
    print("\nBEST SIGMOID ROW")
    print("="*118)
    print("gamma_hi =", best["gamma_hi"])
    print("gamma_lo =", best["gamma_lo"])
    print("Cstar    =", best["Cstar"])
    print("Delta    =", best["Delta"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("median gamma_eff =", best["median_gamma_eff"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"gamma":>10} {"bulge":>10} {"gas":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["gamma_eff"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f}')

SIGMOID COMPACTNESS-DEPENDENT SOURCE-LAW SWEEP
Baseline references:
  constant gamma=1.0 -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24
  best linear-compactness row -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4

(ghi=1.00, glo=0.40, C*=0.55, D=0.08)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 29.66547910902846
median signed inner residual: -0.08867352885389726
median low-mass RMSE: 15.08946042027896
median high-mass RMSE: 57.13965991669732
median gamma_eff: 0.549216980486479
largest adjacent decile jump: 15.165941088780741
decile 1/5/10 signed inner: 0.23294199585106243 0.019549078478199935 -0.36518137282588925
decile 1/5/10 gamma_eff: 0.9765279533971681 0.7853816569632797 0.4032077642273228

(ghi=1.00, glo=0.35, C*=0.60, D=0.08)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 30

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# TWO-SLOPE FIELD-STRENGTH CONSTITUTIVE LAW SWEEP
#
# rho_eff = rho * [1 + A_G * (g_b/GREF)^g_lo * (1 + g_b/g_t)^(g_hi - g_lo)]
#
# Behaviour:
#   g_b << g_t  -> effective slope ~ g_lo
#   g_b >> g_t  -> effective slope ~ g_hi
#
# We test the idea:
#   low field  = stiffer / more rigid   -> g_lo > 1
#   high field = softer / saturating    -> g_hi < 1
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20

PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(g_lo, g_hi, g_t):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            # two-slope constitutive factor
            base = np.power(np.clip(g_b / GREF, 0.0, None), g_lo)
            bridge = np.power(1.0 + np.clip(g_b / g_t, 0.0, None), g_hi - g_lo)
            constitutive = 1.0 + A_G * base * bridge
            constitutive = np.nan_to_num(constitutive, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * constitutive

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            rmax = np.max(r)
            inner_mask = r <= 0.3 * rmax
            compactness = M_enc[inner_mask][-1] / max(Mtot, EPS) if np.any(inner_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("TWO-SLOPE FIELD-STRENGTH CONSTITUTIVE LAW SWEEP")
print("="*118)
print("Baseline references:")
print("  constant gamma=1.0 -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24")
print("  best linear-compactness row -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4")
print()

grid = [
    (1.20, 0.50,  500),
    (1.20, 0.40, 1000),
    (1.10, 0.50, 1000),
    (1.10, 0.40, 1500),
    (1.30, 0.50, 1000),
    (1.30, 0.40, 1500),
    (1.15, 0.35, 1200),
]

all_results = []

for g_lo, g_hi, g_t in grid:
    rows = build_rows(g_lo, g_hi, g_t)
    if len(rows) < 20:
        print(f"(g_lo={g_lo:.2f}, g_hi={g_hi:.2f}, g_t={g_t:.0f}) -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)

    all_results.append({
        "g_lo": g_lo,
        "g_hi": g_hi,
        "g_t": g_t,
        "median_rmse": res["median_rmse"],
        "median_signed_inner": res["median_signed_inner"],
        "median_lowmass_rmse": res["median_lowmass_rmse"],
        "median_highmass_rmse": res["median_highmass_rmse"],
        "max_decile_jump": res["max_decile_jump"],
        "deciles": res["deciles"],
    })

    print(f"(g_lo={g_lo:.2f}, g_hi={g_hi:.2f}, g_t={g_t:.0f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print("decile 1/5/10 signed inner:",
          res["deciles"][0]["s_in"], res["deciles"][4]["s_in"], res["deciles"][9]["s_in"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"g_lo":>8} {"g_hi":>8} {"g_t":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["g_lo"]:8.2f} {r["g_hi"]:8.2f} {r["g_t"]:8.0f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["max_decile_jump"], z["median_rmse"]))[0]
    print("\nBEST TWO-SLOPE ROW")
    print("="*118)
    print("g_lo =", best["g_lo"])
    print("g_hi =", best["g_hi"])
    print("g_t  =", best["g_t"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f}')

TWO-SLOPE FIELD-STRENGTH CONSTITUTIVE LAW SWEEP
Baseline references:
  constant gamma=1.0 -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24
  best linear-compactness row -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4

(g_lo=1.20, g_hi=0.50, g_t=500)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 34.90197823058983
median signed inner residual: -0.27255767259061947
median low-mass RMSE: 17.90716672580872
median high-mass RMSE: 80.53246061235822
largest adjacent decile jump: 31.570191115869193
decile 1/5/10 signed inner: 0.360039009631124 -0.27924228886504787 -0.5087325613532014

(g_lo=1.20, g_hi=0.40, g_t=1000)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 28.67003329220452
median signed inner residual: -0.0873582061581456
median low-mass RMSE: 15.356945599949508
median high-mass RMSE: 5

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# CONSTITUTIVE CURVE DIAGNOSTIC
#
# Compare two best constitutive models:
#
# A) Compactness-dependent gamma:
#    gamma_eff = 0.8 - 0.6 * C
#
# B) Two-slope field-strength law:
#    source_boost - 1 ~ (g_b/GREF)^g_lo * (1 + g_b/g_t)^(g_hi-g_lo)
#    with g_lo=1.2, g_hi=0.4, g_t=1000
#
# We measure the implied local constitutive slope:
#
#   beta_eff = d ln(source_boost - 1) / d ln g_b
#
# across all valid galaxies, then bin by g_b.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
GREF = 1000.0
A_G = 0.40
EPS = 1e-12

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, 1e-9)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

# Best compactness row
def gamma_compactness(C):
    return np.clip(0.8 - 0.6 * C, 0.2, 1.0)

# Best two-slope row
G_LO = 1.2
G_HI = 0.4
G_T  = 1000.0

all_gb = []
all_beta_comp = []
all_beta_two = []
all_boost_comp = []
all_boost_two = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
    M_enc = r * v_bar2 / G
    Mtot = M_enc[-1]
    if not np.isfinite(Mtot) or Mtot <= 0:
        continue

    rmax = np.max(r)
    inner_mask = r <= 0.3 * rmax
    if not np.any(inner_mask):
        continue
    C = M_enc[inner_mask][-1] / Mtot
    if not np.isfinite(C):
        continue

    g_b = v_bar2 / np.maximum(r, 1e-9)
    gnorm = np.clip(g_b / GREF, 0.0, None)

    # ---- Model A: compactness-dependent gamma
    gamma_eff = gamma_compactness(C)
    boost_comp = A_G * np.power(gnorm, gamma_eff)   # source_boost - 1

    # local slope in log-log is just gamma_eff wherever boost>0
    beta_comp = np.full_like(g_b, gamma_eff)

    # ---- Model B: two-slope field-strength law
    boost_two = A_G * np.power(gnorm, G_LO) * np.power(1.0 + np.clip(g_b / G_T, 0.0, None), G_HI - G_LO)

    # numerical effective slope
    valid = (g_b > 0) & (boost_two > 0)
    beta_two = np.full_like(g_b, np.nan)
    if np.sum(valid) >= 5:
        lg = np.log(g_b[valid] + EPS)
        lb = np.log(boost_two[valid] + EPS)
        d_lb = np.gradient(lb, r[valid])
        d_lg = np.gradient(lg, r[valid])
        beta_two[valid] = d_lb / (d_lg + EPS)

    # keep all valid radial points
    keep = (g_b > 0) & np.isfinite(beta_comp) & np.isfinite(beta_two) & np.isfinite(boost_comp) & np.isfinite(boost_two)
    all_gb.extend(g_b[keep].tolist())
    all_beta_comp.extend(beta_comp[keep].tolist())
    all_beta_two.extend(beta_two[keep].tolist())
    all_boost_comp.extend(boost_comp[keep].tolist())
    all_boost_two.extend(boost_two[keep].tolist())

all_gb = np.array(all_gb)
all_beta_comp = np.array(all_beta_comp)
all_beta_two = np.array(all_beta_two)
all_boost_comp = np.array(all_boost_comp)
all_boost_two = np.array(all_boost_two)

print("CONSTITUTIVE CURVE DIAGNOSTIC")
print("="*100)
print("Points pooled:", len(all_gb))
print()

# pooled overall summaries
print("GLOBAL POOLED SUMMARIES")
print("-"*100)
print("Compactness-law median beta_eff:", med(all_beta_comp))
print("Two-slope-law   median beta_eff:", med(all_beta_two))
print("Compactness-law median boost   :", med(all_boost_comp))
print("Two-slope-law   median boost   :", med(all_boost_two))
print()

# g_b bins
logg = np.log10(all_gb)
bins = np.linspace(np.nanpercentile(logg, 5), np.nanpercentile(logg, 95), 9)

print("BINNED BY log10(g_b)")
print("="*100)
print(f'{"bin":<8} {"logg_mid":>10} {"n":>6} {"beta_comp":>12} {"beta_two":>12} {"boost_comp":>14} {"boost_two":>14}')
for i in range(len(bins)-1):
    m = (logg >= bins[i]) & (logg < bins[i+1])
    if np.sum(m) == 0:
        continue
    print(
        f'{i+1:<8} '
        f'{med(logg[m]):10.3f} '
        f'{np.sum(m):6d} '
        f'{med(all_beta_comp[m]):12.3f} '
        f'{med(all_beta_two[m]):12.3f} '
        f'{med(all_boost_comp[m]):14.6f} '
        f'{med(all_boost_two[m]):14.6f}'
    )

# simple low/mid/high field splits
q1, q2 = np.nanpercentile(all_gb, [33.3, 66.7])
low = all_gb < q1
mid = (all_gb >= q1) & (all_gb < q2)
high = all_gb >= q2

print("\nLOW / MID / HIGH FIELD SUMMARY")
print("="*100)
for label, mask in [("LOW", low), ("MID", mid), ("HIGH", high)]:
    print(f"{label} FIELD")
    print(f"  median log10(g_b): {med(np.log10(all_gb[mask])):.3f}")
    print(f"  compactness-law beta_eff: {med(all_beta_comp[mask]):.3f}")
    print(f"  two-slope-law   beta_eff: {med(all_beta_two[mask]):.3f}")
    print(f"  compactness-law boost   : {med(all_boost_comp[mask]):.6f}")
    print(f"  two-slope-law   boost   : {med(all_boost_two[mask]):.6f}")
    print()

print("INTERPRETATION")
print("="*100)
print("Compactness-law: beta_eff should stay nearly constant in g_b within a given galaxy family.")
print("Two-slope-law  : beta_eff should be >1 at low field and drift below 1 at high field.")
print("If the binned beta_two shows that pattern cleanly, your 'rigid outside, soft inside' idea is real.")

CONSTITUTIVE CURVE DIAGNOSTIC
Points pooled: 2165

GLOBAL POOLED SUMMARIES
----------------------------------------------------------------------------------------------------
Compactness-law median beta_eff: 0.3616651687618883
Two-slope-law   median beta_eff: 0.8421259124576149
Compactness-law median boost   : 0.3716690764197446
Two-slope-law   median boost   : 0.19290081804056083

BINNED BY log10(g_b)
bin        logg_mid      n    beta_comp     beta_two     boost_comp      boost_two
1             1.838    190        0.612        1.149       0.086620       0.015298
2             2.123    255        0.513        1.104       0.145462       0.032054
3             2.391    235        0.403        1.041       0.227518       0.062299
4             2.680    207        0.358        0.939       0.307448       0.120914
5             2.966    262        0.378        0.818       0.388270       0.215477
6             3.230    352        0.362        0.695       0.487528       0.341491
7           

In [ ]:
import numpy as np
import glob
import os

# ============================================================
# DIRECT CONSTITUTIVE LAW FROM DATA
#
# Measure:
#   chi(r) = Vinf^2 - V(r)^2
#   beta_data = d ln chi / d ln g_b
#
# Pool across galaxies and bin by g_b.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-12

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, 1e-9)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

all_gb = []
all_beta_data = []
all_chi = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)

    # baryonic field
    g_b = v_bar2 / np.maximum(r, 1e-9)

    # V_infinity estimate from outer region
    outer_mask = r >= 0.8 * np.max(r)
    Vinf = np.median(v_obs[outer_mask])
    if not np.isfinite(Vinf) or Vinf <= 0:
        continue

    # chi from data
    chi = np.maximum(Vinf**2 - v_obs**2, 0.0)

    # need positive chi and g_b
    valid = (chi > 0) & (g_b > 0)
    if np.sum(valid) < 8:
        continue

    r_v = r[valid]
    chi_v = chi[valid]
    gb_v = g_b[valid]

    # compute local slope beta_data = d ln chi / d ln g_b
    ln_chi = np.log(chi_v + EPS)
    ln_gb  = np.log(gb_v + EPS)

    d_ln_chi = np.gradient(ln_chi, r_v)
    d_ln_gb  = np.gradient(ln_gb,  r_v)

    beta = d_ln_chi / (d_ln_gb + EPS)

    keep = np.isfinite(beta)
    all_gb.extend(gb_v[keep].tolist())
    all_beta_data.extend(beta[keep].tolist())
    all_chi.extend(chi_v[keep].tolist())

all_gb = np.array(all_gb)
all_beta_data = np.array(all_beta_data)
all_chi = np.array(all_chi)

print("DIRECT CONSTITUTIVE LAW FROM DATA")
print("="*100)
print("Points pooled:", len(all_gb))
print("Global median beta_data:", med(all_beta_data))
print()

# Bin by g_b
logg = np.log10(all_gb)
bins = np.linspace(np.nanpercentile(logg, 5), np.nanpercentile(logg, 95), 9)

print("BINNED BY log10(g_b)")
print("="*100)
print(f'{"bin":<8} {"logg_mid":>10} {"n":>6} {"beta_data":>12} {"chi_med":>12}')
for i in range(len(bins)-1):
    m = (logg >= bins[i]) & (logg < bins[i+1])
    if np.sum(m) == 0:
        continue
    print(
        f'{i+1:<8} '
        f'{med(logg[m]):10.3f} '
        f'{np.sum(m):6d} '
        f'{med(all_beta_data[m]):12.3f} '
        f'{med(all_chi[m]):12.3f}'
    )

# Low / Mid / High
q1, q2 = np.nanpercentile(all_gb, [33.3, 66.7])
low = all_gb < q1
mid = (all_gb >= q1) & (all_gb < q2)
high = all_gb >= q2

print("\nLOW / MID / HIGH FIELD (DATA)")
print("="*100)
for label, mask in [("LOW", low), ("MID", mid), ("HIGH", high)]:
    print(f"{label} FIELD")
    print(f"  median log10(g_b): {med(np.log10(all_gb[mask])):.3f}")
    print(f"  beta_data        : {med(all_beta_data[mask]):.3f}")
    print(f"  chi median       : {med(all_chi[mask]):.3f}")
    print()

DIRECT CONSTITUTIVE LAW FROM DATA
Points pooled: 1136
Global median beta_data: 0.6566648856838386

BINNED BY log10(g_b)
bin        logg_mid      n    beta_data      chi_med
1             1.797    126        0.975     1032.331
2             2.103    156        1.205      993.475
3             2.357    147        0.668     2381.542
4             2.650     93        1.156     2855.250
5             2.970     92        0.537     5085.500
6             3.219    221        0.636     6175.000
7             3.456    104        0.410     9358.778
8             3.755     83       -0.226    10512.000

LOW / MID / HIGH FIELD (DATA)
LOW FIELD
  median log10(g_b): 1.953
  beta_data        : 1.205
  chi median       : 985.440

MID FIELD
  median log10(g_b): 2.743
  beta_data        : 0.842
  chi median       : 3675.750

HIGH FIELD
  median log10(g_b): 3.459
  beta_data        : 0.158
  chi median       : 8414.005



In [ ]:
import numpy as np
import glob
import os
from scipy.optimize import curve_fit

# ============================================================
# FIT SMOOTH CONSTITUTIVE CURVE beta(g_b) FROM DATA
#
# Data-derived quantity:
#   beta_data(r) = d ln chi / d ln g_b
#   chi(r) = Vinf^2 - V(r)^2
#
# Fit model:
#   beta(g) = beta_hi + (beta_lo - beta_hi) / (1 + (g/g_t)^n)
#
# Interpretation:
#   beta_lo : low-field slope
#   beta_hi : high-field slope
#   g_t     : transition field scale
#   n       : sharpness of transition
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6
EPS = 1e-12

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, 1e-9)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def beta_model(g, beta_lo, beta_hi, g_t, n):
    x = np.power(np.clip(g / g_t, 0.0, None), n)
    return beta_hi + (beta_lo - beta_hi) / (1.0 + x)

all_gb = []
all_beta = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
    g_b = v_bar2 / np.maximum(r, 1e-9)

    outer_mask = r >= 0.8 * np.max(r)
    Vinf = np.median(v_obs[outer_mask])
    if not np.isfinite(Vinf) or Vinf <= 0:
        continue

    chi = np.maximum(Vinf**2 - v_obs**2, 0.0)

    valid = (chi > 0) & (g_b > 0)
    if np.sum(valid) < 8:
        continue

    r_v = r[valid]
    chi_v = chi[valid]
    gb_v = g_b[valid]

    ln_chi = np.log(chi_v + EPS)
    ln_gb  = np.log(gb_v + EPS)

    d_ln_chi = np.gradient(ln_chi, r_v)
    d_ln_gb  = np.gradient(ln_gb, r_v)

    beta = d_ln_chi / (d_ln_gb + EPS)

    keep = np.isfinite(beta) & np.isfinite(gb_v) & (gb_v > 0)

    all_gb.extend(gb_v[keep].tolist())
    all_beta.extend(beta[keep].tolist())

all_gb = np.array(all_gb)
all_beta = np.array(all_beta)

print("SMOOTH CONSTITUTIVE CURVE FIT FROM DATA")
print("="*90)
print("Pooled points:", len(all_gb))
print("Raw beta median:", med(all_beta))
print()

# Robust trimming to stop extreme numerical tails dominating the fit
m = np.isfinite(all_gb) & np.isfinite(all_beta)
m &= (all_beta > -1.0) & (all_beta < 3.0)
g_fit = all_gb[m]
b_fit = all_beta[m]

print("Points after robust trim:", len(g_fit))
print("Trimmed beta median:", med(b_fit))
print()

# Initial guess from your binned summaries:
# low field ~ 1.2, high field ~ 0.2, gt ~ 700-1200, n ~ 1
p0 = [1.2, 0.2, 800.0, 1.0]
bounds = (
    [0.5, -0.5, 10.0, 0.1],   # lower
    [2.0,  1.5, 1e5, 5.0]     # upper
)

popt, pcov = curve_fit(
    beta_model,
    g_fit,
    b_fit,
    p0=p0,
    bounds=bounds,
    maxfev=50000
)

beta_lo, beta_hi, g_t, n = popt
perr = np.sqrt(np.diag(pcov))

pred = beta_model(g_fit, *popt)
rmse = np.sqrt(np.mean((pred - b_fit)**2))
mae = np.mean(np.abs(pred - b_fit))

print("BEST-FIT PARAMETERS")
print("="*90)
print(f"beta_lo = {beta_lo:.6f} ± {perr[0]:.6f}")
print(f"beta_hi = {beta_hi:.6f} ± {perr[1]:.6f}")
print(f"g_t     = {g_t:.6f} ± {perr[2]:.6f}")
print(f"n       = {n:.6f} ± {perr[3]:.6f}")
print()
print(f"Fit RMSE on beta_data = {rmse:.6f}")
print(f"Fit MAE  on beta_data = {mae:.6f}")
print()

# Bin summary against fitted curve
logg = np.log10(g_fit)
bins = np.linspace(np.nanpercentile(logg, 5), np.nanpercentile(logg, 95), 9)

print("BINNED DATA vs FIT")
print("="*90)
print(f'{"bin":<8} {"logg_mid":>10} {"n":>6} {"beta_data":>12} {"beta_fit":>12}')
for i in range(len(bins)-1):
    mm = (logg >= bins[i]) & (logg < bins[i+1])
    if np.sum(mm) == 0:
        continue
    gmid = 10**med(logg[mm])
    print(
        f'{i+1:<8} '
        f'{med(logg[mm]):10.3f} '
        f'{np.sum(mm):6d} '
        f'{med(b_fit[mm]):12.3f} '
        f'{beta_model(gmid, *popt):12.3f}'
    )

# Report low/mid/high field predicted slopes
q1, q2 = np.nanpercentile(g_fit, [33.3, 66.7])
for label, gv in [("LOW", med(g_fit[g_fit < q1])),
                  ("MID", med(g_fit[(g_fit >= q1) & (g_fit < q2)])),
                  ("HIGH", med(g_fit[g_fit >= q2]))]:
    print(f"\n{label} FIELD FIT SUMMARY")
    print("-"*90)
    print(f"median g_b        = {gv:.6f}")
    print(f"beta_fit(g_b)     = {beta_model(gv, *popt):.6f}")

print("\nINTERPRETATION")
print("="*90)
print("beta_lo is the low-field constitutive slope.")
print("beta_hi is the high-field constitutive slope.")
print("g_t is the transition field scale.")
print("n controls how sharp the transition is.")
print("If beta_lo > 1 and beta_hi << 1, the data supports a stiff-to-soft constitutive law.")

SMOOTH CONSTITUTIVE CURVE FIT FROM DATA
Pooled points: 1136
Raw beta median: 0.6566648856838386

Points after robust trim: 535
Trimmed beta median: 0.4847919690091165

BEST-FIT PARAMETERS
beta_lo = 0.786454 ± 0.050979
beta_hi = 0.318442 ± 0.171001
g_t     = 3547.137555 ± 1202.418860
n       = 5.000000 ± 6.629226

Fit RMSE on beta_data = 0.974932
Fit MAE  on beta_data = 0.802901

BINNED DATA vs FIT
bin        logg_mid      n    beta_data     beta_fit
1             1.835     59        0.710        0.786
2             2.131     61        0.410        0.786
3             2.385     76        0.344        0.786
4             2.674     50        1.151        0.786
5             2.936     56        0.539        0.786
6             3.229     86        0.842        0.775
7             3.472     53        0.301        0.651
8             3.711     40        0.000        0.382

LOW FIELD FIT SUMMARY
------------------------------------------------------------------------------------------
median g

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# DATA-FITTED CONSTITUTIVE LAW TEST
#
# Use source law implied by fitted beta(g):
#
#   beta_lo ~ 0.786
#   beta_hi ~ 0.318
#   g_t     ~ 3547
#
# Approximate constitutive boost:
#
#   boost(g_b) = A_G * (g_b/GREF)^0.786 * (1 + g_b/g_t)^(-0.468)
#
# since 0.786 - 0.468 ≈ 0.318
#
# Then:
#   rho_eff = rho * (1 + boost)
#
# Goal:
#   test whether the data-derived constitutive law improves
#   high-mass galaxies and reduces the decile knee.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10

# Data-fitted constitutive law
A_G    = 0.40
GREF   = 1000.0
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI   # ~0.468

RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            rmax = np.max(r)
            inner_mask = r <= 0.3 * rmax
            compactness = M_enc[inner_mask][-1] / max(Mtot, EPS) if np.any(inner_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("DATA-FITTED CONSTITUTIVE LAW TEST")
print("="*110)
print("Using:")
print(f"  beta_lo = {BETA_LO:.6f}")
print(f"  beta_hi = {BETA_HI:.6f}")
print(f"  g_t     = {GT_FIT:.6f}")
print(f"  Delta   = {DELTA_BETA:.6f}")
print()
print("Reference rows:")
print("  constant gamma=1.0         -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24")
print("  best linear-compactness    -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4")
print("  best two-slope field law   -> medRMSE ~ 28.7, s_in ~ -0.087, high-mass ~ 57.4, max_jump ~ 16.8")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))

A = fit_amplitude(rows)
res = evaluate(rows, A)

print("\nGLOBAL SUMMARY")
print("="*110)
print("A_global:", A)
print("Median RMSE:", res["median_rmse"])
print("Median signed inner residual:", res["median_signed_inner"])
print("Median low-mass RMSE:", res["median_lowmass_rmse"])
print("Median high-mass RMSE:", res["median_highmass_rmse"])
print("Largest adjacent decile jump:", res["max_decile_jump"])

print("\nDECILE SUMMARY")
print("="*110)
print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
for i, d in enumerate(res["deciles"], start=1):
    print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

DATA-FITTED CONSTITUTIVE LAW TEST
Using:
  beta_lo = 0.786454
  beta_hi = 0.318442
  g_t     = 3547.137555
  Delta   = 0.468012

Reference rows:
  constant gamma=1.0         -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24
  best linear-compactness    -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4
  best two-slope field law   -> medRMSE ~ 28.7, s_in ~ -0.087, high-mass ~ 57.4, max_jump ~ 16.8

Galaxies solved: 60

GLOBAL SUMMARY
A_global: 1.2883928302627974
Median RMSE: 28.925618333131332
Median signed inner residual: -0.09419464431325714
Median low-mass RMSE: 16.304248084393937
Median high-mass RMSE: 57.871211879193424
Largest adjacent decile jump: 17.765716660741106

DECILE SUMMARY
decile         RMSE       s_in    compact      bulge        gas      boost
1            10.526      0.221      0.291      0.500      0.100      0.071
2            12.405      0.555      0.396      0.478      0.122      0.060
3            15.662     -0.054      0.430      0

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# BETA(g_b) SPLIT BY FIT-RANK
#
# Step 1:
#   Rank galaxies by baseline model RMSE
#
# Step 2:
#   Compute beta_data(r) = d ln chi / d ln g_b from data
#   with chi(r) = Vinf^2 - V(r)^2
#
# Step 3:
#   Pool beta_data(g_b) separately for:
#     - BEST   third
#     - MIDDLE third
#     - WORST  third
#
# This tells us whether the constitutive law itself changes
# across the rank transition.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# baseline frozen model for ranking
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
M_EXP  = 1.00
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

EPS = 1e-12
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, 1e-9)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

# ------------------------------------------------------------
# STEP 1: solve baseline model and rank galaxies by RMSE
# ------------------------------------------------------------
rows = []

for path in sorted(glob.glob(ROT_PATH)):
    data = read_file(path)
    if data is None:
        continue
    r, v_obs, v_gas, v_disk, v_bul = data
    if not passes_quality_cuts(r, v_obs):
        continue

    try:
        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        M_enc = r * v_bar2 / G

        rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
        g_b = v_bar2 / np.maximum(r, 1e-9)

        force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
        force_factor = np.nan_to_num(force_factor, nan=0.0, posinf=0.0, neginf=0.0)
        rho_eff = rho * force_factor

        chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
        if chi is None or not np.all(np.isfinite(chi)):
            continue

        chi = chi - np.min(chi)
        chi[chi < 0] = 0.0

        Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
        Uinf = Uchi[-1]
        if not np.isfinite(Uinf) or Uinf <= 0:
            continue

        target = (1.0 - np.exp(-1.0)) * Uinf
        idx = np.searchsorted(Uchi, target)
        idx = min(idx, len(r) - 1)
        Rs = r[idx]
        if not np.isfinite(Rs) or Rs <= 0:
            continue

        rows.append({
            "name": os.path.basename(path),
            "r": r,
            "v_obs": v_obs,
            "v_gas": v_gas,
            "v_disk": v_disk,
            "v_bul": v_bul,
            "v_bar2": v_bar2,
            "Uinf": Uinf,
            "Rs": Rs,
            "Mbar": M_enc[-1]
        })
    except Exception:
        continue

# fit global amplitude for ranking
num = 0.0
den = 0.0
for row in rows:
    x = row["Uinf"] / row["Rs"]
    y = np.max(row["v_obs"])**2
    num += x * y
    den += x * x
A_global = num / den

for row in rows:
    r = row["r"]
    v_obs = row["v_obs"]
    Vflat = np.sqrt(A_global * (row["Uinf"] / row["Rs"]))
    # baseline predictive shape
    # need original Uchi again? reconstruct not stored. do fast re-solve impossible.
    # store approximate ranking via Vinf deficit instead? No, need exact. Let's recompute from v_bar2.
    # Better: rerun field solve for each row.
    v_gas = row["v_gas"]
    v_disk = row["v_disk"]
    v_bul = row["v_bul"]
    v_disk_s = UPS_DISK * v_disk
    v_bul_s  = UPS_BUL * v_bul
    v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
    M_enc = r * v_bar2 / G
    rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
    g_b = v_bar2 / np.maximum(r, 1e-9)
    force_factor = (1.0 + A_G * (g_b / GREF))**M_EXP
    rho_eff = rho * force_factor
    chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
    chi = chi - np.min(chi)
    chi[chi < 0] = 0.0
    Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
    v_pred = Vflat * np.power(np.clip(Uchi / row["Uinf"], 0.0, None), RESP_EXP)
    row["rmse"] = np.sqrt(np.mean((v_pred - v_obs)**2))

rows = sorted(rows, key=lambda z: z["rmse"])
n = len(rows)
third = n // 3
groups = {
    "BEST": rows[:third],
    "MIDDLE": rows[third:2*third],
    "WORST": rows[2*third:]
}

print("BETA(g_b) SPLIT BY FIT-RANK")
print("="*100)
print("Galaxies analysed:", n)
print("Group sizes:", {k: len(v) for k, v in groups.items()})
print()

# ------------------------------------------------------------
# STEP 2: compute beta_data for each group
# ------------------------------------------------------------
group_data = {}

for label, chunk in groups.items():
    gb_all = []
    beta_all = []
    chi_all = []
    mass_all = []
    bulge_all = []
    compact_all = []

    for row in chunk:
        r = row["r"]
        v_obs = row["v_obs"]
        v_gas = row["v_gas"]
        v_disk = row["v_disk"]
        v_bul = row["v_bul"]

        v_disk_s = UPS_DISK * v_disk
        v_bul_s  = UPS_BUL * v_bul
        v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
        g_b = v_bar2 / np.maximum(r, 1e-9)

        outer_mask = r >= 0.8 * np.max(r)
        Vinf = np.median(v_obs[outer_mask])
        if not np.isfinite(Vinf) or Vinf <= 0:
            continue

        chi = np.maximum(Vinf**2 - v_obs**2, 0.0)
        valid = (chi > 0) & (g_b > 0)
        if np.sum(valid) < 8:
            continue

        r_v = r[valid]
        chi_v = chi[valid]
        gb_v = g_b[valid]

        ln_chi = np.log(chi_v + EPS)
        ln_gb  = np.log(gb_v + EPS)

        d_ln_chi = np.gradient(ln_chi, r_v)
        d_ln_gb  = np.gradient(ln_gb, r_v)
        beta = d_ln_chi / (d_ln_gb + EPS)

        keep = np.isfinite(beta)

        M_enc = r * v_bar2 / G
        Mtot = M_enc[-1]
        inner_mask = r <= 0.3 * np.max(r)
        compact = M_enc[inner_mask][-1] / max(Mtot, EPS) if np.any(inner_mask) else np.nan

        i_vmax = int(np.argmax(v_obs))
        vg = abs(v_gas[i_vmax])
        vd = abs(v_disk_s[i_vmax])
        vb = abs(v_bul_s[i_vmax])
        denom = vg + vd + vb + EPS
        bulge_frac = vb / denom

        gb_all.extend(gb_v[keep].tolist())
        beta_all.extend(beta[keep].tolist())
        chi_all.extend(chi_v[keep].tolist())
        mass_all.extend([Mtot] * np.sum(keep))
        bulge_all.extend([bulge_frac] * np.sum(keep))
        compact_all.extend([compact] * np.sum(keep))

    group_data[label] = {
        "gb": np.array(gb_all),
        "beta": np.array(beta_all),
        "chi": np.array(chi_all),
        "mass": np.array(mass_all),
        "bulge": np.array(bulge_all),
        "compact": np.array(compact_all),
    }

# shared bins
all_gb = np.concatenate([group_data[k]["gb"] for k in group_data if len(group_data[k]["gb"])])
logg_all = np.log10(all_gb)
bins = np.linspace(np.nanpercentile(logg_all, 5), np.nanpercentile(logg_all, 95), 9)

for label in ["BEST", "MIDDLE", "WORST"]:
    gd = group_data[label]
    print(f"{label} GROUP")
    print("-"*100)
    print("Points:", len(gd["gb"]))
    print("Median log10(Mbar):", med(np.log10(gd["mass"])))
    print("Median bulge frac :", med(gd["bulge"]))
    print("Median compactness:", med(gd["compact"]))
    print("Global median beta:", med(gd["beta"]))
    print()
    print(f'{"bin":<8} {"logg_mid":>10} {"n":>6} {"beta_data":>12} {"chi_med":>12}')
    logg = np.log10(gd["gb"])
    for i in range(len(bins)-1):
        m = (logg >= bins[i]) & (logg < bins[i+1])
        if np.sum(m) == 0:
            continue
        print(
            f'{i+1:<8} '
            f'{med(logg[m]):10.3f} '
            f'{np.sum(m):6d} '
            f'{med(gd["beta"][m]):12.3f} '
            f'{med(gd["chi"][m]):12.3f}'
        )
    print()

print("INTERPRETATION")
print("="*100)
print("If WORST galaxies show systematically smaller beta_data at high g_b than BEST galaxies,")
print("then the constitutive law itself depends on morphology / regime, not just g_b alone.")

BETA(g_b) SPLIT BY FIT-RANK
Galaxies analysed: 60
Group sizes: {'BEST': 20, 'MIDDLE': 20, 'WORST': 20}

BEST GROUP
----------------------------------------------------------------------------------------------------
Points: 398
Median log10(Mbar): 9.118708135145186
Median bulge frac : 0.5339265005038016
Median compactness: 0.25734861264483616
Global median beta: 0.8864997890721695

bin        logg_mid      n    beta_data      chi_med
1             1.793    101        0.850     1211.263
2             2.090     84        1.075     1470.665
3             2.330     40        0.212     2027.030
4             2.676     25        1.685     2338.622
5             2.968     36        0.410     5375.251
6             3.248     43        0.170     3405.000
7             3.448     34        0.905     7210.543

MIDDLE GROUP
----------------------------------------------------------------------------------------------------
Points: 451
Median log10(Mbar): 9.845116049952082
Median bulge frac : 0.5998

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# COMPACTNESS-MODULATED CONSTITUTIVE LAW TEST
#
# beta(g,C) = beta_hi + (beta_lo - beta_hi) / (1 + (g/g_t(C))^n)
# g_t(C)    = g0 * C^alpha
#
# Convert beta(g,C) to a boost law:
#
# boost(g,C) = A_G * (g/GREF)^beta_lo * (1 + g/g_t(C))^(-(beta_lo-beta_hi))
#
# since low-field slope -> beta_lo
# and high-field slope -> beta_hi
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Start from data-fit values, then modulate gt(C)
BETA_LO = 0.786454
BETA_HI = 0.318442
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(g0, alpha):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / Mtot if np.any(inner_mass_mask) else np.nan
            if not np.isfinite(compactness) or compactness <= 0:
                continue

            g_b = v_bar2 / (r + EPS)
            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)

            gtC = g0 * np.power(compactness, alpha)
            gtC = max(gtC, 1.0)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / gtC, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "gtC": gtC,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        row["rmse"] = rmse
        row["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan

    rows_sorted = sorted(rows, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "gtC": med([c["gtC"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("COMPACTNESS-MODULATED CONSTITUTIVE LAW TEST")
print("="*118)
print("Reference rows:")
print("  constant gamma=1.0      -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24")
print("  best compactness gamma  -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4")
print("  data-fitted field law   -> medRMSE ~ 28.9, s_in ~ -0.094, high-mass ~ 57.9, max_jump ~ 17.8")
print()

grid = [
    (3500.0, -0.5),
    (3500.0, -1.0),
    (3500.0, -1.5),
    (2500.0, -1.0),
    (4500.0, -1.0),
    (3000.0, -0.7),
    (4000.0, -0.7),
]

all_results = []

for g0, alpha in grid:
    rows = build_rows(g0, alpha)
    if len(rows) < 20:
        print(f"(g0={g0:.0f}, alpha={alpha:.2f}) -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)

    all_results.append({
        "g0": g0,
        "alpha": alpha,
        "median_rmse": res["median_rmse"],
        "median_signed_inner": res["median_signed_inner"],
        "median_lowmass_rmse": res["median_lowmass_rmse"],
        "median_highmass_rmse": res["median_highmass_rmse"],
        "max_decile_jump": res["max_decile_jump"],
        "deciles": res["deciles"],
    })

    print(f"(g0={g0:.0f}, alpha={alpha:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print("decile 1/5/10 signed inner:",
          res["deciles"][0]["s_in"], res["deciles"][4]["s_in"], res["deciles"][9]["s_in"])
    print("decile 1/5/10 g_t(C):",
          res["deciles"][0]["gtC"], res["deciles"][4]["gtC"], res["deciles"][9]["gtC"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"g0":>8} {"alpha":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["g0"]:8.0f} {r["alpha"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["max_decile_jump"], z["median_rmse"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("g0 =", best["g0"])
    print("alpha =", best["alpha"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"g_t(C)":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["gtC"]:10.1f} {d["boost"]:10.3f}')

COMPACTNESS-MODULATED CONSTITUTIVE LAW TEST
Reference rows:
  constant gamma=1.0      -> medRMSE ~ 27.7, s_in ~ -0.14, high-mass ~ 69, max_jump ~ 24
  best compactness gamma  -> medRMSE ~ 30.0, s_in ~ -0.08, high-mass ~ 53.6, max_jump ~ 15.4
  data-fitted field law   -> medRMSE ~ 28.9, s_in ~ -0.094, high-mass ~ 57.9, max_jump ~ 17.8

(g0=3500, alpha=-0.50)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 28.908462516558885
median signed inner residual: -0.08778639075799297
median low-mass RMSE: 16.218083002407774
median high-mass RMSE: 57.71086645417185
largest adjacent decile jump: 16.48906198266792
decile 1/5/10 signed inner: 0.22128938811684595 0.22374091567806614 -0.4049591331072365
decile 1/5/10 g_t(C): 6500.367192259986 4934.699499163906 3551.4619015633916

(g0=3500, alpha=-1.00)
----------------------------------------------------------------------------------------------------------------------


In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# DATA-FITTED FIELD LAW + EXPLICIT BULGE CHANNEL SWEEP
#
# Backbone field law:
#   boost(g_b) = A_G * (g_b/GREF)^BETA_LO * (1 + g_b/GT_FIT)^(-DELTA_BETA)
#
# Prediction:
#   V_pred^2(r) = V_field^2(r) + lambda_b * V_bul^2(r) * f(r/r_b)
#
# with gentle bulge windows:
#   f_exp(x)  = exp(-x)
#   f_lor(x)  = 1/(1+x)
#
# Goal:
#   test whether the remaining bulge-heavy failures need an explicit
#   inner bulge branch on top of the constitutive field law.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_exp(x):
    return np.exp(-x)

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lam_b, r_b, window_fn):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        x = r / max(r_b, 1e-6)
        w = window_fn(x)
        v_bulge2 = lam_b * (v_bul_s**2) * w

        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("DATA-FITTED FIELD LAW + BULGE CHANNEL SWEEP")
print("="*118)
print("Reference rows:")
print("  data-fitted field only    -> medRMSE ~ 28.93, s_in ~ -0.094, high-mass ~ 57.87, max_jump ~ 17.77")
print("  best compactness gamma    -> medRMSE ~ 30.00, s_in ~ -0.080, high-mass ~ 53.62, max_jump ~ 15.35")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))
A = fit_amplitude(rows)
print("A_global:", A)
print()

grid = [
    ("lor", 0.10, 0.5),
    ("lor", 0.10, 1.0),
    ("lor", 0.20, 0.5),
    ("lor", 0.20, 1.0),
    ("lor", 0.30, 1.0),
    ("exp", 0.10, 0.5),
    ("exp", 0.10, 1.0),
    ("exp", 0.20, 0.5),
    ("exp", 0.20, 1.0),
    ("exp", 0.30, 1.0),
]

all_results = []

for kind, lam_b, r_b in grid:
    fn = f_lor if kind == "lor" else f_exp
    res = evaluate(rows, A, lam_b, r_b, fn)

    all_results.append({
        "kind": kind,
        "lam_b": lam_b,
        "r_b": r_b,
        **res
    })

    print(f"(window={kind}, lambda_b={lam_b:.2f}, r_b={r_b:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print("decile 1/5/10 signed inner:",
          res["deciles"][0]["s_in"], res["deciles"][4]["s_in"], res["deciles"][9]["s_in"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"win":>8} {"lam_b":>8} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["kind"]:>8} {r["lam_b"]:8.2f} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["max_decile_jump"], z["median_rmse"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("window =", best["kind"])
    print("lambda_b =", best["lam_b"])
    print("r_b =", best["r_b"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

DATA-FITTED FIELD LAW + BULGE CHANNEL SWEEP
Reference rows:
  data-fitted field only    -> medRMSE ~ 28.93, s_in ~ -0.094, high-mass ~ 57.87, max_jump ~ 17.77
  best compactness gamma    -> medRMSE ~ 30.00, s_in ~ -0.080, high-mass ~ 53.62, max_jump ~ 15.35

Galaxies solved: 60
A_global: 1.2883928302627974

(window=lor, lambda_b=0.10, r_b=0.50)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.942125523434086
median signed inner residual: -0.08891549288006417
median low-mass RMSE: 15.61673892475394
median high-mass RMSE: 56.66809782808248
largest adjacent decile jump: 18.57351718637632
decile 1/5/10 signed inner: 0.22302701889890225 -0.17938258104291005 -0.4091780628304151

(window=lor, lambda_b=0.10, r_b=1.00)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.624869238385564
median signed inner residual: -0.0851317370

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# NARROW REFINEMENT: DATA-FITTED FIELD LAW + LORENTZIAN BULGE
#
# V_pred^2(r) = V_field^2(r) + lambda_b * V_bul^2(r) / (1 + r/r_b)
#
# Narrow sweep around current best:
#   lambda_b ~ 0.3
#   r_b      ~ 1.0
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lam_b, r_b):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        x = r / max(r_b, 1e-6)
        v_bulge2 = lam_b * (v_bul_s**2) * f_lor(x)

        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("NARROW LORENTZIAN BULGE-CHANNEL REFINEMENT")
print("="*118)
print("Reference best broad row:")
print("  lambda_b=0.30, r_b=1.00 -> medRMSE ~ 27.80, s_in ~ -0.046, high-mass ~ 54.33, max_jump ~ 18.88")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))
A = fit_amplitude(rows)
print("A_global:", A)
print()

grid = []
for lam_b in [0.20, 0.25, 0.30, 0.35, 0.40]:
    for r_b in [0.70, 0.85, 1.00, 1.20, 1.50]:
        grid.append((lam_b, r_b))

all_results = []

for lam_b, r_b in grid:
    res = evaluate(rows, A, lam_b, r_b)
    all_results.append({
        "lam_b": lam_b,
        "r_b": r_b,
        **res
    })

    print(f"(lambda_b={lam_b:.2f}, r_b={r_b:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"lam_b":>8} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["lam_b"]:8.2f} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("lambda_b =", best["lam_b"])
    print("r_b =", best["r_b"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

NARROW LORENTZIAN BULGE-CHANNEL REFINEMENT
Reference best broad row:
  lambda_b=0.30, r_b=1.00 -> medRMSE ~ 27.80, s_in ~ -0.046, high-mass ~ 54.33, max_jump ~ 18.88

Galaxies solved: 60
A_global: 1.2883928302627974

(lambda_b=0.20, r_b=0.70)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.567164014346023
median signed inner residual: -0.08039771803327161
median low-mass RMSE: 15.187513765701006
median high-mass RMSE: 55.65489434406721
largest adjacent decile jump: 19.25607105964052

(lambda_b=0.20, r_b=0.85)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.59235523778133
median signed inner residual: -0.07818075989014309
median low-mass RMSE: 15.111528014749224
median high-mass RMSE: 55.407056658449
largest adjacent decile jump: 19.34859032836009

(lambda_b=0.20, r_b=1.00)
------------------------------------------

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# EDGE-EXTENSION REFINEMENT:
# DATA-FITTED FIELD LAW + LORENTZIAN BULGE CHANNEL
#
# Extend around current best:
#   lambda_b = 0.40
#   r_b      = 1.50
#
# Test:
#   V_pred^2(r) = V_field^2(r) + lambda_b * V_bul^2(r) / (1 + r/r_b)
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lam_b, r_b):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        x = r / max(r_b, 1e-6)
        v_bulge2 = lam_b * (v_bul_s**2) * f_lor(x)

        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("EDGE-EXTENSION LORENTZIAN BULGE-CHANNEL REFINEMENT")
print("="*118)
print("Reference current best:")
print("  lambda_b=0.40, r_b=1.50 -> medRMSE ~ 27.75, s_in ~ -0.020, high-mass ~ 52.42, max_jump ~ 18.13")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))
A = fit_amplitude(rows)
print("A_global:", A)
print()

grid = []
for lam_b in [0.40, 0.45, 0.50, 0.55]:
    for r_b in [1.50, 1.75, 2.00, 2.50]:
        grid.append((lam_b, r_b))

all_results = []

for lam_b, r_b in grid:
    res = evaluate(rows, A, lam_b, r_b)
    all_results.append({
        "lam_b": lam_b,
        "r_b": r_b,
        **res
    })

    print(f"(lambda_b={lam_b:.2f}, r_b={r_b:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"lam_b":>8} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["lam_b"]:8.2f} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("lambda_b =", best["lam_b"])
    print("r_b =", best["r_b"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

EDGE-EXTENSION LORENTZIAN BULGE-CHANNEL REFINEMENT
Reference current best:
  lambda_b=0.40, r_b=1.50 -> medRMSE ~ 27.75, s_in ~ -0.020, high-mass ~ 52.42, max_jump ~ 18.13

Galaxies solved: 60
A_global: 1.2883928302627974

(lambda_b=0.40, r_b=1.50)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.750136826268026
median signed inner residual: -0.01962847611544778
median low-mass RMSE: 14.563118290891609
median high-mass RMSE: 52.424708988765104
largest adjacent decile jump: 18.132438215770108

(lambda_b=0.40, r_b=1.75)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.655011265899056
median signed inner residual: -0.017822909109419915
median low-mass RMSE: 14.494476921332007
median high-mass RMSE: 51.94089347763335
largest adjacent decile jump: 18.068396626436027

(lambda_b=0.40, r_b=2.00)
-----------------------------

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# SECOND EDGE-EXTENSION:
# DATA-FITTED FIELD LAW + LORENTZIAN BULGE CHANNEL
#
# Continue from current best:
#   lambda_b = 0.55
#   r_b      = 2.50
#
# Test:
#   V_pred^2(r) = V_field^2(r) + lambda_b * V_bul^2(r) / (1 + r/r_b)
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lam_b, r_b):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        x = r / max(r_b, 1e-6)
        v_bulge2 = lam_b * (v_bul_s**2) * f_lor(x)

        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("SECOND EDGE-EXTENSION LORENTZIAN BULGE-CHANNEL REFINEMENT")
print("="*118)
print("Reference current best:")
print("  lambda_b=0.55, r_b=2.50 -> medRMSE ~ 27.19, s_in ~ +0.003, high-mass ~ 48.74, max_jump ~ 15.79")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))
A = fit_amplitude(rows)
print("A_global:", A)
print()

grid = []
for lam_b in [0.55, 0.60, 0.65, 0.70, 0.75]:
    for r_b in [2.50, 3.00, 3.50, 4.00]:
        grid.append((lam_b, r_b))

all_results = []

for lam_b, r_b in grid:
    res = evaluate(rows, A, lam_b, r_b)
    all_results.append({
        "lam_b": lam_b,
        "r_b": r_b,
        **res
    })

    print(f"(lambda_b={lam_b:.2f}, r_b={r_b:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"lam_b":>8} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["lam_b"]:8.2f} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("lambda_b =", best["lam_b"])
    print("r_b =", best["r_b"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

SECOND EDGE-EXTENSION LORENTZIAN BULGE-CHANNEL REFINEMENT
Reference current best:
  lambda_b=0.55, r_b=2.50 -> medRMSE ~ 27.19, s_in ~ +0.003, high-mass ~ 48.74, max_jump ~ 15.79

Galaxies solved: 60
A_global: 1.2883928302627974

(lambda_b=0.55, r_b=2.50)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.19169587898398
median signed inner residual: 0.003387918888771927
median low-mass RMSE: 14.003029062658225
median high-mass RMSE: 48.739872647316304
largest adjacent decile jump: 15.788961915634118

(lambda_b=0.55, r_b=3.00)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 27.3359391757767
median signed inner residual: 0.007429521350756551
median low-mass RMSE: 13.884977217634752
median high-mass RMSE: 47.8301018947213
largest adjacent decile jump: 15.242930741001473

(lambda_b=0.55, r_b=3.50)
---------------------------

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# MICRO-GRID REFINEMENT AROUND CURRENT BEST BASIN
#
# Model:
#   V_pred^2(r) = V_field^2(r) + lambda_b * V_bul^2(r) / (1 + r/r_b)
#
# Field backbone fixed to:
#   data-fitted constitutive law
#
# Micro-grid around:
#   lambda_b ~ 0.65
#   r_b      ~ 3.5
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian(r, rho_eff, v_bar2, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_fac = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_fac = np.nan_to_num(s_fac, nan=1.0, posinf=1.0, neginf=1.0)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_fac

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def f_lor(x):
    return 1.0 / (1.0 + x)

def build_rows():
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)

            rho_eff = rho * (1.0 + boost)

            chi = solve_screened_p_laplacian(r, rho_eff, v_bar2, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "v_bul_s": v_bul_s,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A, lam_b, r_b):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        v_bul_s = row["v_bul_s"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat2 = A * (Uinf / Rs)
        v_field2 = Vflat2 * np.power(np.clip(Uchi / Uinf, 0.0, None), 2.0 * RESP_EXP)

        x = r / max(r_b, 1e-6)
        v_bulge2 = lam_b * (v_bul_s**2) * f_lor(x)

        v_pred = np.sqrt(np.maximum(v_field2 + v_bulge2, 0.0))

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("MICRO-GRID AROUND BEST BASIN")
print("="*118)
print("Reference basin:")
print("  (0.60,4.00) -> medRMSE ~ 26.36, s_in ~ +0.018, high-mass ~ 45.68")
print("  (0.65,3.50) -> medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.64")
print("  (0.70,3.00) -> medRMSE ~ 26.34, s_in ~ +0.019, high-mass ~ 45.77")
print()

rows = build_rows()
print("Galaxies solved:", len(rows))
A = fit_amplitude(rows)
print("A_global:", A)
print()

grid = []
for lam_b in [0.60, 0.625, 0.65, 0.675, 0.70]:
    for r_b in [3.00, 3.25, 3.50, 3.75, 4.00]:
        grid.append((lam_b, r_b))

all_results = []

for lam_b, r_b in grid:
    res = evaluate(rows, A, lam_b, r_b)
    all_results.append({
        "lam_b": lam_b,
        "r_b": r_b,
        **res
    })

    print(f"(lambda_b={lam_b:.3f}, r_b={r_b:.2f})")
    print("-"*118)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"lam_b":>8} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["lam_b"]:8.3f} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("lambda_b =", best["lam_b"])
    print("r_b =", best["r_b"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f}')

MICRO-GRID AROUND BEST BASIN
Reference basin:
  (0.60,4.00) -> medRMSE ~ 26.36, s_in ~ +0.018, high-mass ~ 45.68
  (0.65,3.50) -> medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.64
  (0.70,3.00) -> medRMSE ~ 26.34, s_in ~ +0.019, high-mass ~ 45.77

Galaxies solved: 60
A_global: 1.2883928302627974

(lambda_b=0.600, r_b=3.00)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 26.89713285744889
median signed inner residual: 0.011227867025175491
median low-mass RMSE: 13.776103898418773
median high-mass RMSE: 47.13176084679973
largest adjacent decile jump: 15.447236871985481

(lambda_b=0.600, r_b=3.25)
----------------------------------------------------------------------------------------------------------------------
median RMSE: 26.865069138893006
median signed inner residual: 0.013178812299036658
median low-mass RMSE: 13.719123806015482
median high-mass RMSE: 46.685021522833324
largest adjacent decile jump: 

In [ ]:
import numpy as np
import glob
import os
from scipy.integrate import cumulative_trapezoid

# ============================================================
# RIGID-CORE OPERATOR TEST
#
# Move bulge correction from added velocity term into operator:
#
#   div( S(r) * |grad chi|^(p-2) grad chi ) = rho_eff
#
# with
#   S(r) = 1 + Cb * Vbul^2(r) / (1 + r/rb)
#
# Compare against prior explicit bulge-channel successes.
# ============================================================

WORKDIR = "/content/mts_realdata_workspace_v2"
ROTMOD_DIR = os.path.join(WORKDIR, "ROTMOD_LTG_4")
ROT_PATH = os.path.join(ROTMOD_DIR, "*.dat")

UPS_DISK = 0.5
UPS_BUL  = 0.7
G = 4.30091e-6

# Frozen backbone
P_OP   = 3.5
MU2    = 1e-7
KAPPA  = 0.10
A_G    = 0.40
GREF   = 1000.0
RESP_EXP = 0.20
PHI_S  = 40000.0
N_SCR  = 1.0

# Data-fitted constitutive law
BETA_LO = 0.786454
BETA_HI = 0.318442
GT_FIT  = 3547.137555
DELTA_BETA = BETA_LO - BETA_HI

EPS = 1e-6
GRAD_EPS = 1e-8

MIN_POINTS = 20
MIN_RADIAL_RANGE = 5.0
OUTER_FLAT_TOL = 0.20
MIN_INNER_POINTS = 5

def med(arr):
    arr = np.array([x for x in arr if np.isfinite(x)])
    return np.median(arr) if len(arr) else np.nan

def read_file(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            if line.strip() and not line.startswith("#"):
                parts = line.split()
                if len(parts) >= 5:
                    try:
                        rows.append([float(x) for x in parts[:5]])
                    except:
                        pass
    if not rows:
        return None
    arr = np.array(rows)
    r, v_obs, v_gas, v_disk, v_bul = arr.T
    mask = (
        np.isfinite(r) & np.isfinite(v_obs) & np.isfinite(v_gas) &
        np.isfinite(v_disk) & np.isfinite(v_bul) & (r > 0)
    )
    r, v_obs, v_gas, v_disk, v_bul = r[mask], v_obs[mask], v_gas[mask], v_disk[mask], v_bul[mask]
    if len(r) == 0:
        return None
    o = np.argsort(r)
    return r[o], v_obs[o], v_gas[o], v_disk[o], v_bul[o]

def passes_quality_cuts(r, v_obs):
    if len(r) < MIN_POINTS:
        return False
    rmin, rmax = np.min(r), np.max(r)
    if rmin <= 0 or not np.isfinite(rmin) or not np.isfinite(rmax):
        return False
    if (rmax / rmin) < MIN_RADIAL_RANGE:
        return False
    outer_mask = r >= 0.8 * rmax
    if np.sum(outer_mask) < 2:
        return False
    v_outer = v_obs[outer_mask]
    v_edge = v_obs[-1]
    if not np.isfinite(v_edge) or v_edge <= 0:
        return False
    outer_spread = np.abs(np.median(v_outer) - v_edge) / max(v_edge, EPS)
    if outer_spread > OUTER_FLAT_TOL:
        return False
    if np.sum(r <= 0.5 * rmax) < MIN_INNER_POINTS:
        return False
    return True

def solve_screened_p_laplacian_stiff(r, rho_eff, v_bar2, stiff_mult, phi_s, n_scr, p_op, mu2, n_iter=160, relax=0.25):
    n = len(r)
    chi = np.zeros(n)

    s_scr = 1.0 / (1.0 + np.power(np.clip(v_bar2 / phi_s, 0.0, None), n_scr))
    s_scr = np.nan_to_num(s_scr, nan=1.0, posinf=1.0, neginf=1.0)
    stiff_mult = np.nan_to_num(stiff_mult, nan=1.0, posinf=1.0, neginf=1.0)
    stiff_mult = np.maximum(stiff_mult, 1e-6)

    for _ in range(n_iter):
        dchi = np.gradient(chi, r)
        sigma = stiff_mult * np.power(np.abs(dchi) + GRAD_EPS, p_op - 2.0) * s_scr

        A = np.zeros((n, n))
        b = np.zeros(n)

        A[0, 0] = 1.0
        A[0, 1] = -1.0

        for i in range(1, n - 1):
            rL, rC, rR = r[i - 1], r[i], r[i + 1]
            hL = rC - rL
            hR = rR - rC
            sigL = sigma[i - 1]
            sigR = sigma[i + 1]

            AL = (rL**2 * sigL) / hL
            AR = (rR**2 * sigR) / hR

            A[i, i - 1] = AL
            A[i, i]     = -(AL + AR) - mu2 * rC**2
            A[i, i + 1] = AR
            b[i] = -rC**2 * rho_eff[i]

        A[n - 1, n - 2] = -1.0
        A[n - 1, n - 1] = 1.0

        try:
            chi_new = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            return None

        if not np.all(np.isfinite(chi_new)):
            return None

        chi = (1.0 - relax) * chi + relax * chi_new

    return chi

def build_rows(Cb, r_b):
    rows = []
    for path in sorted(glob.glob(ROT_PATH)):
        data = read_file(path)
        if data is None:
            continue
        r, v_obs, v_gas, v_disk, v_bul = data
        if not passes_quality_cuts(r, v_obs):
            continue

        try:
            v_disk_s = UPS_DISK * v_disk
            v_bul_s  = UPS_BUL * v_bul
            v_bar2 = np.maximum(v_gas**2 + v_disk_s**2 + v_bul_s**2, 0.0)
            M_enc = r * v_bar2 / G
            Mtot = M_enc[-1]
            if not np.isfinite(Mtot) or Mtot <= 0:
                continue

            rho = np.gradient(M_enc, r) / (4.0 * np.pi * r**2 + 1e-6)
            g_b = v_bar2 / (r + EPS)

            boost = A_G * np.power(np.clip(g_b / GREF, 0.0, None), BETA_LO) \
                      * np.power(1.0 + np.clip(g_b / GT_FIT, 0.0, None), -DELTA_BETA)
            boost = np.nan_to_num(boost, nan=0.0, posinf=0.0, neginf=0.0)
            rho_eff = rho * (1.0 + boost)

            stiff_mult = 1.0 + Cb * (v_bul_s**2) / (1.0 + r / max(r_b, 1e-6))
            stiff_mult = np.clip(stiff_mult, 1.0, 1e8)

            chi = solve_screened_p_laplacian_stiff(r, rho_eff, v_bar2, stiff_mult, PHI_S, N_SCR, P_OP, MU2)
            if chi is None or not np.all(np.isfinite(chi)):
                continue

            chi = chi - np.min(chi)
            chi[chi < 0] = 0.0

            Uchi = cumulative_trapezoid(chi * r**KAPPA, r, initial=0.0)
            Uinf = Uchi[-1]
            if not np.isfinite(Uinf) or Uinf <= 0:
                continue

            target = (1.0 - np.exp(-1.0)) * Uinf
            idx = np.searchsorted(Uchi, target)
            idx = min(idx, len(r) - 1)
            Rs = r[idx]
            if not np.isfinite(Rs) or Rs <= 0:
                continue

            i_vmax = int(np.argmax(v_obs))
            vg = abs(v_gas[i_vmax])
            vd = abs(v_disk_s[i_vmax])
            vb = abs(v_bul_s[i_vmax])
            denom = vg + vd + vb + EPS

            inner_mass_mask = r <= 0.3 * np.max(r)
            compactness = M_enc[inner_mass_mask][-1] / max(Mtot, EPS) if np.any(inner_mass_mask) else np.nan

            rows.append({
                "name": os.path.basename(path),
                "r": r,
                "v_obs": v_obs,
                "Mbar": Mtot,
                "Uchi": Uchi,
                "Uinf": Uinf,
                "Rs": Rs,
                "compactness": compactness,
                "bulge_frac": vb / denom,
                "gas_frac": vg / denom,
                "boost_med": med(boost),
                "stiff_med": med(stiff_mult),
            })
        except Exception:
            continue
    return rows

def fit_amplitude(rows):
    num = 0.0
    den = 0.0
    for row in rows:
        x = row["Uinf"] / row["Rs"]
        y = np.max(row["v_obs"])**2
        num += x * y
        den += x * x
    return num / den if den > 0 else np.nan

def evaluate(rows, A):
    out = []
    for row in rows:
        r = row["r"]
        v_obs = row["v_obs"]
        Uchi = row["Uchi"]
        Uinf = row["Uinf"]
        Rs = row["Rs"]

        Vflat = np.sqrt(A * (Uinf / Rs))
        v_pred = Vflat * np.power(np.clip(Uchi / Uinf, 0.0, None), RESP_EXP)

        rmse = np.sqrt(np.mean((v_pred - v_obs)**2))
        xr = r / Rs
        frac_signed = (v_pred - v_obs) / np.maximum(v_obs, 1.0)

        rr = dict(row)
        rr["rmse"] = rmse
        rr["signed_inner"] = np.median(frac_signed[xr < 0.7]) if np.any(xr < 0.7) else np.nan
        out.append(rr)

    rows_sorted = sorted(out, key=lambda z: z["rmse"])
    rmse_all = np.array([r["rmse"] for r in rows_sorted])
    M_all = np.array([r["Mbar"] for r in rows_sorted])

    low = rmse_all[M_all < 1e10]
    high = rmse_all[M_all >= 1e10]

    edges = np.linspace(0, len(rows_sorted), 11, dtype=int)
    dec = []
    for i in range(10):
        chunk = rows_sorted[edges[i]:edges[i+1]]
        dec.append({
            "rmse": med([c["rmse"] for c in chunk]),
            "s_in": med([c["signed_inner"] for c in chunk]),
            "compact": med([c["compactness"] for c in chunk]),
            "bulge": med([c["bulge_frac"] for c in chunk]),
            "gas": med([c["gas_frac"] for c in chunk]),
            "boost": med([c["boost_med"] for c in chunk]),
            "stiff": med([c["stiff_med"] for c in chunk]),
        })

    jumps = np.diff([d["rmse"] for d in dec])

    return {
        "median_rmse": float(np.median(rmse_all)),
        "median_signed_inner": med([r["signed_inner"] for r in rows_sorted]),
        "median_lowmass_rmse": float(np.median(low)) if len(low) else np.nan,
        "median_highmass_rmse": float(np.median(high)) if len(high) else np.nan,
        "max_decile_jump": float(np.max(jumps)) if len(jumps) else np.nan,
        "deciles": dec,
    }

print("RIGID-CORE OPERATOR TEST")
print("="*118)
print("Compare to explicit bulge-channel best:")
print("  lambda_b≈0.675, r_b≈3.25 -> medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.61, max_jump ~ 15.84")
print()

grid = [
    (1e-5, 2.5),
    (2e-5, 2.5),
    (5e-5, 2.5),
    (1e-5, 3.25),
    (2e-5, 3.25),
    (5e-5, 3.25),
    (1e-5, 4.0),
    (2e-5, 4.0),
    (5e-5, 4.0),
]

all_results = []

for Cb, r_b in grid:
    rows = build_rows(Cb, r_b)
    if len(rows) < 20:
        print(f"(Cb={Cb:.1e}, r_b={r_b:.2f}) -> insufficient valid galaxies")
        continue

    A = fit_amplitude(rows)
    res = evaluate(rows, A)

    all_results.append({
        "Cb": Cb,
        "r_b": r_b,
        "A": A,
        **res
    })

    print(f"(Cb={Cb:.1e}, r_b={r_b:.2f})")
    print("-"*118)
    print("A_global:", A)
    print("median RMSE:", res["median_rmse"])
    print("median signed inner residual:", res["median_signed_inner"])
    print("median low-mass RMSE:", res["median_lowmass_rmse"])
    print("median high-mass RMSE:", res["median_highmass_rmse"])
    print("largest adjacent decile jump:", res["max_decile_jump"])
    print()

print("\nCOMPACT SUMMARY")
print("="*118)
print(f'{"Cb":>10} {"r_b":>8} {"med_RMSE":>12} {"s_in":>12} {"RMSE_low":>12} {"RMSE_high":>12} {"max_jump":>12}')
for r in all_results:
    print(f'{r["Cb"]:10.1e} {r["r_b"]:8.2f} {r["median_rmse"]:12.3f} {r["median_signed_inner"]:12.3f} {r["median_lowmass_rmse"]:12.3f} {r["median_highmass_rmse"]:12.3f} {r["max_decile_jump"]:12.3f}')

if all_results:
    best = sorted(all_results, key=lambda z: (z["median_highmass_rmse"], z["median_signed_inner"]**2, z["median_rmse"], z["max_decile_jump"]))[0]
    print("\nBEST ROW")
    print("="*118)
    print("Cb =", best["Cb"])
    print("r_b =", best["r_b"])
    print("A_global =", best["A"])
    print("median RMSE =", best["median_rmse"])
    print("median signed inner =", best["median_signed_inner"])
    print("median low-mass RMSE =", best["median_lowmass_rmse"])
    print("median high-mass RMSE =", best["median_highmass_rmse"])
    print("largest adjacent decile jump =", best["max_decile_jump"])
    print()
    print(f'{"decile":<8} {"RMSE":>10} {"s_in":>10} {"compact":>10} {"bulge":>10} {"gas":>10} {"boost":>10} {"stiff":>10}')
    for i, d in enumerate(best["deciles"], start=1):
        print(f'{i:<8} {d["rmse"]:10.3f} {d["s_in"]:10.3f} {d["compact"]:10.3f} {d["bulge"]:10.3f} {d["gas"]:10.3f} {d["boost"]:10.3f} {d["stiff"]:10.3f}')

RIGID-CORE OPERATOR TEST
Compare to explicit bulge-channel best:
  lambda_b≈0.675, r_b≈3.25 -> medRMSE ~ 26.33, s_in ~ +0.019, high-mass ~ 45.61, max_jump ~ 15.84

(Cb=1.0e-05, r_b=2.50)
----------------------------------------------------------------------------------------------------------------------
A_global: 0.1367110046314045
median RMSE: 93.17570557303802
median signed inner residual: -0.7020163604118129
median low-mass RMSE: 47.958928739884755
median high-mass RMSE: 151.66397605446488
largest adjacent decile jump: 38.57232328104655

(Cb=2.0e-05, r_b=2.50)
----------------------------------------------------------------------------------------------------------------------
A_global: 1.3108925949319823
median RMSE: 29.113210788253873
median signed inner residual: -0.10373826684349002
median low-mass RMSE: 15.826913432096452
median high-mass RMSE: 59.993215311368765
largest adjacent decile jump: 20.414169601926744

(Cb=5.0e-05, r_b=2.50)
------------------------------------------